<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/13_Synthetic_Data_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =============================================================================
# NOTEBOOK 13 — SYNTHETIC DATA GENERATION
# Research: SPP-GAN — Privacy-Preserving Synthetic Tabular Data Generation
# Format: 26 independent Google Colab code sections, aligned with Notebook 12
# =============================================================================

# =============================================================================
# SECTION 1 — HEADER & SCOPE
# =============================================================================

print("=" * 100)
print("NOTEBOOK 13 — SYNTHETIC DATA GENERATION")
print("=" * 100)

NOTEBOOK_NAME = "Notebook 13 — Synthetic Data Generation"
FRAMEWORK_NAME = "SPP-GAN"
DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

print("Framework :", FRAMEWORK_NAME)
print("Datasets  :", DATASET_IDS)
print("Purpose   : Generate, restore, validate, fingerprint, and persist synthetic data.")

NOTEBOOK 13 — SYNTHETIC DATA GENERATION
Framework : SPP-GAN
Datasets  : ['adult_income', 'bank_marketing', 'diabetes_130us']
Purpose   : Generate, restore, validate, fingerprint, and persist synthetic data.


In [2]:
# =============================================================================
# SECTION 2 — LOAD CONFIGURATION
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import os
import json
import hashlib
import random
import time
import gc
import platform
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_ROOT = DRIVE_ROOT / "MyDrive"
PROJECT_ROOT = MYDRIVE_ROOT / "SPP_GAN_Research"

print()
print("-" * 100)
print("GOOGLE DRIVE VERIFICATION")
print("-" * 100)

if not MYDRIVE_ROOT.exists():
    try:
        from google.colab import drive
        drive.mount(str(DRIVE_ROOT), force_remount=False)
    except Exception as exc:
        raise RuntimeError(
            "Google Drive mount failed. Mount Drive in Colab and rerun Section 2."
        ) from exc

if not MYDRIVE_ROOT.exists() or not MYDRIVE_ROOT.is_dir():
    raise FileNotFoundError(f"MyDrive is unavailable: {MYDRIVE_ROOT}")

if not PROJECT_ROOT.exists() or not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Canonical SPP-GAN project root not found: {PROJECT_ROOT}"
    )

NB02_ROOT = PROJECT_ROOT / "data" / "processed" / "notebook_02"
NB12_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_12"

NB13_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_13"
NB13_CONFIG_ROOT = NB13_ROOT / "configuration"
NB13_SYNTHETIC_ROOT = NB13_ROOT / "synthetic_datasets"
NB13_METADATA_ROOT = NB13_ROOT / "metadata"
NB13_MANIFEST_ROOT = NB13_ROOT / "manifests"
NB13_VALIDATION_ROOT = NB13_ROOT / "validation"
NB13_LOG_ROOT = NB13_ROOT / "logs"

for directory in [
    NB13_CONFIG_ROOT,
    NB13_SYNTHETIC_ROOT,
    NB13_METADATA_ROOT,
    NB13_MANIFEST_ROOT,
    NB13_VALIDATION_ROOT,
    NB13_LOG_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)

MASTER_SEED = 2025
GENERATION_REPETITIONS = 1
SAMPLE_SIZE_OVERRIDE = None       # None = use persisted Notebook 12 training-row count
GENERATION_BATCH_SIZE = 1024      # bounded batches for Colab RAM/GPU compatibility
SAVE_PARQUET = False

print(f"✓ MyDrive     : {MYDRIVE_ROOT}")
print(f"✓ Project root: {PROJECT_ROOT}")
print(f"✓ Notebook 02 : {NB02_ROOT}")
print(f"✓ Notebook 12 : {NB12_ROOT}")
print(f"✓ Notebook 13 : {NB13_ROOT}")

if not NB02_ROOT.exists():
    raise FileNotFoundError(f"Notebook 02 root not found: {NB02_ROOT}")
if not NB12_ROOT.exists():
    raise FileNotFoundError(f"Notebook 12 root not found: {NB12_ROOT}")


----------------------------------------------------------------------------------------------------
GOOGLE DRIVE VERIFICATION
----------------------------------------------------------------------------------------------------
Mounted at /content/drive
✓ MyDrive     : /content/drive/MyDrive
✓ Project root: /content/drive/MyDrive/SPP_GAN_Research
✓ Notebook 02 : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
✓ Notebook 12 : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_12
✓ Notebook 13 : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13


In [6]:
# =============================================================================
# SECTION 3 — DISCOVER TRAINED MODELS
# =============================================================================

from pathlib import Path
import hashlib
import pandas as pd

print("=" * 100)
print("SECTION 3 — DISCOVER TRAINED MODELS")
print("=" * 100)

# -----------------------------------------------------------------------------
# 1. Validate Paths and Dataset Registry
# -----------------------------------------------------------------------------

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Project root not found: {PROJECT_ROOT}")

if not NB12_ROOT.is_dir():
    raise FileNotFoundError(f"Notebook 12 root not found: {NB12_ROOT}")

DATASET_NAMES = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

print(f"✓ Project root : {PROJECT_ROOT}")
print(f"✓ Notebook 12  : {NB12_ROOT}")

# -----------------------------------------------------------------------------
# 2. Load the Authoritative Notebook 12B Registry
# -----------------------------------------------------------------------------

MODEL_REGISTRY_PATH = (
    NB12_ROOT / "registries" /
    "notebook_12B_final_model_registry.csv"
)

if not MODEL_REGISTRY_PATH.is_file():
    raise FileNotFoundError(
        f"Authoritative Notebook 12B registry not found: "
        f"{MODEL_REGISTRY_PATH}"
    )

MODEL_REGISTRY_DF = pd.read_csv(MODEL_REGISTRY_PATH)

REQUIRED_COLUMNS = {
    "dataset",
    "repetition_id",
    "status",
    "final_model_path",
    "final_model_sha256",
}

missing_columns = REQUIRED_COLUMNS - set(MODEL_REGISTRY_DF.columns)

if missing_columns:
    raise ValueError(
        f"Notebook 12B registry is missing required columns: "
        f"{sorted(missing_columns)}. Available columns: "
        f"{MODEL_REGISTRY_DF.columns.tolist()}"
    )

print(f"✓ Model registry : {MODEL_REGISTRY_PATH}")
print(f"✓ Registry rows  : {len(MODEL_REGISTRY_DF):,}")

# -----------------------------------------------------------------------------
# 3. Validate Registry Status and Select a Repetition
# -----------------------------------------------------------------------------

registry = MODEL_REGISTRY_DF.copy()

registry["dataset"] = (
    registry["dataset"].astype(str).str.strip().str.lower()
)
registry["status"] = (
    registry["status"].astype(str).str.strip().str.upper()
)
registry["repetition_id"] = pd.to_numeric(
    registry["repetition_id"], errors="coerce"
)

registry["final_model_path"] = (
    registry["final_model_path"].astype(str).str.strip()
)
registry["final_model_sha256"] = (
    registry["final_model_sha256"].astype(str).str.strip().str.lower()
)

SELECTED_GENERATORS = {}
REGISTRY_ERRORS = []

for dataset in DATASET_NAMES:
    rows = registry.loc[
        (registry["dataset"] == dataset)
        & (registry["status"] == "PASS")
    ].copy()

    if rows.empty:
        REGISTRY_ERRORS.append(
            f"{dataset}: no registry row has status PASS."
        )
        continue

    # Deterministically select repetition 1 for generation.
    # This is a fixed-repetition policy, not a claim that repetition 1
    # is empirically superior to the other repetitions.
    rows = rows.loc[rows["repetition_id"] == 1]

    if len(rows) != 1:
        REGISTRY_ERRORS.append(
            f"{dataset}: expected exactly one PASS record for repetition 1; "
            f"found {len(rows)}."
        )
        continue

    row = rows.iloc[0]
    raw_path = row["final_model_path"]

    if not raw_path or raw_path.lower() in {"nan", "none", "null"}:
        REGISTRY_ERRORS.append(
            f"{dataset}: final_model_path is empty."
        )
        continue

    model_path = Path(raw_path)

    if not model_path.is_absolute():
        model_path = PROJECT_ROOT / model_path

    model_path = model_path.resolve()

    if not model_path.is_file():
        REGISTRY_ERRORS.append(
            f"{dataset}: registered final model does not exist: {model_path}"
        )
        continue

    expected_hash = row["final_model_sha256"]

    if len(expected_hash) != 64 or any(
        char not in "0123456789abcdef" for char in expected_hash
    ):
        REGISTRY_ERRORS.append(
            f"{dataset}: invalid SHA-256 in registry for {model_path}."
        )
        continue

    # Verify artifact integrity before loading the model.
    digest = hashlib.sha256()

    with model_path.open("rb") as model_file:
        for chunk in iter(lambda: model_file.read(1024 * 1024), b""):
            digest.update(chunk)

    actual_hash = digest.hexdigest()

    if actual_hash != expected_hash:
        REGISTRY_ERRORS.append(
            f"{dataset}: SHA-256 mismatch for {model_path}. "
            f"Expected {expected_hash}; got {actual_hash}."
        )
        continue

    SELECTED_GENERATORS[dataset] = {
        "dataset": dataset,
        "repetition_id": int(row["repetition_id"]),
        "model_path": model_path,
        "model_sha256": actual_hash,
        "status": row["status"],
        "registry_row": row.to_dict(),
    }

# -----------------------------------------------------------------------------
# 4. Discover Candidate Artifacts for Audit Only
# -----------------------------------------------------------------------------

MODEL_SEARCH_ROOTS = [
    NB12_ROOT / "models",
    NB12_ROOT / "checkpoints",
    NB12_ROOT / "artifacts",
    NB12_ROOT / "training",
]

MODEL_EXTENSIONS = {
    ".pt", ".pth", ".ckpt", ".pkl", ".pickle", ".joblib"
}

MODEL_CANDIDATES = {dataset: [] for dataset in DATASET_NAMES}

for search_root in MODEL_SEARCH_ROOTS:
    if not search_root.is_dir():
        continue

    for path in search_root.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in MODEL_EXTENSIONS:
            continue

        if any(
            part.lower() in {
                "__pycache__", ".ipynb_checkpoints", "temp", "tmp"
            }
            for part in path.parts
        ):
            continue

        path_text = str(path).lower()

        for dataset in DATASET_NAMES:
            if dataset in path_text:
                MODEL_CANDIDATES[dataset].append(path)
                break

for dataset in DATASET_NAMES:
    MODEL_CANDIDATES[dataset] = sorted(
        set(MODEL_CANDIDATES[dataset]),
        key=lambda path: str(path).lower(),
    )

# -----------------------------------------------------------------------------
# 5. Build and Persist Discovery Summary
# -----------------------------------------------------------------------------

summary_rows = []

for dataset in DATASET_NAMES:
    selected = SELECTED_GENERATORS.get(dataset)

    summary_rows.append({
        "dataset": dataset,
        "candidate_count": len(MODEL_CANDIDATES[dataset]),
        "selected_repetition": (
            selected["repetition_id"] if selected else None
        ),
        "selected_generator_path": (
            str(selected["model_path"]) if selected else None
        ),
        "selected_generator_sha256": (
            selected["model_sha256"] if selected else None
        ),
        "registry_status": (
            selected["status"] if selected else None
        ),
        "discovery_status": (
            "VALIDATED_REGISTRY_MODEL"
            if selected else "SELECTION_OR_VALIDATION_FAILED"
        ),
    })

MODEL_DISCOVERY_SUMMARY = pd.DataFrame(summary_rows)

print("\n" + "-" * 100)
print("MODEL DISCOVERY SUMMARY")
print("-" * 100)
print(MODEL_DISCOVERY_SUMMARY.to_string(index=False))

for dataset in DATASET_NAMES:
    print(f"\n{dataset.upper()} — DISCOVERED CANDIDATES")
    for path in MODEL_CANDIDATES[dataset][:10]:
        print(f"  - {path}")

NB13_LOG_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_DISCOVERY_SUMMARY_PATH = (
    NB13_LOG_ROOT / "model_discovery_summary.csv"
)

MODEL_DISCOVERY_SUMMARY.to_csv(
    MODEL_DISCOVERY_SUMMARY_PATH,
    index=False,
)

print(f"\n✓ Discovery summary saved: {MODEL_DISCOVERY_SUMMARY_PATH}")

# -----------------------------------------------------------------------------
# 6. Fail Closed if Any Registered Model Is Invalid
# -----------------------------------------------------------------------------

if REGISTRY_ERRORS:
    for error in REGISTRY_ERRORS:
        print(f"⚠ {error}")

if set(SELECTED_GENERATORS) != set(DATASET_NAMES):
    raise RuntimeError(
        "Notebook 12B generator validation failed. Resolve the registry, "
        "artifact-path, status, or SHA-256 errors before synthetic generation."
    )

print("✓ All selected generators match their registered SHA-256 hashes.")
print("✓ Fixed repetition policy: repetition 1 for each dataset.")
print("SECTION 3 COMPLETED")

SECTION 3 — DISCOVER TRAINED MODELS
✓ Project root : /content/drive/MyDrive/SPP_GAN_Research
✓ Notebook 12  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_12
✓ Model registry : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_12/registries/notebook_12B_final_model_registry.csv
✓ Registry rows  : 15

----------------------------------------------------------------------------------------------------
MODEL DISCOVERY SUMMARY
----------------------------------------------------------------------------------------------------
       dataset  candidate_count  selected_repetition                                                                                                                           selected_generator_path                                        selected_generator_sha256 registry_status         discovery_status
  adult_income              176                    1     /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_12/m

In [7]:
# ==================================================================================================
# SECTION 4 — VALIDATE MODEL ARTIFACTS
# ==================================================================================================

print("=" * 100)
print("SECTION 4 — VALIDATE MODEL ARTIFACTS")
print("=" * 100)

import hashlib
import pandas as pd
from pathlib import Path

# --------------------------------------------------------------------------------------------------
# 1. Define SHA-256 Fingerprinting
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# --------------------------------------------------------------------------------------------------
# 2. Validate Notebook 12 Registry
# --------------------------------------------------------------------------------------------------

assert (
    MODEL_REGISTRY_AVAILABLE
    and isinstance(MODEL_REGISTRY_DF, pd.DataFrame)
    and not MODEL_REGISTRY_DF.empty
), "Notebook 12 final-model registry is unavailable or empty."

required_registry_columns = {
    "dataset",
    "repetition_id",
    "status",
    "final_model_path",
    "final_model_sha256",
    "final_model_size_bytes",
}

missing_columns = required_registry_columns.difference(
    MODEL_REGISTRY_DF.columns
)

assert not missing_columns, (
    f"Registry is missing required columns: {sorted(missing_columns)}"
)

registry = MODEL_REGISTRY_DF.copy()

registry["dataset"] = (
    registry["dataset"].astype(str).str.strip().str.lower()
)

registry["status"] = (
    registry["status"].astype(str).str.strip().str.upper()
)

registry["repetition_id"] = pd.to_numeric(
    registry["repetition_id"],
    errors="coerce",
)

registry["final_model_size_bytes"] = pd.to_numeric(
    registry["final_model_size_bytes"],
    errors="coerce",
)

# Validate that the registry contains exactly one record for each dataset/repetition.
assert not registry.duplicated(
    subset=["dataset", "repetition_id"]
).any(), "Duplicate dataset/repetition records found in the model registry."

# This section validates every registered final model. It does not arbitrarily
# declare one repetition to be the best; all PASS repetitions remain identifiable.
MODEL_ARTIFACTS = {}
MODEL_VALIDATION_ROWS = []

# --------------------------------------------------------------------------------------------------
# 3. Validate Every Registered Final Generator
# --------------------------------------------------------------------------------------------------

for _, record in registry.iterrows():
    dataset = record["dataset"]
    repetition_id = record["repetition_id"]
    recorded_path = str(record["final_model_path"]).strip()

    if dataset not in DATASET_NAMES:
        continue

    artifact_path = Path(recorded_path)

    exists = artifact_path.is_file()
    actual_size = artifact_path.stat().st_size if exists else None

    expected_size = record["final_model_size_bytes"]
    expected_sha256 = str(record["final_model_sha256"]).strip().lower()

    actual_sha256 = (
        sha256_file(artifact_path)
        if exists and actual_size > 0
        else None
    )

    size_matches = (
        actual_size is not None
        and pd.notna(expected_size)
        and actual_size == int(expected_size)
    )

    hash_matches = (
        actual_sha256 is not None
        and expected_sha256
        and actual_sha256 == expected_sha256
    )

    registry_pass = record["status"] == "PASS"
    nonempty_file = exists and actual_size > 0

    if (
        registry_pass
        and nonempty_file
        and size_matches
        and hash_matches
    ):
        validation_status = "VALIDATED"
    elif not exists:
        validation_status = "MISSING_FILE"
    elif not nonempty_file:
        validation_status = "EMPTY_FILE"
    elif not size_matches:
        validation_status = "SIZE_MISMATCH"
    elif not hash_matches:
        validation_status = "SHA256_MISMATCH"
    else:
        validation_status = "REGISTRY_STATUS_NOT_PASS"

    MODEL_VALIDATION_ROWS.append(
        {
            "dataset": dataset,
            "repetition_id": (
                int(repetition_id)
                if pd.notna(repetition_id)
                else None
            ),
            "artifact_path": recorded_path,
            "registry_status": record["status"],
            "exists": exists,
            "size_bytes": actual_size,
            "expected_size_bytes": (
                int(expected_size)
                if pd.notna(expected_size)
                else None
            ),
            "size_matches": size_matches,
            "sha256": actual_sha256,
            "expected_sha256": expected_sha256,
            "sha256_matches": hash_matches,
            "selection_source": "NOTEBOOK_12_FINAL_MODEL_REGISTRY",
            "status": validation_status,
        }
    )

    if validation_status == "VALIDATED":
        MODEL_ARTIFACTS.setdefault(dataset, {})[
            int(repetition_id)
        ] = artifact_path


# --------------------------------------------------------------------------------------------------
# 4. Build Validation Report
# --------------------------------------------------------------------------------------------------

MODEL_VALIDATION_DF = pd.DataFrame(MODEL_VALIDATION_ROWS)

NB13_VALIDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_VALIDATION_PATH = (
    NB13_VALIDATION_ROOT / "model_artifact_validation.csv"
)

MODEL_VALIDATION_DF.to_csv(
    MODEL_VALIDATION_PATH,
    index=False,
)

print("\n" + "-" * 100)
print("MODEL ARTIFACT VALIDATION SUMMARY")
print("-" * 100)

if MODEL_VALIDATION_DF.empty:
    print("⚠ No registered model artifacts matched the configured datasets.")
else:
    display(MODEL_VALIDATION_DF)

print("\nValidated repetitions by dataset:")

for dataset in DATASET_NAMES:
    repetitions = sorted(MODEL_ARTIFACTS.get(dataset, {}).keys())

    print(
        f"  {dataset}: {len(repetitions)} validated "
        f"repetition(s) {repetitions}"
    )

print(f"\n✓ Validation report saved: {MODEL_VALIDATION_PATH}")

# --------------------------------------------------------------------------------------------------
# 5. Verify Dataset Coverage
# --------------------------------------------------------------------------------------------------

missing_datasets = [
    dataset
    for dataset in DATASET_NAMES
    if not MODEL_ARTIFACTS.get(dataset)
]

if missing_datasets:
    print("\n⚠ Some datasets have no validated generator artifacts:")
    for dataset in missing_datasets:
        print(f"  - {dataset}")
else:
    print("\n✓ All configured datasets have validated generator artifacts.")

print(
    "\nNote: All valid registered repetitions are retained. "
    "The generation section must explicitly determine whether to generate "
    "from every repetition or from a documented selected repetition."
)

SECTION 4 — VALIDATE MODEL ARTIFACTS

----------------------------------------------------------------------------------------------------
MODEL ARTIFACT VALIDATION SUMMARY
----------------------------------------------------------------------------------------------------


,dataset,repetition_id,artifact_path,registry_status,exists,size_bytes,expected_size_bytes,size_matches,sha256,expected_sha256,sha256_matches,selection_source,status
0,adult_income,1,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2712245,2712245,True,3147dc0dd607f0b242d35daed9e4c7a17830f9589fe29b...,3147dc0dd607f0b242d35daed9e4c7a17830f9589fe29b...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED
1,adult_income,2,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2712245,2712245,True,60e8d6409ddd8f5f2e9a82cd758829a736e46c4921913e...,60e8d6409ddd8f5f2e9a82cd758829a736e46c4921913e...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED
2,adult_income,3,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2712245,2712245,True,48e62c40e789f84822f4c26b75137464996fbff3274b8e...,48e62c40e789f84822f4c26b75137464996fbff3274b8e...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED
3,adult_income,4,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2712309,2712309,True,183431ff4d80d2d1228f9adad8470986db81c1d5b664d3...,183431ff4d80d2d1228f9adad8470986db81c1d5b664d3...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED
4,adult_income,5,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2712245,2712245,True,aa1828fd4b41c1450c7c447f1c4206c1a95213bf9fe0fa...,aa1828fd4b41c1450c7c447f1c4206c1a95213bf9fe0fa...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED
5,bank_marketing,1,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2379509,2379509,True,5bb3cc6fcea00cb21f417257963552a16c5e7faaa149ba...,5bb3cc6fcea00cb21f417257963552a16c5e7faaa149ba...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED
6,bank_marketing,2,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2379509,2379509,True,e9055c31fab00a8be54c818a8cde714572680908ae9c75...,e9055c31fab00a8be54c818a8cde714572680908ae9c75...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED
7,bank_marketing,3,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2379509,2379509,True,797dd47e7c97106cce5d02bc52da26bbc9408567a37877...,797dd47e7c97106cce5d02bc52da26bbc9408567a37877...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED
8,bank_marketing,4,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2379573,2379573,True,fed615ef19110561dddae97a6d67d63950dcb67adece08...,fed615ef19110561dddae97a6d67d63950dcb67adece08...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED
9,bank_marketing,5,/content/drive/MyDrive/SPP_GAN_Research/result...,PASS,True,2379509,2379509,True,9ef74c66d380fbbbb5e33c900c00e23f74dcd7c76e9005...,9ef74c66d380fbbbb5e33c900c00e23f74dcd7c76e9005...,True,NOTEBOOK_12_FINAL_MODEL_REGISTRY,VALIDATED



Validated repetitions by dataset:
  adult_income: 5 validated repetition(s) [1, 2, 3, 4, 5]
  bank_marketing: 5 validated repetition(s) [1, 2, 3, 4, 5]
  diabetes_130us: 5 validated repetition(s) [1, 2, 3, 4, 5]

✓ Validation report saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/model_artifact_validation.csv

✓ All configured datasets have validated generator artifacts.

Note: All valid registered repetitions are retained. The generation section must explicitly determine whether to generate from every repetition or from a documented selected repetition.


In [8]:
# ==================================================================================================
# SECTION 5 — LOAD PREPROCESSORS/SCHEMAS
# ==================================================================================================

print("=" * 100)
print("SECTION 5 — LOAD PREPROCESSORS/SCHEMAS")
print("=" * 100)

import json
import joblib
import pandas as pd
from pathlib import Path

# --------------------------------------------------------------------------------------------------
# 1. Define Notebook 02 Artifact Search Roots
# --------------------------------------------------------------------------------------------------

PREPROCESSOR_ROOT_CANDIDATES = [
    NB02_ROOT / "preprocessors",
    NB02_ROOT / "artifacts" / "preprocessors",
    NB02_ROOT / "artifacts",
    NB02_ROOT,
]

SCHEMA_ROOT_CANDIDATES = [
    NB02_ROOT / "schemas",
    NB02_ROOT / "artifacts" / "schemas",
    NB02_ROOT / "artifacts",
    NB02_ROOT,
]

NATIVE_ROOT_CANDIDATES = [
    NB02_ROOT / "native",
    NB02_ROOT / "processed",
    NB02_ROOT / "datasets",
    NB02_ROOT / "splits",
    NB02_ROOT / "artifacts",
    NB02_ROOT,
]

# --------------------------------------------------------------------------------------------------
# 2. Discover Candidate Artifacts Without Arbitrary Selection
# --------------------------------------------------------------------------------------------------

def discover_dataset_artifacts(
    roots,
    dataset,
    extensions,
    keywords=None,
):
    """
    Return all matching artifact paths in deterministic order.
    Multiple matches are preserved so that ambiguous files are not silently
    treated as the canonical artifact.
    """
    keywords = [word.lower() for word in (keywords or [])]
    candidates = set()

    for root in roots:
        if not root.exists():
            continue

        for path in root.rglob("*"):
            if not path.is_file():
                continue

            if path.suffix.lower() not in extensions:
                continue

            path_text = str(path).lower()

            if dataset.lower() not in path_text:
                continue

            if keywords and not any(
                word in path.name.lower() for word in keywords
            ):
                continue

            if any(
                excluded in path_text
                for excluded in [
                    "/.ipynb_checkpoints/",
                    "/__pycache__/",
                    "/temp/",
                    "/tmp/",
                ]
            ):
                continue

            candidates.add(path.resolve())

    return sorted(candidates, key=lambda path: str(path).lower())


# --------------------------------------------------------------------------------------------------
# 3. Initialize Artifact Registries
# --------------------------------------------------------------------------------------------------

PREPROCESSORS = {}
PREPROCESSOR_PATHS = {}
PREPROCESSOR_CANDIDATES = {}

SCHEMAS = {}
SCHEMA_PATHS = {}
SCHEMA_CANDIDATES = {}

REFERENCE_TRAIN = {}
REFERENCE_PATHS = {}
REFERENCE_CANDIDATES = {}

ARTIFACT_DISCOVERY_ROWS = []

# --------------------------------------------------------------------------------------------------
# 4. Discover Preprocessors, Schemas, and Reference Datasets
# --------------------------------------------------------------------------------------------------

for dataset in DATASET_IDS:

    # Search for serialized preprocessing artifacts.
    prep_candidates = discover_dataset_artifacts(
        PREPROCESSOR_ROOT_CANDIDATES,
        dataset,
        {".joblib", ".pkl", ".pickle"},
        ["preprocessor", "transformer", "encoder"],
    )

    # Search for schema-related JSON files.
    schema_candidates = discover_dataset_artifacts(
        SCHEMA_ROOT_CANDIDATES,
        dataset,
        {".json"},
        ["schema", "feature", "manifest"],
    )

    # Search for potential reference datasets.
    native_candidates = discover_dataset_artifacts(
        NATIVE_ROOT_CANDIDATES,
        dataset,
        {".csv"},
        ["native", "processed", "train", "dataset"],
    )

    PREPROCESSOR_CANDIDATES[dataset] = prep_candidates
    SCHEMA_CANDIDATES[dataset] = schema_candidates
    REFERENCE_CANDIDATES[dataset] = native_candidates

    # Only automatically load when exactly one candidate exists.
    # Multiple candidates require an explicit canonical-path decision.
    prep_path = (
        prep_candidates[0]
        if len(prep_candidates) == 1
        else None
    )

    schema_path = (
        schema_candidates[0]
        if len(schema_candidates) == 1
        else None
    )

    native_path = (
        native_candidates[0]
        if len(native_candidates) == 1
        else None
    )

    PREPROCESSOR_PATHS[dataset] = prep_path
    SCHEMA_PATHS[dataset] = schema_path
    REFERENCE_PATHS[dataset] = native_path

    # Load preprocessor.
    if prep_path is not None:
        try:
            PREPROCESSORS[dataset] = joblib.load(prep_path)
            prep_status = "LOADED"
        except Exception as exc:
            prep_status = f"LOAD_FAILED: {exc}"
    elif not prep_candidates:
        prep_status = "NOT_FOUND"
    else:
        prep_status = f"AMBIGUOUS: {len(prep_candidates)} candidates"

    # Load schema.
    if schema_path is not None:
        try:
            with open(schema_path, "r", encoding="utf-8") as handle:
                SCHEMAS[dataset] = json.load(handle)
            schema_status = "LOADED"
        except Exception as exc:
            schema_status = f"LOAD_FAILED: {exc}"
    elif not schema_candidates:
        schema_status = "NOT_FOUND"
    else:
        schema_status = f"AMBIGUOUS: {len(schema_candidates)} candidates"

    # Load reference dataset.
    if native_path is not None:
        try:
            REFERENCE_TRAIN[dataset] = pd.read_csv(
                native_path,
                low_memory=False,
            )
            reference_status = "LOADED"
        except Exception as exc:
            reference_status = f"LOAD_FAILED: {exc}"
    elif not native_candidates:
        reference_status = "NOT_FOUND"
    else:
        reference_status = f"AMBIGUOUS: {len(native_candidates)} candidates"

    ARTIFACT_DISCOVERY_ROWS.append(
        {
            "dataset": dataset,
            "preprocessor_candidates": len(prep_candidates),
            "preprocessor_path": str(prep_path) if prep_path else None,
            "preprocessor_status": prep_status,
            "schema_candidates": len(schema_candidates),
            "schema_path": str(schema_path) if schema_path else None,
            "schema_status": schema_status,
            "reference_candidates": len(native_candidates),
            "reference_path": str(native_path) if native_path else None,
            "reference_status": reference_status,
        }
    )

    print(f"\n{dataset}")
    print(f"  Preprocessor: {prep_status}")
    print(f"  Schema      : {schema_status}")
    print(f"  Reference   : {reference_status}")

    for category, paths in [
        ("Preprocessor candidates", prep_candidates),
        ("Schema candidates", schema_candidates),
        ("Reference candidates", native_candidates),
    ]:
        if paths:
            print(f"  {category}:")
            for path in paths[:10]:
                print(f"    - {path}")

# --------------------------------------------------------------------------------------------------
# 5. Load Notebook 12 Configuration Snapshot
# --------------------------------------------------------------------------------------------------

NB12_CONFIGURATION_PATH = (
    NB12_ROOT / "configuration" / "sppgan_training_configuration.json"
)

NB12_CONFIGURATION = {}

if NB12_CONFIGURATION_PATH.is_file():
    try:
        with open(
            NB12_CONFIGURATION_PATH,
            "r",
            encoding="utf-8",
        ) as handle:
            NB12_CONFIGURATION = json.load(handle)

        print(
            "\n✓ Loaded Notebook 12 configuration:",
            NB12_CONFIGURATION_PATH,
        )

    except Exception as exc:
        print(f"\n⚠ Could not load Notebook 12 configuration: {exc}")

else:
    print(
        "\n⚠ Notebook 12 configuration snapshot not found:",
        NB12_CONFIGURATION_PATH,
    )

# --------------------------------------------------------------------------------------------------
# 6. Save Artifact Discovery Report
# --------------------------------------------------------------------------------------------------

ARTIFACT_DISCOVERY_DF = pd.DataFrame(ARTIFACT_DISCOVERY_ROWS)

NB13_VALIDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

ARTIFACT_DISCOVERY_PATH = (
    NB13_VALIDATION_ROOT / "preprocessor_schema_discovery.csv"
)

ARTIFACT_DISCOVERY_DF.to_csv(
    ARTIFACT_DISCOVERY_PATH,
    index=False,
)

print("\n" + "-" * 100)
print("SECTION 5 DISCOVERY SUMMARY")
print("-" * 100)

display(ARTIFACT_DISCOVERY_DF)

print(f"\n✓ Discovery report saved: {ARTIFACT_DISCOVERY_PATH}")

print(
    "\n⚠ Generation must not proceed until the canonical preprocessing artifacts, "
    "feature schema, and reference data are identified and validated."
)

SECTION 5 — LOAD PREPROCESSORS/SCHEMAS

adult_income
  Preprocessor: LOADED
  Schema      : LOADED
  Reference   : LOADED
  Preprocessor candidates:
    - /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/preprocessors/adult_income/train_fitted_preprocessor.joblib
  Schema candidates:
    - /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas/adult_income/preprocessing_schema.json
  Reference candidates:
    - /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native/adult_income/train.csv

bank_marketing
  Preprocessor: LOADED
  Schema      : LOADED
  Reference   : LOADED
  Preprocessor candidates:
    - /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/preprocessors/bank_marketing/train_fitted_preprocessor.joblib
  Schema candidates:
    - /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas/bank_marketing/preprocessing_schema.json
  Reference candidates:
    - /content/drive/MyDrive/SPP_GAN

,dataset,preprocessor_candidates,preprocessor_path,preprocessor_status,schema_candidates,schema_path,schema_status,reference_candidates,reference_path,reference_status
0,adult_income,1,/content/drive/MyDrive/SPP_GAN_Research/data/p...,LOADED,1,/content/drive/MyDrive/SPP_GAN_Research/data/p...,LOADED,1,/content/drive/MyDrive/SPP_GAN_Research/data/p...,LOADED
1,bank_marketing,1,/content/drive/MyDrive/SPP_GAN_Research/data/p...,LOADED,1,/content/drive/MyDrive/SPP_GAN_Research/data/p...,LOADED,1,/content/drive/MyDrive/SPP_GAN_Research/data/p...,LOADED
2,diabetes_130us,1,/content/drive/MyDrive/SPP_GAN_Research/data/p...,LOADED,1,/content/drive/MyDrive/SPP_GAN_Research/data/p...,LOADED,1,/content/drive/MyDrive/SPP_GAN_Research/data/p...,LOADED



✓ Discovery report saved: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/preprocessor_schema_discovery.csv

⚠ Generation must not proceed until the canonical preprocessing artifacts, feature schema, and reference data are identified and validated.


In [9]:
# =============================================================================
# SECTION 6 — CONFIGURE GENERATION
# =============================================================================

TARGET_DEFAULTS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

def resolve_target(dataset):
    schema = SCHEMAS.get(dataset, {})
    if isinstance(schema, dict):
        for key in ("target", "target_column", "target_name"):
            value = schema.get(key)
            if isinstance(value, str) and value.strip():
                return value
        for parent_key in ("dataset", "metadata", "configuration", "columns"):
            nested = schema.get(parent_key)
            if isinstance(nested, dict):
                for key in ("target", "target_column", "target_name"):
                    value = nested.get(key)
                    if isinstance(value, str) and value.strip():
                        return value
    return TARGET_DEFAULTS[dataset]

TARGET_COLUMNS = {dataset: resolve_target(dataset) for dataset in DATASET_IDS}

GENERATION_CONFIG = {
    "framework": "SPP-GAN",
    "notebook": "Notebook 13",
    "datasets": DATASET_IDS,
    "master_seed": MASTER_SEED,
    "generation_repetitions": GENERATION_REPETITIONS,
    "sample_size_override": SAMPLE_SIZE_OVERRIDE,
    "batch_size": GENERATION_BATCH_SIZE,
    "save_parquet": SAVE_PARQUET,
    "target_columns": TARGET_COLUMNS,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}

GENERATION_CONFIG_PATH = NB13_CONFIG_ROOT / "sppgan_generation_configuration.json"
with open(GENERATION_CONFIG_PATH, "w", encoding="utf-8") as handle:
    json.dump(GENERATION_CONFIG, handle, indent=2, ensure_ascii=False)

print(json.dumps(GENERATION_CONFIG, indent=2))

{
  "framework": "SPP-GAN",
  "notebook": "Notebook 13",
  "datasets": [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
  ],
  "master_seed": 2025,
  "generation_repetitions": 1,
  "sample_size_override": null,
  "batch_size": 1024,
  "save_parquet": false,
  "target_columns": {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted"
  },
  "created_utc": "2026-10-16T06:57:37.136316+00:00"
}


In [10]:
# =============================================================================
# SECTION 7 — SET GENERATION SEEDS
# =============================================================================

def generation_seed(dataset, repetition=0):
    return MASTER_SEED + DATASET_IDS.index(dataset) * 10000 + repetition

def set_generation_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

SEED_PLAN = pd.DataFrame([
    {
        "dataset": dataset,
        "repetition": repetition,
        "seed": generation_seed(dataset, repetition),
    }
    for dataset in DATASET_IDS
    for repetition in range(GENERATION_REPETITIONS)
])

SEED_PLAN_PATH = NB13_METADATA_ROOT / "generation_seed_plan.csv"
SEED_PLAN.to_csv(SEED_PLAN_PATH, index=False)
display(SEED_PLAN)

,dataset,repetition,seed
0,adult_income,0,2025
1,bank_marketing,0,12025
2,diabetes_130us,0,22025


In [15]:

# ==================================================================================================
# SECTION 8 — GENERATE SYNTHETIC SAMPLES
# ==================================================================================================

print("=" * 100)
print("SECTION 8 — GENERATE SYNTHETIC SAMPLES")
print("=" * 100)

import gc
import json
import time
import hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from collections import OrderedDict

# --------------------------------------------------------------------------------
# 1. CONFIGURATION AND PRECHECKS
# --------------------------------------------------------------------------------

PROJECT_ROOT = Path("/content/drive/MyDrive/SPP_GAN_Research")
NB08_ARCH_ROOT = (
    PROJECT_ROOT / "results" / "notebooks" / "notebook_08" / "models"
)
NB13_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_13"
NB13_LOG_ROOT = NB13_ROOT / "logs"
NB13_VALIDATION_ROOT = NB13_ROOT / "validation"

NB13_LOG_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Generation device: {DEVICE}")

required_globals = [
    "DATASET_IDS",
    "MODEL_ARTIFACTS",
    "REFERENCE_TRAIN",
    "GENERATION_REPETITIONS",
    "GENERATION_BATCH_SIZE",
    "SAMPLE_SIZE_OVERRIDE",
    "MASTER_SEED",
]

missing_globals = [
    name for name in required_globals if name not in globals()
]

if missing_globals:
    raise RuntimeError(
        f"Missing required variables: {missing_globals}. "
        "Run the preceding Notebook 13 sections first."
    )

if not DATASET_IDS:
    raise ValueError("DATASET_IDS is empty.")

if len(set(DATASET_IDS)) != len(DATASET_IDS):
    raise ValueError("DATASET_IDS contains duplicates.")

if int(GENERATION_REPETITIONS) < 1:
    raise ValueError("GENERATION_REPETITIONS must be at least 1.")

if int(GENERATION_BATCH_SIZE) < 1:
    raise ValueError("GENERATION_BATCH_SIZE must be positive.")

if SAMPLE_SIZE_OVERRIDE is not None and int(SAMPLE_SIZE_OVERRIDE) <= 0:
    raise ValueError("SAMPLE_SIZE_OVERRIDE must be positive or None.")

# --------------------------------------------------------------------------------
# 2. DATASET ARCHITECTURE CONTRACTS
# --------------------------------------------------------------------------------
# These expected dimensions must agree with the architecture actually trained
# in Notebook 12. Do not modify them merely to bypass validation failures.

ARCHITECTURE_EXPECTATIONS = {
    "adult_income": {
        "latent_dim": 128,
        "hidden_dims": [256, 256],
        "num_numerical": 6,
        "categorical_cardinalities": [8, 16, 7, 14, 6, 5, 2, 41],
        "transformed_dim": 105,
    },
    "bank_marketing": {
        "latent_dim": 128,
        "hidden_dims": [256, 256],
        "num_numerical": 7,
        "categorical_cardinalities": [12, 3, 4, 2, 2, 2, 3, 12, 4],
        "transformed_dim": 51,
    },
    "diabetes_130us": {
        "latent_dim": 128,
        "hidden_dims": [256, 256],
        "num_numerical": 11,
        "categorical_cardinalities": [
            5, 3, 10, 9, 16, 69, 690, 704, 734,
            3, 3, 4, 4, 4, 4, 4, 1, 4, 4, 2,
            4, 4, 4, 4, 2, 2, 1, 1, 4, 4,
            2, 1, 2, 2, 2, 2,
        ],
        "transformed_dim": 2329,
    },
}

# --------------------------------------------------------------------------------
# 3. COMMON HELPERS
# --------------------------------------------------------------------------------

def normalized_name(value):
    return str(value).strip().lower()


def resolve_dataset_entry(container, dataset, label):
    if not isinstance(container, dict):
        raise TypeError(f"{label} must be a dictionary.")

    matches = [
        value for key, value in container.items()
        if normalized_name(key) == normalized_name(dataset)
    ]

    if len(matches) != 1:
        raise KeyError(
            f"Expected one {dataset!r} entry in {label}; "
            f"found {len(matches)}."
        )

    return matches[0]


def resolve_model_path(dataset, repetition_id):
    entries = resolve_dataset_entry(
        MODEL_ARTIFACTS,
        dataset,
        "MODEL_ARTIFACTS",
    )

    if isinstance(entries, (str, Path)):
        if int(GENERATION_REPETITIONS) != 1 or repetition_id != 1:
            raise ValueError(
                f"{dataset}: only one checkpoint path is registered."
            )
        return Path(entries)

    if not isinstance(entries, dict):
        raise TypeError(
            f"Unsupported model registry entry for {dataset}: "
            f"{type(entries).__name__}."
        )

    matches = [
        value for key, value in entries.items()
        if str(key).strip() == str(repetition_id)
    ]

    if len(matches) != 1:
        raise KeyError(
            f"Expected one checkpoint for {dataset}, "
            f"repetition {repetition_id}; found {len(matches)}."
        )

    return Path(matches[0])


def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def resolve_generation_seed(dataset, repetition_id):
    seed_plan = globals().get("SEED_PLAN")

    if isinstance(seed_plan, pd.DataFrame):
        required = {"dataset", "repetition", "seed"}

        if required.issubset(seed_plan.columns):
            rows = seed_plan[
                seed_plan["dataset"].astype(str).str.lower().eq(
                    dataset.lower()
                )
                & pd.to_numeric(
                    seed_plan["repetition"], errors="coerce"
                ).eq(repetition_id - 1)
            ]

            if len(rows) == 1:
                return int(rows.iloc[0]["seed"])

            if len(rows) > 1:
                raise ValueError(
                    f"Duplicate seed entries for {dataset}, "
                    f"repetition {repetition_id - 1}."
                )

    dataset_index = next(
        (
            index for index, name in enumerate(DATASET_IDS)
            if normalized_name(name) == normalized_name(dataset)
        ),
        None,
    )

    if dataset_index is None:
        raise KeyError(f"Dataset {dataset!r} is not registered.")

    return int(MASTER_SEED) + dataset_index * 10000 + repetition_id - 1


# --------------------------------------------------------------------------------
# 4. LOAD SECTION 4 VALIDATION REPORT
# --------------------------------------------------------------------------------

VALIDATION_REPORT_PATH = (
    NB13_VALIDATION_ROOT / "model_artifact_validation.csv"
)

if not VALIDATION_REPORT_PATH.is_file():
    raise FileNotFoundError(
        f"Section 4 validation report not found: "
        f"{VALIDATION_REPORT_PATH}"
    )

VALIDATION_REPORT = pd.read_csv(VALIDATION_REPORT_PATH)

if not {"dataset", "repetition_id"}.issubset(
    VALIDATION_REPORT.columns
):
    raise ValueError(
        "Validation report must contain dataset and repetition_id. "
        f"Columns: {VALIDATION_REPORT.columns.tolist()}"
    )

digest_candidates = [
    "sha256",
    "actual_sha256",
    "file_sha256",
    "final_model_sha256",
    "checkpoint_sha256",
]

DIGEST_COLUMN = next(
    (
        column for column in digest_candidates
        if column in VALIDATION_REPORT.columns
    ),
    None,
)

if DIGEST_COLUMN is None:
    raise ValueError(
        "No recognized SHA256 column exists in the Section 4 report."
    )

ACCEPTED_STATUSES = {
    "PASS", "PASSED", "VALID", "VALIDATED", "OK"
}


def get_validated_digest(dataset, repetition_id):
    rows = VALIDATION_REPORT[
        VALIDATION_REPORT["dataset"].astype(str).str.strip().str.lower().eq(
            dataset.lower()
        )
        & pd.to_numeric(
            VALIDATION_REPORT["repetition_id"], errors="coerce"
        ).eq(int(repetition_id))
    ]

    if len(rows) != 1:
        raise ValueError(
            f"Expected one validation row for {dataset}, "
            f"repetition {repetition_id}; found {len(rows)}."
        )

    row = rows.iloc[0]

    if "status" in VALIDATION_REPORT.columns:
        status = str(row["status"]).strip().upper()

        if status not in ACCEPTED_STATUSES:
            raise ValueError(
                f"Artifact validation status is not accepted: "
                f"{dataset}, repetition={repetition_id}, status={status!r}."
            )

    digest = str(row[DIGEST_COLUMN]).strip().lower()

    if (
        len(digest) != 64
        or any(character not in "0123456789abcdef" for character in digest)
    ):
        raise ValueError(
            f"Invalid SHA256 digest for {dataset}, "
            f"repetition {repetition_id}."
        )

    return digest


print("Section 4 validation report loaded.")
print(f"Validation rows: {len(VALIDATION_REPORT)}")
print(f"SHA256 column: {DIGEST_COLUMN}")

if "status" in VALIDATION_REPORT.columns:
    print(
        "Validation statuses:",
        sorted(VALIDATION_REPORT["status"].dropna().astype(str).unique()),
    )

# --------------------------------------------------------------------------------
# 5. LOAD CANONICAL ARCHITECTURE METADATA
# --------------------------------------------------------------------------------

def load_architecture(dataset):
    if dataset not in ARCHITECTURE_EXPECTATIONS:
        raise KeyError(
            f"No architecture contract registered for {dataset!r}."
        )

    path = NB08_ARCH_ROOT / dataset / "sppgan_architecture.json"

    if not path.is_file():
        raise FileNotFoundError(
            f"Canonical architecture file missing: {path}. "
            "Restore the correct architecture metadata before generation."
        )

    with path.open("r", encoding="utf-8") as file:
        metadata = json.load(file)

    def find_value(obj, keys):
        if isinstance(obj, dict):
            for key, value in obj.items():
                if key in keys and value is not None:
                    return value

            for value in obj.values():
                result = find_value(value, keys)
                if result is not None:
                    return result

        elif isinstance(obj, list):
            for value in obj:
                result = find_value(value, keys)
                if result is not None:
                    return result

        return None

    latent_dim = find_value(metadata, {"latent_dim"})
    hidden_dims = find_value(
        metadata,
        {"generator_hidden_dims", "hidden_dims"},
    )
    representation = find_value(metadata, {"input_representation"})

    if latent_dim is None:
        raise ValueError(f"{dataset}: latent_dim is missing.")

    if not isinstance(hidden_dims, list) or not hidden_dims:
        raise ValueError(f"{dataset}: generator hidden_dims are missing.")

    if not isinstance(representation, dict):
        raise ValueError(
            f"{dataset}: input_representation is missing or invalid."
        )

    cardinalities = representation.get("categorical_cardinalities")
    num_numerical = representation.get("num_numerical")
    transformed_dim = representation.get("logical_feature_dim")

    if not isinstance(cardinalities, list):
        raise ValueError(
            f"{dataset}: categorical_cardinalities is missing."
        )

    if num_numerical is None or transformed_dim is None:
        raise ValueError(
            f"{dataset}: numerical count or logical_feature_dim is missing."
        )

    contract = {
        "latent_dim": int(latent_dim),
        "hidden_dims": [int(value) for value in hidden_dims],
        "num_numerical": int(num_numerical),
        "categorical_cardinalities": [
            int(value) for value in cardinalities
        ],
        "transformed_dim": int(transformed_dim),
        "architecture_path": str(path),
    }

    expected = ARCHITECTURE_EXPECTATIONS[dataset]

    for key, expected_value in expected.items():
        if contract[key] != expected_value:
            raise ValueError(
                f"{dataset}: {key} mismatch. "
                f"Metadata={contract[key]!r}; expected={expected_value!r}."
            )

    calculated_dim = (
        contract["num_numerical"]
        + sum(contract["categorical_cardinalities"])
    )

    if calculated_dim != contract["transformed_dim"]:
        raise ValueError(
            f"{dataset}: calculated transformed width {calculated_dim} "
            f"does not equal declared width {contract['transformed_dim']}."
        )

    return contract


# --------------------------------------------------------------------------------
# 6. LOAD CHECKPOINT SAFELY
# --------------------------------------------------------------------------------
# This uses weights_only=True and a limited NumPy allowlist.
# It does not automatically execute arbitrary pickle payloads.

def load_checkpoint(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.is_file():
        raise FileNotFoundError(
            f"Checkpoint not found: {checkpoint_path}"
        )

    safe_globals = []

    # Use the current NumPy namespace first to avoid np.core deprecation.
    numpy_core = getattr(np, "_core", None)
    multiarray = getattr(numpy_core, "multiarray", None)

    if multiarray is None:
        raise RuntimeError(
            "NumPy _core.multiarray is unavailable. "
            "Check the installed NumPy version."
        )

    for name in ("_reconstruct", "scalar"):
        value = getattr(multiarray, name, None)
        if value is not None:
            safe_globals.append(value)

    safe_globals.extend([np.ndarray, np.dtype, np.uint32])

    dtypes_module = getattr(np, "dtypes", None)

    if dtypes_module is not None:
        for name in (
            "UInt8DType",
            "UInt16DType",
            "UInt32DType",
            "UInt64DType",
            "Int8DType",
            "Int16DType",
            "Int32DType",
            "Int64DType",
            "Float16DType",
            "Float32DType",
            "Float64DType",
            "BoolDType",
        ):
            dtype_class = getattr(dtypes_module, name, None)
            if dtype_class is not None:
                safe_globals.append(dtype_class)

    safe_globals = list({
        id(item): item for item in safe_globals
    }.values())

    try:
        with torch.serialization.safe_globals(safe_globals):
            checkpoint = torch.load(
                checkpoint_path,
                map_location="cpu",
                weights_only=True,
            )

    except Exception as exc:
        raise RuntimeError(
            f"Safe checkpoint loading failed: {checkpoint_path}\n"
            f"{type(exc).__name__}: {exc}\n"
            "Do not automatically switch to weights_only=False. "
            "Inspect Notebook 12's checkpoint serialization and allowlist "
            "only the specific additional type if its safety is understood."
        ) from exc

    if not isinstance(checkpoint, dict):
        raise TypeError(
            "Expected a dictionary checkpoint, received "
            f"{type(checkpoint).__name__}."
        )

    return checkpoint


# --------------------------------------------------------------------------------
# 7. DEFINE GENERATOR AND CHECKPOINT STRUCTURE
# --------------------------------------------------------------------------------

class SPPGANGenerator(nn.Module):
    def __init__(self, latent_dim, hidden_dims, transformed_dim):
        super().__init__()

        if len(hidden_dims) != 2:
            raise ValueError(
                "The current generator implementation expects two hidden layers."
            )

        self.network = nn.Sequential(
            nn.Linear(latent_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Linear(hidden_dims[1], transformed_dim),
        )

    def forward(self, z):
        return self.network(z)


def extract_generator_state_dict(checkpoint):
    """
    Recognize common checkpoint layouts without guessing model weights.
    """
    if not isinstance(checkpoint, dict):
        raise TypeError("Checkpoint must be a dictionary.")

    candidates = [
        "generator_state_dict",
        "generator",
        "G_state_dict",
        "G",
        "state_dict",
    ]

    for key in candidates:
        value = checkpoint.get(key)

        if isinstance(value, (dict, OrderedDict)) and value:
            if all(isinstance(k, str) for k in value):
                return value, key

    # Allow a checkpoint that is itself a plain state_dict.
    if checkpoint and all(
        isinstance(key, str) and torch.is_tensor(value)
        for key, value in checkpoint.items()
    ):
        return checkpoint, "<root-state-dict>"

    raise KeyError(
        "Could not identify a generator state_dict in the checkpoint. "
        f"Top-level keys: {list(checkpoint.keys())[:50]}. "
        "Inspect the save format in Notebook 12 and map the correct key."
    )


# --------------------------------------------------------------------------------
# 8. DECODE CATEGORICAL LOGITS
# --------------------------------------------------------------------------------

def decode_generator_output(raw_output, contract, seed):
    if raw_output.ndim != 2:
        raise ValueError(
            f"Expected 2D generator output, got {raw_output.shape}."
        )

    if not np.isfinite(raw_output).all():
        raise ValueError("Generator output contains NaN or infinity.")

    num_numerical = contract["num_numerical"]
    cardinalities = contract["categorical_cardinalities"]

    numerical = raw_output[:, :num_numerical]
    categorical_logits = raw_output[:, num_numerical:]

    expected_width = sum(cardinalities)

    if categorical_logits.shape[1] != expected_width:
        raise ValueError(
            f"Categorical width mismatch: received "
            f"{categorical_logits.shape[1]}, expected {expected_width}."
        )

    blocks = [numerical]
    rng = np.random.default_rng(int(seed))
    offset = 0

    for cardinality in cardinalities:
        logits = categorical_logits[:, offset:offset + cardinality]
        offset += cardinality

        logits = logits - logits.max(axis=1, keepdims=True)
        probabilities = np.exp(logits)
        totals = probabilities.sum(axis=1, keepdims=True)

        if (
            not np.isfinite(probabilities).all()
            or np.any(totals <= 0)
        ):
            raise ValueError("Invalid categorical probabilities.")

        probabilities /= totals

        categories = np.fromiter(
            (
                rng.choice(cardinality, p=row)
                for row in probabilities
            ),
            dtype=np.int64,
            count=len(probabilities),
        )

        one_hot = np.zeros(
            (len(raw_output), cardinality),
            dtype=np.float32,
        )
        one_hot[np.arange(len(raw_output)), categories] = 1.0
        blocks.append(one_hot)

    transformed = np.concatenate(blocks, axis=1).astype(
        np.float32,
        copy=False,
    )

    if transformed.shape[1] != contract["transformed_dim"]:
        raise ValueError(
            f"Decoded width {transformed.shape[1]} does not match "
            f"{contract['transformed_dim']}."
        )

    if not np.isfinite(transformed).all():
        raise ValueError("Decoded output contains NaN or infinity.")

    return transformed


# --------------------------------------------------------------------------------
# 9. GENERATE SYNTHETIC SAMPLES
# --------------------------------------------------------------------------------

GENERATED_ARRAYS = {}
GENERATION_RUNTIME = {}
GENERATION_ERRORS = []
GENERATION_METADATA = []

for dataset in DATASET_IDS:
    print("-" * 100)
    print(f"GENERATING: {dataset}")
    print("-" * 100)

    GENERATED_ARRAYS[dataset] = {}

    try:
        reference_df = resolve_dataset_entry(
            REFERENCE_TRAIN,
            dataset,
            "REFERENCE_TRAIN",
        )

        if not isinstance(reference_df, pd.DataFrame):
            raise TypeError(
                f"Reference training object for {dataset} is not a DataFrame."
            )

        if reference_df.empty:
            raise ValueError(f"Reference training data is empty for {dataset}.")

        contract = load_architecture(dataset)

        sample_count = (
            int(SAMPLE_SIZE_OVERRIDE)
            if SAMPLE_SIZE_OVERRIDE is not None
            else len(reference_df)
        )

        print(f"Native reference columns : {len(reference_df.columns)}")
        print(f"Transformed model width  : {contract['transformed_dim']}")
        print(f"Numerical features       : {contract['num_numerical']}")
        print(
            f"Categorical features     : "
            f"{len(contract['categorical_cardinalities'])}"
        )
        print(f"Requested samples        : {sample_count}")

        model_registry_entry = resolve_dataset_entry(
            MODEL_ARTIFACTS,
            dataset,
            "MODEL_ARTIFACTS",
        )

        if isinstance(model_registry_entry, dict):
            available_repetitions = sorted(
                int(key) for key in model_registry_entry
            )
            repetition_ids = available_repetitions[
                :int(GENERATION_REPETITIONS)
            ]
        else:
            repetition_ids = [1]

        if len(repetition_ids) < int(GENERATION_REPETITIONS):
            raise ValueError(
                f"{dataset}: requested {GENERATION_REPETITIONS} repetitions "
                f"but only {len(repetition_ids)} are registered."
            )

        for repetition_id in repetition_ids:
            generator = None
            checkpoint = None
            raw_batches = []

            try:
                model_path = resolve_model_path(dataset, repetition_id)

                if not model_path.is_file():
                    raise FileNotFoundError(
                        f"Checkpoint not found: {model_path}"
                    )

                expected_digest = get_validated_digest(
                    dataset,
                    repetition_id,
                )
                actual_digest = sha256_file(model_path)

                if actual_digest.lower() != expected_digest:
                    raise ValueError(
                        f"SHA256 mismatch for {dataset}, "
                        f"repetition {repetition_id}."
                    )

                seed = resolve_generation_seed(dataset, repetition_id)
                checkpoint = load_checkpoint(model_path)

                state_dict, state_dict_key = extract_generator_state_dict(
                    checkpoint
                )

                # Validate the expected parameter names and tensor dimensions.
                expected_keys = {
                    "network.0.weight",
                    "network.0.bias",
                    "network.2.weight",
                    "network.2.bias",
                    "network.4.weight",
                    "network.4.bias",
                }

                if set(state_dict.keys()) != expected_keys:
                    raise ValueError(
                        f"Generator state_dict keys do not match the "
                        f"expected architecture. Found: "
                        f"{list(state_dict.keys())[:30]}. "
                        "Confirm the generator class and layer names in "
                        "Notebook 12."
                    )

                latent_dim = int(state_dict["network.0.weight"].shape[1])
                hidden_dims = [
                    int(state_dict["network.0.weight"].shape[0]),
                    int(state_dict["network.2.weight"].shape[0]),
                ]
                output_dim = int(state_dict["network.4.weight"].shape[0])

                if latent_dim != contract["latent_dim"]:
                    raise ValueError(
                        f"Latent dimension mismatch: checkpoint={latent_dim}, "
                        f"metadata={contract['latent_dim']}."
                    )

                if hidden_dims != contract["hidden_dims"]:
                    raise ValueError(
                        f"Hidden dimensions mismatch: checkpoint={hidden_dims}, "
                        f"metadata={contract['hidden_dims']}."
                    )

                if output_dim != contract["transformed_dim"]:
                    raise ValueError(
                        f"Output width mismatch: checkpoint={output_dim}, "
                        f"metadata={contract['transformed_dim']}."
                    )

                generator = SPPGANGenerator(
                    latent_dim=latent_dim,
                    hidden_dims=hidden_dims,
                    transformed_dim=output_dim,
                ).to(DEVICE)

                generator.load_state_dict(state_dict, strict=True)
                generator.eval()

                latent_rng = torch.Generator(device="cpu")
                latent_rng.manual_seed(int(seed))

                start_time = time.perf_counter()

                with torch.inference_mode():
                    for start in range(
                        0,
                        sample_count,
                        int(GENERATION_BATCH_SIZE),
                    ):
                        batch_size = min(
                            int(GENERATION_BATCH_SIZE),
                            sample_count - start,
                        )

                        latent = torch.randn(
                            batch_size,
                            latent_dim,
                            generator=latent_rng,
                            device="cpu",
                        ).to(DEVICE)

                        batch_output = (
                            generator(latent).detach().cpu().numpy()
                        )

                        if not np.isfinite(batch_output).all():
                            raise ValueError(
                                "Generator produced NaN or infinity."
                            )

                        raw_batches.append(batch_output)

                raw_output = np.concatenate(raw_batches, axis=0)

                if raw_output.shape != (
                    sample_count,
                    contract["transformed_dim"],
                ):
                    raise ValueError(
                        f"Unexpected output shape: {raw_output.shape}."
                    )

                transformed_output = decode_generator_output(
                    raw_output,
                    contract,
                    seed,
                )

                elapsed = time.perf_counter() - start_time

                GENERATED_ARRAYS[dataset][repetition_id] = transformed_output
                GENERATION_RUNTIME[(dataset, repetition_id)] = elapsed

                GENERATION_METADATA.append({
                    "dataset": dataset,
                    "repetition_id": repetition_id,
                    "status": "PASS",
                    "sample_count": sample_count,
                    "native_reference_columns": len(reference_df.columns),
                    "transformed_dim": contract["transformed_dim"],
                    "num_numerical": contract["num_numerical"],
                    "num_categorical": len(
                        contract["categorical_cardinalities"]
                    ),
                    "latent_dim": latent_dim,
                    "seed": seed,
                    "device": str(DEVICE),
                    "runtime_seconds": elapsed,
                    "checkpoint_path": str(model_path),
                    "checkpoint_sha256": actual_digest,
                    "state_dict_key": state_dict_key,
                    "architecture_path": contract["architecture_path"],
                })

                print(
                    f"PASS | repetition={repetition_id} | "
                    f"shape={transformed_output.shape} | "
                    f"runtime={elapsed:.2f}s"
                )

                del raw_output, transformed_output

            except Exception as exc:
                GENERATION_ERRORS.append({
                    "dataset": dataset,
                    "repetition_id": repetition_id,
                    "error_type": type(exc).__name__,
                    "error_message": str(exc),
                })

                print(
                    f"GENERATION FAILED | repetition={repetition_id} | "
                    f"{type(exc).__name__}: {exc}"
                )

            finally:
                raw_batches.clear()

                if generator is not None:
                    del generator

                if checkpoint is not None:
                    del checkpoint

                gc.collect()

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

    except Exception as exc:
        GENERATION_ERRORS.append({
            "dataset": dataset,
            "repetition_id": None,
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        })

        print(f"DATASET FAILED | {type(exc).__name__}: {exc}")


# --------------------------------------------------------------------------------
# 10. SAVE LOGS AND VERIFY COMPLETION
# --------------------------------------------------------------------------------

ERROR_LOG_PATH = NB13_LOG_ROOT / "generation_errors.csv"
METADATA_LOG_PATH = NB13_LOG_ROOT / "generation_metadata.csv"

pd.DataFrame(
    GENERATION_ERRORS,
    columns=[
        "dataset",
        "repetition_id",
        "error_type",
        "error_message",
    ],
).to_csv(ERROR_LOG_PATH, index=False)

pd.DataFrame(GENERATION_METADATA).to_csv(
    METADATA_LOG_PATH,
    index=False,
)

successful_outputs = sum(
    len(outputs)
    for outputs in GENERATED_ARRAYS.values()
)

expected_outputs = len(DATASET_IDS) * int(GENERATION_REPETITIONS)

print("\n" + "=" * 100)
print("SECTION 8 GENERATION SUMMARY")
print("=" * 100)
print(f"Datasets requested : {len(DATASET_IDS)}")
print(f"Expected outputs   : {expected_outputs}")
print(f"Successful outputs : {successful_outputs}")
print(f"Generation errors  : {len(GENERATION_ERRORS)}")
print(f"Error log          : {ERROR_LOG_PATH}")
print(f"Metadata log       : {METADATA_LOG_PATH}")

for dataset in DATASET_IDS:
    for repetition_id, array in GENERATED_ARRAYS.get(dataset, {}).items():
        print(
            f"{dataset} | repetition={repetition_id} | "
            f"shape={array.shape}"
        )

if successful_outputs != expected_outputs or GENERATION_ERRORS:
    raise RuntimeError(
        "Section 8 did not complete successfully. Inspect "
        f"{ERROR_LOG_PATH} and resolve the reported errors before "
        "proceeding to inverse transformation."
    )

print(
    "Section 8 completed successfully. Generated arrays remain available "
    "in GENERATED_ARRAYS for the next notebook section."
)


SECTION 8 — GENERATE SYNTHETIC SAMPLES
Generation device: cuda
Section 4 validation report loaded.
Validation rows: 15
SHA256 column: sha256
Validation statuses: ['VALIDATED']
----------------------------------------------------------------------------------------------------
GENERATING: adult_income
----------------------------------------------------------------------------------------------------
Native reference columns : 16
Transformed model width  : 105
Numerical features       : 6
Categorical features     : 8
Requested samples        : 34189
PASS | repetition=1 | shape=(34189, 105) | runtime=11.55s
----------------------------------------------------------------------------------------------------
GENERATING: bank_marketing
----------------------------------------------------------------------------------------------------
Native reference columns : 18
Transformed model width  : 51
Numerical features       : 7
Categorical features     : 9
Requested samples        : 31647
PASS | 

In [20]:

# =============================================================================
# NOTEBOOK 13 — SECTION 9: INVERSE TRANSFORM AND SEMANTIC VALIDATION
# =============================================================================
# Purpose:
#   1. Validate Notebook 02 schema and fitted preprocessing artifacts.
#   2. Decode generated arrays into native feature space.
#   3. Preserve raw inverse-transformed synthetic data.
#   4. Apply documented, type-aware post-processing to a separate copy.
#   5. Save semantic diagnostics, post-processing changes, and metadata.
#
# Important:
#   - Never overwrite GENERATED_ARRAYS.
#   - Raw and post-processed datasets are separate artifacts.
#   - Range clipping uses observed training minima/maxima and may reduce fidelity.
#   - This section does not restore the target column; Section 13 handles that.
# =============================================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 9 — INVERSE TRANSFORM AND SEMANTIC VALIDATION")
print("=" * 100)

# =============================================================================
# 1. CONFIGURATION AND REQUIRED ARTIFACTS
# =============================================================================

REQUIRED_DATASETS = (
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
)

# Explicit post-processing policy.
# Original generated arrays and raw decoded data are never modified.
ROUND_INTEGER_LIKE_FEATURES = True
CLIP_TO_REFERENCE_RANGE = True
INTEGER_TOLERANCE = 1e-6

for required_name in (
    "GENERATED_ARRAYS",
    "PREPROCESSORS",
    "SCHEMAS",
    "REFERENCE_TRAIN",
    "PROJECT_ROOT",
    "NB02_ROOT",
    "NB13_LOG_ROOT",
):
    if required_name not in globals():
        raise RuntimeError(
            f"Required artifact is unavailable: {required_name}. "
            "Run the relevant earlier Notebook 13 sections first."
        )

NB13_LOG_ROOT = Path(NB13_LOG_ROOT)
NB13_LOG_ROOT.mkdir(parents=True, exist_ok=True)

RAW_OUTPUT_ROOT = (
    Path(PROJECT_ROOT)
    / "data"
    / "synthetic"
    / "notebook_13"
    / "raw_inverse_transformed"
)

POSTPROCESSED_OUTPUT_ROOT = (
    Path(PROJECT_ROOT)
    / "data"
    / "synthetic"
    / "notebook_13"
    / "postprocessed"
)

RAW_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
POSTPROCESSED_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if not isinstance(GENERATED_ARRAYS, dict) or not GENERATED_ARRAYS:
    raise RuntimeError(
        "GENERATED_ARRAYS is missing or empty. Run Section 8 first."
    )

# =============================================================================
# 2. EXTRACT GENERATED ARRAYS
# =============================================================================

def extract_generated_array(dataset, stored_value):
    """Extract one generated array without modifying the source artifact."""

    value = stored_value

    if isinstance(value, dict):
        repetition_id = 1

        if repetition_id in value:
            value = value[repetition_id]
        elif str(repetition_id) in value:
            value = value[str(repetition_id)]
        elif len(value) == 1:
            repetition_id, value = next(iter(value.items()))
            print(
                f"{dataset}: using stored repetition "
                f"{repetition_id!r}."
            )
        else:
            raise KeyError(
                f"{dataset}: cannot resolve repetition {repetition_id}; "
                f"available keys={list(value.keys())}."
            )

    if isinstance(value, pd.DataFrame):
        array = value.to_numpy()
    elif hasattr(value, "detach") and hasattr(value, "cpu"):
        array = value.detach().cpu().numpy()
    else:
        array = np.asarray(value)

    if array.ndim != 2 or array.shape[0] == 0:
        raise ValueError(
            f"{dataset}: expected a nonempty 2-D array; "
            f"received shape {array.shape}."
        )

    array = array.astype(np.float64, copy=False)

    if not np.isfinite(array).all():
        raise ValueError(
            f"{dataset}: generated array contains NaN or infinity."
        )

    return array.copy()


# =============================================================================
# 3. VALIDATE NOTEBOOK 02 SCHEMA AND PREPROCESSOR
# =============================================================================

def get_dataset_contract(dataset):
    schema = SCHEMAS.get(dataset)

    if not isinstance(schema, dict):
        raise TypeError(
            f"{dataset}: Notebook 02 schema is unavailable."
        )

    modeling = schema.get("modeling_schema", {})
    transformation = schema.get("transformation_schema", {})

    feature_columns = modeling.get("preprocessing_columns")
    numeric_columns = modeling.get("numeric_columns")
    categorical_columns = modeling.get("categorical_columns")
    transformed_names = transformation.get("transformed_feature_names")
    target_column = modeling.get("target_column")

    required_lists = (
        feature_columns,
        numeric_columns,
        categorical_columns,
        transformed_names,
    )

    if not all(isinstance(items, list) for items in required_lists):
        raise ValueError(
            f"{dataset}: required Notebook 02 schema lists are missing."
        )

    if len(feature_columns) != len(set(feature_columns)):
        raise ValueError(
            f"{dataset}: duplicate source feature names in schema."
        )

    if set(numeric_columns).intersection(categorical_columns):
        raise ValueError(
            f"{dataset}: numeric and categorical feature definitions overlap."
        )

    if set(numeric_columns).union(categorical_columns) != set(feature_columns):
        raise ValueError(
            f"{dataset}: numeric and categorical features do not cover "
            "the source feature list exactly."
        )

    preprocessor = PREPROCESSORS.get(dataset)

    if preprocessor is None:
        raise RuntimeError(
            f"{dataset}: fitted Notebook 02 preprocessor is unavailable."
        )

    if not hasattr(preprocessor, "feature_names_in_"):
        raise RuntimeError(
            f"{dataset}: fitted preprocessor has no feature_names_in_."
        )

    if not hasattr(preprocessor, "get_feature_names_out"):
        raise RuntimeError(
            f"{dataset}: preprocessor does not expose transformed feature names."
        )

    fitted_inputs = list(preprocessor.feature_names_in_)
    fitted_names = list(preprocessor.get_feature_names_out())

    if fitted_inputs != feature_columns:
        raise ValueError(
            f"{dataset}: fitted preprocessor input order disagrees "
            "with Notebook 02 schema."
        )

    if fitted_names != transformed_names:
        raise ValueError(
            f"{dataset}: fitted transformed feature order disagrees "
            "with Notebook 02 transformation_schema."
        )

    expected_width = int(
        transformation.get("transformed_feature_count", -1)
    )

    if expected_width != len(fitted_names):
        raise ValueError(
            f"{dataset}: transformed feature count disagrees with schema."
        )

    return {
        "schema": schema,
        "feature_columns": feature_columns,
        "numeric_columns": numeric_columns,
        "categorical_columns": categorical_columns,
        "transformed_names": transformed_names,
        "target_column": target_column,
        "preprocessor": preprocessor,
        "expected_width": expected_width,
    }


# =============================================================================
# 4. RESOLVE FITTED NUMERIC AND CATEGORICAL TRANSFORMERS
# =============================================================================

def get_fitted_components(dataset, contract):
    preprocessor = contract["preprocessor"]

    numeric_entry = None
    categorical_entry = None

    for _, transformer, columns in preprocessor.transformers_:
        column_list = (
            list(columns)
            if not isinstance(columns, str)
            else [columns]
        )

        if column_list == contract["numeric_columns"]:
            numeric_entry = transformer

        if column_list == contract["categorical_columns"]:
            categorical_entry = transformer

    if numeric_entry is None:
        raise RuntimeError(
            f"{dataset}: could not identify the fitted numeric transformer."
        )

    if categorical_entry is None:
        raise RuntimeError(
            f"{dataset}: could not identify the fitted categorical transformer."
        )

    numeric_steps = getattr(numeric_entry, "named_steps", {})
    categorical_steps = getattr(categorical_entry, "named_steps", {})

    scaler = numeric_steps.get("scaler")
    encoder = categorical_steps.get("encoder")

    if scaler is None or not hasattr(scaler, "inverse_transform"):
        raise RuntimeError(
            f"{dataset}: fitted numeric scaler with inverse_transform() "
            "was not found."
        )

    if encoder is None or not hasattr(encoder, "categories_"):
        raise RuntimeError(
            f"{dataset}: fitted OneHotEncoder categories were not found."
        )

    if len(encoder.categories_) != len(contract["categorical_columns"]):
        raise ValueError(
            f"{dataset}: encoder category count does not match "
            "categorical source feature count."
        )

    # Confirm the declared numeric-first, categorical-second layout.
    numeric_width = len(contract["numeric_columns"])
    categorical_width = sum(
        len(categories) for categories in encoder.categories_
    )

    if numeric_width + categorical_width != contract["expected_width"]:
        raise ValueError(
            f"{dataset}: fitted numeric and categorical widths do not "
            "match the transformed schema width."
        )

    return scaler, encoder


# =============================================================================
# 5. RESOLVE THE REFERENCE TRAINING DATA
# =============================================================================

def get_reference_frame(dataset, contract):
    """
    Resolve the native reference training DataFrame.

    Supports a DataFrame directly or a dictionary containing a training
    DataFrame under common keys. Does not use test data.
    """

    reference = REFERENCE_TRAIN.get(dataset)

    if isinstance(reference, pd.DataFrame):
        frame = reference

    elif isinstance(reference, dict):
        frame = None

        for key in ("train", "training", "X_train", "features"):
            candidate = reference.get(key)

            if isinstance(candidate, pd.DataFrame):
                frame = candidate
                break

        if frame is None:
            raise TypeError(
                f"{dataset}: REFERENCE_TRAIN dictionary does not contain "
                "a recognized training DataFrame."
            )

    else:
        raise TypeError(
            f"{dataset}: REFERENCE_TRAIN must contain a DataFrame or "
            "a dictionary with a training DataFrame."
        )

    missing = [
        column
        for column in contract["feature_columns"]
        if column not in frame.columns
    ]

    if missing:
        raise ValueError(
            f"{dataset}: reference training frame is missing features: "
            f"{missing}"
        )

    # Restrict to the source feature columns; the target is handled elsewhere.
    return frame[contract["feature_columns"]].copy()


# =============================================================================
# 6. INVERSE-TRANSFORM GENERATED ARRAYS
# =============================================================================

def inverse_transform_dataset(dataset, stored_value):
    array = extract_generated_array(dataset, stored_value)
    contract = get_dataset_contract(dataset)
    scaler, encoder = get_fitted_components(dataset, contract)

    if array.shape[1] != contract["expected_width"]:
        raise ValueError(
            f"{dataset}: generated width {array.shape[1]} does not match "
            f"Notebook 02 width {contract['expected_width']}."
        )

    numeric_columns = contract["numeric_columns"]
    categorical_columns = contract["categorical_columns"]

    numeric_width = len(numeric_columns)
    numeric_indices = list(range(numeric_width))

    # Notebook 02's fitted transformer order is numeric first, categorical
    # one-hot blocks second. Verify total widths before decoding.
    categorical_positions = {}
    cursor = numeric_width

    for feature, categories in zip(
        categorical_columns,
        encoder.categories_,
    ):
        block_width = len(categories)
        categorical_positions[feature] = (cursor, cursor + block_width)
        cursor += block_width

    if cursor != array.shape[1]:
        raise ValueError(
            f"{dataset}: transformed block widths total {cursor}, "
            f"but generated width is {array.shape[1]}."
        )

    numeric_scaled = array[:, numeric_indices]
    numeric_restored = scaler.inverse_transform(numeric_scaled)

    if numeric_restored.shape != (len(array), len(numeric_columns)):
        raise ValueError(
            f"{dataset}: inverse-scaled numeric array has an unexpected shape."
        )

    restored = pd.DataFrame(index=np.arange(len(array)))

    for index, feature in enumerate(numeric_columns):
        restored[feature] = numeric_restored[:, index]

    for index, feature in enumerate(categorical_columns):
        start, end = categorical_positions[feature]
        block = array[:, start:end]
        categories = np.asarray(encoder.categories_[index], dtype=object)

        if block.shape[1] != len(categories) or block.shape[1] == 0:
            raise ValueError(
                f"{dataset}/{feature}: categorical block width "
                "does not match fitted encoder categories."
            )

        # Decode using the fitted encoder vocabulary.
        category_indices = np.argmax(block, axis=1)
        restored[feature] = categories[category_indices]

    restored = restored[contract["feature_columns"]].reset_index(drop=True)

    if list(restored.columns) != contract["feature_columns"]:
        raise ValueError(
            f"{dataset}: restored feature order does not match Notebook 02."
        )

    if len(restored) != len(array):
        raise ValueError(
            f"{dataset}: row count changed during inverse transformation."
        )

    target_column = contract["target_column"]

    if target_column and target_column in restored.columns:
        raise ValueError(
            f"{dataset}: target column unexpectedly appeared in feature output."
        )

    return restored, array, contract


# =============================================================================
# 7. TYPE-AWARE POST-PROCESSING AND AUDIT
# =============================================================================

def postprocess_numeric_features(dataset, raw_frame, reference_frame, contract):
    """
    Create a separate post-processed copy.

    Integer-like status is inferred from the native reference training data:
    the non-null reference values must be numeric and effectively integral.

    Optional range clipping uses observed reference-training minima/maxima.
    Both rounding and clipping are counted and documented.
    """

    processed = raw_frame.copy(deep=True)
    actions = []
    integer_features = []
    range_features = []

    for feature in contract["numeric_columns"]:
        reference_values = pd.to_numeric(
            reference_frame[feature],
            errors="coerce",
        ).dropna()

        generated_values = pd.to_numeric(
            raw_frame[feature],
            errors="coerce",
        )

        if reference_values.empty:
            actions.append({
                "dataset": dataset,
                "feature": feature,
                "action": "SKIPPED",
                "reason": "No numeric reference values available.",
                "rounded_count": 0,
                "clipped_low_count": 0,
                "clipped_high_count": 0,
                "reference_min": np.nan,
                "reference_max": np.nan,
            })
            continue

        ref_array = reference_values.to_numpy(dtype=float)
        gen_array = generated_values.to_numpy(dtype=float)

        integer_like = bool(
            np.all(
                np.isclose(
                    ref_array,
                    np.round(ref_array),
                    atol=INTEGER_TOLERANCE,
                    rtol=0.0,
                )
            )
        )

        ref_min = float(np.min(ref_array))
        ref_max = float(np.max(ref_array))

        rounded_count = 0
        clipped_low_count = 0
        clipped_high_count = 0

        # Round only when the reference training feature is integer-like.
        if ROUND_INTEGER_LIKE_FEATURES and integer_like:
            finite_mask = np.isfinite(gen_array)
            before = gen_array.copy()

            gen_array[finite_mask] = np.rint(gen_array[finite_mask])

            rounded_count = int(
                np.sum(
                    finite_mask
                    & ~np.isclose(
                        before,
                        gen_array,
                        atol=INTEGER_TOLERANCE,
                        rtol=0.0,
                    )
                )
            )

            integer_features.append(feature)

        # Optional heuristic: constrain values to the observed training range.
        # Keep this setting consistent across all compared methods.
        if CLIP_TO_REFERENCE_RANGE:
            finite_mask = np.isfinite(gen_array)

            clipped_low_count = int(
                np.sum(finite_mask & (gen_array < ref_min))
            )
            clipped_high_count = int(
                np.sum(finite_mask & (gen_array > ref_max))
            )

            gen_array[finite_mask] = np.clip(
                gen_array[finite_mask],
                ref_min,
                ref_max,
            )

            range_features.append(feature)

        processed[feature] = gen_array

        actions.append({
            "dataset": dataset,
            "feature": feature,
            "action": "ROUND_AND_RANGE_CHECK",
            "reason": (
                "Integer rounding applied only when the reference training "
                "feature is integer-like; range clipping follows configuration."
            ),
            "rounded_count": rounded_count,
            "clipped_low_count": clipped_low_count,
            "clipped_high_count": clipped_high_count,
            "reference_min": ref_min,
            "reference_max": ref_max,
        })

    return processed, actions, integer_features, range_features


# =============================================================================
# 8. SEMANTIC DIAGNOSTICS
# =============================================================================

def build_semantic_diagnostics(
    dataset,
    raw_frame,
    processed_frame,
    reference_frame,
    contract,
):
    issues = []
    numeric_summary = []

    for feature in contract["numeric_columns"]:
        reference_values = pd.to_numeric(
            reference_frame[feature],
            errors="coerce",
        ).dropna()

        raw_values = pd.to_numeric(
            raw_frame[feature],
            errors="coerce",
        ).to_numpy(dtype=float)

        processed_values = pd.to_numeric(
            processed_frame[feature],
            errors="coerce",
        ).to_numpy(dtype=float)

        if reference_values.empty:
            continue

        ref_values = reference_values.to_numpy(dtype=float)
        ref_min = float(np.min(ref_values))
        ref_max = float(np.max(ref_values))

        integer_like = bool(
            np.all(
                np.isclose(
                    ref_values,
                    np.round(ref_values),
                    atol=INTEGER_TOLERANCE,
                    rtol=0.0,
                )
            )
        )

        finite_raw = np.isfinite(raw_values)
        non_integer_mask = (
            finite_raw
            & ~np.isclose(
                raw_values,
                np.round(raw_values),
                atol=INTEGER_TOLERANCE,
                rtol=0.0,
            )
        ) if integer_like else np.zeros(len(raw_values), dtype=bool)

        outside_mask = finite_raw & (
            (raw_values < ref_min) | (raw_values > ref_max)
        )

        non_integer_count = int(non_integer_mask.sum())
        outside_count = int(outside_mask.sum())

        if non_integer_count:
            issues.append({
                "dataset": dataset,
                "feature": feature,
                "issue": "NON_INTEGER_VALUES",
                "count": non_integer_count,
                "details": (
                    "Raw decoded values are fractional although the "
                    "reference training feature is integer-like."
                ),
            })

        if outside_count:
            issues.append({
                "dataset": dataset,
                "feature": feature,
                "issue": "OUTSIDE_REFERENCE_RANGE",
                "count": outside_count,
                "details": (
                    f"Raw decoded values fall outside "
                    f"[{ref_min}, {ref_max}]."
                ),
            })

        processed_finite = np.isfinite(processed_values)

        processed_non_integer = (
            int(
                (
                    processed_finite
                    & ~np.isclose(
                        processed_values,
                        np.round(processed_values),
                        atol=INTEGER_TOLERANCE,
                        rtol=0.0,
                    )
                ).sum()
            )
            if integer_like
            else 0
        )

        processed_outside = int(
            (
                processed_finite
                & (
                    (processed_values < ref_min)
                    | (processed_values > ref_max)
                )
            ).sum()
        )

        numeric_summary.append({
            "dataset": dataset,
            "feature": feature,
            "raw_generated_min": (
                float(np.nanmin(raw_values[finite_raw]))
                if finite_raw.any() else np.nan
            ),
            "raw_generated_max": (
                float(np.nanmax(raw_values[finite_raw]))
                if finite_raw.any() else np.nan
            ),
            "processed_min": (
                float(np.nanmin(processed_values[processed_finite]))
                if processed_finite.any() else np.nan
            ),
            "processed_max": (
                float(np.nanmax(processed_values[processed_finite]))
                if processed_finite.any() else np.nan
            ),
            "reference_min": ref_min,
            "reference_max": ref_max,
            "integer_like_in_reference": integer_like,
            "raw_non_integer_count": non_integer_count,
            "raw_outside_reference_range_count": outside_count,
            "processed_non_integer_count": processed_non_integer,
            "processed_outside_reference_range_count": processed_outside,
        })

    # Validate categorical values against the fitted encoder vocabulary.
    _, encoder = get_fitted_components(dataset, contract)

    for index, feature in enumerate(contract["categorical_columns"]):
        valid_categories = set(encoder.categories_[index].tolist())

        raw_values = raw_frame[feature].to_numpy(dtype=object)
        invalid_count = int(
            sum(value not in valid_categories for value in raw_values)
        )

        if invalid_count:
            issues.append({
                "dataset": dataset,
                "feature": feature,
                "issue": "INVALID_CATEGORICAL_VALUE",
                "count": invalid_count,
                "details": (
                    "Decoded value is not present in the fitted "
                    "OneHotEncoder vocabulary."
                ),
            })

    return issues, numeric_summary


# =============================================================================
# 9. PROCESS ALL DATASETS
# =============================================================================

RAW_SYNTHETIC_DATAFRAMES = {}
SYNTHETIC_DATAFRAMES = {}

INVERSE_TRANSFORM_ERRORS = []
INVERSE_TRANSFORM_METADATA = []
POSTPROCESSING_ACTIONS = []
SEMANTIC_ISSUES = []
NUMERIC_SUMMARIES = []

for dataset in REQUIRED_DATASETS:
    print("-" * 100)
    print("DATASET:", dataset)

    if dataset not in GENERATED_ARRAYS:
        message = "Generated output is missing from GENERATED_ARRAYS."

        INVERSE_TRANSFORM_ERRORS.append({
            "dataset": dataset,
            "error": message,
        })

        INVERSE_TRANSFORM_METADATA.append({
            "dataset": dataset,
            "status": "FAIL",
            "error": message,
        })

        print("FAIL:", message)
        continue

    try:
        raw_frame, generated_array, contract = inverse_transform_dataset(
            dataset,
            GENERATED_ARRAYS[dataset],
        )

        reference_frame = get_reference_frame(dataset, contract)

        processed_frame, actions, integer_features, range_features = (
            postprocess_numeric_features(
                dataset,
                raw_frame,
                reference_frame,
                contract,
            )
        )

        issues, numeric_summary = build_semantic_diagnostics(
            dataset,
            raw_frame,
            processed_frame,
            reference_frame,
            contract,
        )

        # Store separate in-memory artifacts.
        RAW_SYNTHETIC_DATAFRAMES[dataset] = raw_frame.copy(deep=True)
        SYNTHETIC_DATAFRAMES[dataset] = processed_frame.copy(deep=True)

        POSTPROCESSING_ACTIONS.extend(actions)
        SEMANTIC_ISSUES.extend(issues)
        NUMERIC_SUMMARIES.extend(numeric_summary)

        # Save feature-only outputs. Target restoration is deferred.
        raw_path = RAW_OUTPUT_ROOT / f"{dataset}_synthetic_raw.csv"
        processed_path = (
            POSTPROCESSED_OUTPUT_ROOT
            / f"{dataset}_synthetic_postprocessed.csv"
        )

        raw_frame.to_csv(raw_path, index=False)
        processed_frame.to_csv(processed_path, index=False)

        INVERSE_TRANSFORM_METADATA.append({
            "dataset": dataset,
            "status": "PASS",
            "transformed_rows": int(generated_array.shape[0]),
            "transformed_width": int(generated_array.shape[1]),
            "native_feature_rows": int(raw_frame.shape[0]),
            "native_feature_columns": int(raw_frame.shape[1]),
            "raw_csv": str(raw_path),
            "postprocessed_csv": str(processed_path),
            "round_integer_like_features": ROUND_INTEGER_LIKE_FEATURES,
            "clip_to_reference_range": CLIP_TO_REFERENCE_RANGE,
            "integer_like_features": json.dumps(integer_features),
            "range_checked_features": json.dumps(range_features),
            "semantic_issue_count": int(
                sum(item["count"] for item in issues)
            ),
            "target_deferred_to_section_13": contract["target_column"],
            "error": None,
        })

        print("PASS | Raw native shape:", raw_frame.shape)
        print("PASS | Post-processed shape:", processed_frame.shape)
        print("PASS | Notebook 02 column order validated.")
        print("Integer-like features:", len(integer_features))
        print("Range-checked features:", len(range_features))
        print("Raw semantic issue entries:", len(issues))
        print("Raw output:", raw_path)
        print("Post-processed output:", processed_path)
        print("Target restoration is deferred to Section 13.")

    except Exception as exc:
        message = f"{type(exc).__name__}: {exc}"

        INVERSE_TRANSFORM_ERRORS.append({
            "dataset": dataset,
            "error": message,
        })

        INVERSE_TRANSFORM_METADATA.append({
            "dataset": dataset,
            "status": "FAIL",
            "error": message,
        })

        print("FAIL:", message)


# =============================================================================
# 10. SAVE DIAGNOSTICS AND POST-PROCESSING AUDIT
# =============================================================================

pd.DataFrame(
    INVERSE_TRANSFORM_ERRORS,
    columns=["dataset", "error"],
).to_csv(
    NB13_LOG_ROOT / "inverse_transform_errors.csv",
    index=False,
)

pd.DataFrame(INVERSE_TRANSFORM_METADATA).to_csv(
    NB13_LOG_ROOT / "inverse_transform_metadata.csv",
    index=False,
)

pd.DataFrame(POSTPROCESSING_ACTIONS).to_csv(
    NB13_LOG_ROOT / "postprocessing_actions.csv",
    index=False,
)

pd.DataFrame(
    SEMANTIC_ISSUES,
    columns=["dataset", "feature", "issue", "count", "details"],
).to_csv(
    NB13_LOG_ROOT / "inverse_transform_semantic_issues.csv",
    index=False,
)

pd.DataFrame(NUMERIC_SUMMARIES).to_csv(
    NB13_LOG_ROOT / "inverse_transform_numeric_summary.csv",
    index=False,
)

# Save the explicit policy used for this run.
policy = {
    "section": "Notebook 13 Section 9",
    "round_integer_like_features": ROUND_INTEGER_LIKE_FEATURES,
    "clip_to_reference_range": CLIP_TO_REFERENCE_RANGE,
    "integer_tolerance": INTEGER_TOLERANCE,
    "reference_for_type_and_range": "native reference training data",
    "raw_output_preserved": True,
    "target_restoration": "deferred to Section 13",
    "privacy_claim": (
        "This section does not establish end-to-end differential privacy."
    ),
}

with open(
    NB13_LOG_ROOT / "section_9_postprocessing_policy.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(policy, file, indent=2)


# =============================================================================
# 11. COMPLETION SUMMARY
# =============================================================================

print("=" * 100)
print("SECTION 9 SUMMARY")
print("=" * 100)

print("Expected datasets:", len(REQUIRED_DATASETS))
print("Successful inverse transforms:", len(RAW_SYNTHETIC_DATAFRAMES))
print("Failures:", len(INVERSE_TRANSFORM_ERRORS))
print("Raw datasets:", RAW_OUTPUT_ROOT)
print("Post-processed datasets:", POSTPROCESSED_OUTPUT_ROOT)
print("Error log:", NB13_LOG_ROOT / "inverse_transform_errors.csv")
print("Metadata log:", NB13_LOG_ROOT / "inverse_transform_metadata.csv")
print("Post-processing audit:", NB13_LOG_ROOT / "postprocessing_actions.csv")
print("Semantic issues:", NB13_LOG_ROOT / "inverse_transform_semantic_issues.csv")
print("Numeric summary:", NB13_LOG_ROOT / "inverse_transform_numeric_summary.csv")
print("Policy:", NB13_LOG_ROOT / "section_9_postprocessing_policy.json")

if INVERSE_TRANSFORM_ERRORS:
    print(
        "\nWARNING: Resolve Section 9 failures before running later sections."
    )
else:
    print(
        "\nPASS: All datasets were inverse-transformed and saved."
    )
    print(
        "NOTE: SYNTHETIC_DATAFRAMES contains the post-processed feature-only "
        "data; RAW_SYNTHETIC_DATAFRAMES preserves the uncorrected decoding."
    )
    print(
        "IMPORTANT: Range clipping is a heuristic post-processing operation. "
        "Evaluate raw and post-processed data separately for fidelity and utility."
    )
    print(
        "LIMITATION: This section does not establish end-to-end "
        "differential privacy."
    )


SECTION 9 — INVERSE TRANSFORM AND SEMANTIC VALIDATION
----------------------------------------------------------------------------------------------------
DATASET: adult_income
PASS | Raw native shape: (34189, 14)
PASS | Post-processed shape: (34189, 14)
PASS | Notebook 02 column order validated.
Integer-like features: 6
Range-checked features: 6
Raw semantic issue entries: 0
Raw output: /content/drive/MyDrive/SPP_GAN_Research/data/synthetic/notebook_13/raw_inverse_transformed/adult_income_synthetic_raw.csv
Post-processed output: /content/drive/MyDrive/SPP_GAN_Research/data/synthetic/notebook_13/postprocessed/adult_income_synthetic_postprocessed.csv
Target restoration is deferred to Section 13.
----------------------------------------------------------------------------------------------------
DATASET: bank_marketing
PASS | Raw native shape: (31647, 16)
PASS | Post-processed shape: (31647, 16)
PASS | Notebook 02 column order validated.
Integer-like features: 7
Range-checked features: 7

In [21]:
# =============================================================================
# SECTION 10 — RESTORE FEATURE TYPES
# =============================================================================

print("=" * 100)
print("SECTION 10 — RESTORE FEATURE TYPES")
print("=" * 100)

import pandas as pd
import numpy as np

def restore_feature_types(dataset, dataframe):
    result = dataframe.copy()
    reference = REFERENCE_TRAIN.get(dataset)

    if reference is None:
        raise ValueError(
            f"{dataset}: Notebook 02 reference training data is unavailable."
        )

    # Validate feature alignment; the target is restored in Section 13.
    expected_columns = [
        column for column in reference.columns
        if column in result.columns
    ]

    if list(result.columns) != expected_columns:
        raise ValueError(
            f"{dataset}: feature columns do not match the expected order. "
            f"Expected {expected_columns}, got {list(result.columns)}."
        )

    for column in result.columns:
        source_dtype = reference[column].dtype

        try:
            if pd.api.types.is_bool_dtype(source_dtype):
                values = pd.to_numeric(result[column], errors="coerce")
                result[column] = values.round().clip(0, 1).astype("Int64")

            elif pd.api.types.is_integer_dtype(source_dtype):
                values = pd.to_numeric(result[column], errors="coerce")
                result[column] = values.round().astype("Int64")

            elif pd.api.types.is_float_dtype(source_dtype):
                result[column] = pd.to_numeric(
                    result[column], errors="coerce"
                ).astype("float64")

            elif isinstance(source_dtype, pd.CategoricalDtype):
                result[column] = result[column].astype("object")

            elif (
                pd.api.types.is_object_dtype(source_dtype)
                or pd.api.types.is_string_dtype(source_dtype)
            ):
                result[column] = result[column].astype("object")

        except (TypeError, ValueError, OverflowError) as exc:
            raise ValueError(
                f"{dataset}/{column}: feature type restoration failed: {exc}"
            ) from exc

    # Confirm restoration did not alter row count or column order.
    if len(result) != len(dataframe):
        raise RuntimeError(f"{dataset}: row count changed during type restoration.")

    if list(result.columns) != list(dataframe.columns):
        raise RuntimeError(f"{dataset}: column order changed during type restoration.")

    return result


TYPE_RESTORATION_ERRORS = []

for dataset in list(SYNTHETIC_DATAFRAMES):
    try:
        before = SYNTHETIC_DATAFRAMES[dataset]

        SYNTHETIC_DATAFRAMES[dataset] = restore_feature_types(
            dataset, before
        )

        after = SYNTHETIC_DATAFRAMES[dataset]

        print(
            f"PASS | {dataset} | "
            f"Rows: {len(after):,} | "
            f"Features: {len(after.columns)}"
        )

    except Exception as exc:
        TYPE_RESTORATION_ERRORS.append({
            "dataset": dataset,
            "error": str(exc)
        })
        print(f"FAIL | {dataset}: {exc}")

print("-" * 100)
print(f"Expected datasets: {len(REFERENCE_TRAIN)}")
print(f"Successful restorations: {len(SYNTHETIC_DATAFRAMES) - len(TYPE_RESTORATION_ERRORS)}")
print(f"Failures: {len(TYPE_RESTORATION_ERRORS)}")

if TYPE_RESTORATION_ERRORS:
    error_path = NB13_LOG_ROOT / "feature_type_restoration_errors.csv"
    pd.DataFrame(TYPE_RESTORATION_ERRORS).to_csv(error_path, index=False)
    print(f"Error log: {error_path}")
    raise RuntimeError(
        "Section 10 failed. Resolve feature type restoration errors before proceeding."
    )

print("PASS: Feature types restored without changing row counts or column order.")
print("NOTE: Semantic validity and target restoration are assessed in later sections.")

SECTION 10 — RESTORE FEATURE TYPES
PASS | adult_income | Rows: 34,189 | Features: 14
PASS | bank_marketing | Rows: 31,647 | Features: 16
PASS | diabetes_130us | Rows: 71,236 | Features: 47
----------------------------------------------------------------------------------------------------
Expected datasets: 3
Successful restorations: 3
Failures: 0
PASS: Feature types restored without changing row counts or column order.
NOTE: Semantic validity and target restoration are assessed in later sections.


In [22]:
# =============================================================================
# SECTION 11 — RESTORE CATEGORIES
# =============================================================================

print("=" * 100)
print("SECTION 11 — RESTORE CATEGORIES")
print("=" * 100)

import pandas as pd

CATEGORY_RESTORATION_ERRORS = []

def restore_categories(dataset, dataframe):
    result = dataframe.copy()
    reference = REFERENCE_TRAIN.get(dataset)

    if reference is None:
        raise ValueError(
            f"{dataset}: Notebook 02 reference training data is unavailable."
        )

    if list(result.columns) != [
        column for column in reference.columns if column in result.columns
    ]:
        raise ValueError(
            f"{dataset}: feature column order does not match the reference data."
        )

    for column in result.columns:
        source_dtype = reference[column].dtype

        if isinstance(source_dtype, pd.CategoricalDtype):
            valid_categories = list(reference[column].cat.categories)
            result[column] = pd.Categorical(
                result[column],
                categories=valid_categories,
                ordered=source_dtype.ordered,
            )

        elif (
            pd.api.types.is_object_dtype(source_dtype)
            or pd.api.types.is_string_dtype(source_dtype)
        ):
            # Preserve decoded category labels without inventing new categories.
            result[column] = result[column].astype("object")

    return result


for dataset in list(SYNTHETIC_DATAFRAMES):
    try:
        before = SYNTHETIC_DATAFRAMES[dataset]

        SYNTHETIC_DATAFRAMES[dataset] = restore_categories(
            dataset, before
        )

        after = SYNTHETIC_DATAFRAMES[dataset]

        if len(after) != len(before):
            raise RuntimeError("Row count changed during category restoration.")

        if list(after.columns) != list(before.columns):
            raise RuntimeError("Column order changed during category restoration.")

        print(
            f"PASS | {dataset} | "
            f"Rows: {len(after):,} | "
            f"Features: {len(after.columns)}"
        )

    except Exception as exc:
        CATEGORY_RESTORATION_ERRORS.append({
            "dataset": dataset,
            "error": str(exc),
        })
        print(f"FAIL | {dataset}: {exc}")


print("-" * 100)
print(f"Datasets processed: {len(SYNTHETIC_DATAFRAMES)}")
print(
    "Successful restorations: "
    f"{len(SYNTHETIC_DATAFRAMES) - len(CATEGORY_RESTORATION_ERRORS)}"
)
print(f"Failures: {len(CATEGORY_RESTORATION_ERRORS)}")

if CATEGORY_RESTORATION_ERRORS:
    error_path = NB13_LOG_ROOT / "category_restoration_errors.csv"
    pd.DataFrame(CATEGORY_RESTORATION_ERRORS).to_csv(
        error_path, index=False
    )
    print(f"Error log: {error_path}")
    raise RuntimeError(
        "Section 11 failed. Resolve category restoration errors before proceeding."
    )

print("PASS: Categorical dtypes and reference category definitions restored where applicable.")
print("NOTE: This step does not force generated values into valid categories; category validity is checked separately.")

SECTION 11 — RESTORE CATEGORIES
PASS | adult_income | Rows: 34,189 | Features: 14
PASS | bank_marketing | Rows: 31,647 | Features: 16
PASS | diabetes_130us | Rows: 71,236 | Features: 47
----------------------------------------------------------------------------------------------------
Datasets processed: 3
Successful restorations: 3
Failures: 0
PASS: Categorical dtypes and reference category definitions restored where applicable.
NOTE: This step does not force generated values into valid categories; category validity is checked separately.


In [23]:
# =============================================================================
# SECTION 12 — RESTORE NUMERICAL VALUES
# =============================================================================

print("=" * 100)
print("SECTION 12 — RESTORE NUMERICAL VALUES")
print("=" * 100)

import pandas as pd
import numpy as np

NUMERICAL_RESTORATION_ERRORS = []

def restore_numerical_values(dataset, dataframe):
    result = dataframe.copy()
    reference = REFERENCE_TRAIN.get(dataset)

    if reference is None:
        raise ValueError(
            f"{dataset}: Notebook 02 reference training data is unavailable."
        )

    if list(result.columns) != [
        column for column in reference.columns if column in result.columns
    ]:
        raise ValueError(
            f"{dataset}: feature column order does not match reference data."
        )

    for column in result.columns:
        source_dtype = reference[column].dtype

        if (
            pd.api.types.is_numeric_dtype(source_dtype)
            and not pd.api.types.is_bool_dtype(source_dtype)
        ):
            values = pd.to_numeric(result[column], errors="coerce")

            # Restore integer-valued features without silently clipping values.
            if pd.api.types.is_integer_dtype(source_dtype):
                finite_mask = values.notna() & np.isfinite(values)

                if finite_mask.any():
                    values.loc[finite_mask] = values.loc[finite_mask].round()

                # Nullable integer dtype preserves any generated NaN values.
                result[column] = values.astype("Int64")

            else:
                result[column] = values.astype("float64")

    return result


for dataset in list(SYNTHETIC_DATAFRAMES):
    try:
        before = SYNTHETIC_DATAFRAMES[dataset]

        SYNTHETIC_DATAFRAMES[dataset] = restore_numerical_values(
            dataset, before
        )

        after = SYNTHETIC_DATAFRAMES[dataset]

        if len(after) != len(before):
            raise RuntimeError("Row count changed during numerical restoration.")

        if list(after.columns) != list(before.columns):
            raise RuntimeError("Column order changed during numerical restoration.")

        print(
            f"PASS | {dataset} | "
            f"Rows: {len(after):,} | "
            f"Features: {len(after.columns)}"
        )

    except Exception as exc:
        NUMERICAL_RESTORATION_ERRORS.append({
            "dataset": dataset,
            "error": str(exc),
        })
        print(f"FAIL | {dataset}: {exc}")


print("-" * 100)
print(f"Datasets processed: {len(SYNTHETIC_DATAFRAMES)}")
print(
    "Successful restorations: "
    f"{len(SYNTHETIC_DATAFRAMES) - len(NUMERICAL_RESTORATION_ERRORS)}"
)
print(f"Failures: {len(NUMERICAL_RESTORATION_ERRORS)}")

if NUMERICAL_RESTORATION_ERRORS:
    error_path = NB13_LOG_ROOT / "numerical_restoration_errors.csv"
    pd.DataFrame(NUMERICAL_RESTORATION_ERRORS).to_csv(
        error_path, index=False
    )
    print(f"Error log: {error_path}")
    raise RuntimeError(
        "Section 12 failed. Resolve numerical restoration errors before proceeding."
    )

print("PASS: Numeric columns restored using reference dtypes.")
print("NOTE: Integer rounding restores representation only; it does not establish valid ranges or semantic correctness.")

SECTION 12 — RESTORE NUMERICAL VALUES
PASS | adult_income | Rows: 34,189 | Features: 14
PASS | bank_marketing | Rows: 31,647 | Features: 16
PASS | diabetes_130us | Rows: 71,236 | Features: 47
----------------------------------------------------------------------------------------------------
Datasets processed: 3
Successful restorations: 3
Failures: 0
PASS: Numeric columns restored using reference dtypes.
NOTE: Integer rounding restores representation only; it does not establish valid ranges or semantic correctness.


In [25]:

# =============================================================================
# NOTEBOOK 13 — SECTION 13: TARGET RESTORATION
# Classifier Comparison, Target Assignment, and Distribution Diagnostics
# =============================================================================

import json
import time
import hashlib
import warnings
import traceback
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

print("=" * 100)
print("NOTEBOOK 13 — SECTION 13: TARGET RESTORATION")
print("=" * 100)

# =============================================================================
# 13.1 — CONFIGURATION AND PATHS
# =============================================================================

DATASET_TARGETS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

EXPECTED_DATASETS = list(DATASET_TARGETS)
MASTER_SEED = int(globals().get("MASTER_SEED", 2025))
VALIDATION_SIZE = 0.20

MAX_ITERATIONS = {
    "adult_income": 2000,
    "bank_marketing": 2000,
    "diabetes_130us": 3000,
}

# Used only to exclude known record identifiers from model predictors.
IDENTIFIER_COLUMNS = {
    "id", "row_id", "record_id", "patient_id", "customer_id",
    "encounter_id", "__original_row_id__",
}

# Diagnostic threshold, not a formal hypothesis-test threshold.
TV_DISTANCE_REVIEW_THRESHOLD = 0.10

required_globals = [
    "NB13_ROOT",
    "SYNTHETIC_DATAFRAMES",
    "REFERENCE_TRAIN",
]

for name in required_globals:
    if name not in globals():
        raise NameError(
            f"{name} is missing. Run the preceding Notebook 13 sections first."
        )

NB13_ROOT = Path(NB13_ROOT)
SYNTHETIC_DIR = NB13_ROOT / "synthetic_datasets"
METADATA_DIR = NB13_ROOT / "metadata"
VALIDATION_DIR = NB13_ROOT / "validation"
LOG_DIR = NB13_ROOT / "logs"

for directory in [
    SYNTHETIC_DIR, METADATA_DIR, VALIDATION_DIR, LOG_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

OUTPUT_VALIDATION = VALIDATION_DIR / "target_classifier_validation.csv"
OUTPUT_DISTRIBUTIONS = VALIDATION_DIR / "target_distribution_comparison.csv"
OUTPUT_REPORTS = VALIDATION_DIR / "target_classifier_classification_reports.json"
OUTPUT_METADATA = METADATA_DIR / "target_restoration_metadata.json"
OUTPUT_ERRORS = LOG_DIR / "target_restoration_errors.csv"
OUTPUT_VERIFICATION = VALIDATION_DIR / "target_restoration_final_verification.csv"

print("Datasets    :", EXPECTED_DATASETS)
print("Targets     :", DATASET_TARGETS)
print("Output root :", NB13_ROOT)
print("Master seed :", MASTER_SEED)
print("Validation  :", f"{VALIDATION_SIZE:.0%} held-out real data")
print("TV review threshold:", TV_DISTANCE_REVIEW_THRESHOLD)

# =============================================================================
# 13.2 — INPUT VALIDATION
# =============================================================================

for dataset in EXPECTED_DATASETS:
    if dataset not in SYNTHETIC_DATAFRAMES:
        raise KeyError(f"Missing synthetic feature dataset: {dataset}")

    if dataset not in REFERENCE_TRAIN:
        raise KeyError(f"Missing reference dataset: {dataset}")

    synthetic_df = SYNTHETIC_DATAFRAMES[dataset]
    reference_df = REFERENCE_TRAIN[dataset]
    target = DATASET_TARGETS[dataset]

    if not isinstance(synthetic_df, pd.DataFrame):
        raise TypeError(f"{dataset}: synthetic data must be a DataFrame.")

    if not isinstance(reference_df, pd.DataFrame):
        raise TypeError(f"{dataset}: reference data must be a DataFrame.")

    if target not in reference_df.columns:
        raise KeyError(f"{dataset}: target '{target}' not found in reference data.")

    if target in synthetic_df.columns:
        raise ValueError(
            f"{dataset}: target '{target}' already exists in synthetic features. "
            "Check the earlier Notebook 13 sections."
        )

    if not synthetic_df.columns.is_unique:
        raise ValueError(f"{dataset}: synthetic feature names are not unique.")

    if not reference_df.columns.is_unique:
        raise ValueError(f"{dataset}: reference column names are not unique.")

    print(
        f"{dataset:<18} reference={reference_df.shape} "
        f"synthetic_features={synthetic_df.shape}"
    )

# =============================================================================
# 13.3 — HELPER FUNCTIONS
# =============================================================================

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def dataframe_sha256(df):
    digest = hashlib.sha256()
    digest.update(
        json.dumps(
            {
                "columns": [str(c) for c in df.columns],
                "dtypes": [str(dtype) for dtype in df.dtypes],
                "shape": list(df.shape),
            },
            sort_keys=True,
        ).encode("utf-8")
    )
    digest.update(
        pd.util.hash_pandas_object(
            df, index=True, categorize=True
        ).values.tobytes()
    )
    return digest.hexdigest()


def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}

    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]

    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value) if np.isfinite(value) else None

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()

    if value is None:
        return None

    if isinstance(value, (str, int, float, bool)):
        if isinstance(value, float) and not np.isfinite(value):
            return None
        return value

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    return str(value)


def make_onehot():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
            dtype=np.float64,
        )
    except TypeError:
        # Compatibility with older scikit-learn releases.
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=True,
            dtype=np.float64,
        )


def prepare_features(reference_df, synthetic_df, target):
    """
    Align synthetic predictors to real predictors without changing row order.
    Target is excluded from predictors and restored only after prediction.
    """
    reference_predictors = reference_df.drop(columns=[target]).copy()
    synthetic_predictors = synthetic_df.copy()

    reference_ids = [
        c for c in reference_predictors.columns
        if str(c).strip().lower() in IDENTIFIER_COLUMNS
    ]
    synthetic_ids = [
        c for c in synthetic_predictors.columns
        if str(c).strip().lower() in IDENTIFIER_COLUMNS
    ]

    reference_predictors.drop(columns=reference_ids, inplace=True)
    synthetic_predictors.drop(columns=synthetic_ids, inplace=True)

    missing = [
        c for c in reference_predictors.columns
        if c not in synthetic_predictors.columns
    ]
    extra = [
        c for c in synthetic_predictors.columns
        if c not in reference_predictors.columns
    ]

    if missing or extra:
        raise ValueError(
            f"Feature schema mismatch. Missing={missing}; extra={extra}"
        )

    synthetic_predictors = synthetic_predictors[
        reference_predictors.columns
    ]

    if not all(isinstance(c, str) for c in reference_predictors.columns):
        raise TypeError("Feature column names must be strings.")

    # Normalize categorical dtype handling between real and synthetic data.
    for column in reference_predictors.columns:
        source_dtype = reference_predictors[column].dtype

        is_categorical = (
            pd.api.types.is_object_dtype(source_dtype)
            or isinstance(source_dtype, pd.CategoricalDtype)
            or pd.api.types.is_string_dtype(source_dtype)
        )

        if is_categorical:
            reference_predictors[column] = (
                reference_predictors[column].astype("object")
            )
            synthetic_predictors[column] = (
                synthetic_predictors[column].astype("object")
            )

    all_missing = reference_predictors.columns[
        reference_predictors.isna().all()
    ].tolist()

    if all_missing:
        raise ValueError(
            f"Reference features entirely missing: {all_missing}"
        )

    return reference_predictors, synthetic_predictors


def make_pipeline(X, dataset, class_weight):
    numeric_columns = X.select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()

    categorical_columns = [
        column for column in X.columns
        if column not in numeric_columns
    ]

    transformers = []

    if numeric_columns:
        transformers.append(
            (
                "numeric",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_columns,
            )
        )

    if categorical_columns:
        transformers.append(
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(strategy="most_frequent"),
                        ),
                        ("onehot", make_onehot()),
                    ]
                ),
                categorical_columns,
            )
        )

    if not transformers:
        raise ValueError("No usable predictors remain.")

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=1.0,
    )

    classifier = SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=0.0001,
        max_iter=MAX_ITERATIONS[dataset],
        tol=1e-3,
        random_state=MASTER_SEED,
        average=True,
        class_weight=class_weight,
    )

    model = Pipeline(
        [
            ("preprocessor", preprocessor),
            ("classifier", classifier),
        ]
    )

    return model, numeric_columns, categorical_columns


def tv_distance(reference_target, synthetic_target):
    """Total variation distance between empirical class distributions."""
    p = reference_target.astype(str).value_counts(normalize=True)
    q = synthetic_target.astype(str).value_counts(normalize=True)
    labels = p.index.union(q.index)
    p = p.reindex(labels, fill_value=0.0)
    q = q.reindex(labels, fill_value=0.0)
    return float(0.5 * np.abs(p - q).sum())


def distribution_records(dataset, target, reference_y, synthetic_y):
    p = reference_y.astype(str).value_counts()
    q = synthetic_y.astype(str).value_counts()
    pp = reference_y.astype(str).value_counts(normalize=True)
    qq = synthetic_y.astype(str).value_counts(normalize=True)

    labels = sorted(set(p.index) | set(q.index))
    rows = []

    for label in labels:
        reference_prop = float(pp.get(label, 0.0))
        synthetic_prop = float(qq.get(label, 0.0))

        rows.append(
            {
                "dataset": dataset,
                "target": target,
                "class_label": label,
                "reference_count": int(p.get(label, 0)),
                "synthetic_count": int(q.get(label, 0)),
                "reference_proportion": reference_prop,
                "synthetic_proportion": synthetic_prop,
                "proportion_difference": (
                    synthetic_prop - reference_prop
                ),
                "absolute_proportion_difference": abs(
                    synthetic_prop - reference_prop
                ),
            }
        )

    return rows


def get_distribution_diagnostics(reference_y, synthetic_y):
    distance = tv_distance(reference_y, synthetic_y)

    reference_counts = reference_y.value_counts()
    minority_label = reference_counts.idxmin()

    minority_reference_share = float(
        (reference_y == minority_label).mean()
    )
    minority_synthetic_share = float(
        (synthetic_y == minority_label).mean()
    )

    missing_prediction_classes = [
        str(label)
        for label in reference_y.unique()
        if not (synthetic_y == label).any()
    ]

    return {
        "tv_distance": distance,
        "minority_class": str(minority_label),
        "minority_reference_share": minority_reference_share,
        "minority_synthetic_share": minority_synthetic_share,
        "classes_with_zero_predictions": missing_prediction_classes,
        "distribution_review_flag": bool(
            distance > TV_DISTANCE_REVIEW_THRESHOLD
            or missing_prediction_classes
        ),
    }


# =============================================================================
# 13.4 — INITIALIZE REGISTRIES
# =============================================================================

TARGET_RESTORATION_RESULTS = {}
TARGET_RESTORATION_METADATA = {}
TARGET_CLASSIFICATION_REPORTS = {}
TARGET_VALIDATION_ROWS = []
TARGET_DISTRIBUTION_ROWS = []
TARGET_RESTORATION_ERRORS = []

# =============================================================================
# 13.5 — CANDIDATE EVALUATION AND TARGET ASSIGNMENT
# =============================================================================

for dataset_index, dataset in enumerate(EXPECTED_DATASETS):
    print("\n" + "=" * 100)
    print(f"DATASET: {dataset}")
    print("=" * 100)

    start_time = time.perf_counter()

    try:
        target = DATASET_TARGETS[dataset]
        dataset_seed = MASTER_SEED + dataset_index

        reference = REFERENCE_TRAIN[dataset].copy()
        synthetic_features = SYNTHETIC_DATAFRAMES[dataset].copy()

        reference_rows_before_filter = len(reference)

        # Remove rows with missing reference targets only.
        reference = reference.loc[reference[target].notna()].copy()

        if reference.empty:
            raise ValueError("Reference data has no non-missing target rows.")

        y = reference[target].copy()

        if y.nunique() < 2:
            raise ValueError(
                "Classification requires at least two target classes."
            )

        X, Xs = prepare_features(
            reference, synthetic_features, target
        )

        if len(X) != len(y):
            raise AssertionError("Reference predictor/target row mismatch.")

        # Use the same held-out real-data split for both candidate models.
        counts = y.value_counts()
        stratify = y if counts.min() >= 2 else None

        X_train, X_valid, y_train, y_valid = train_test_split(
            X,
            y,
            test_size=VALIDATION_SIZE,
            random_state=dataset_seed,
            stratify=stratify,
        )

        candidates = {
            "unweighted": None,
            "balanced": "balanced",
        }

        candidate_results = {}
        candidate_models = {}

        for candidate_name, class_weight in candidates.items():
            model, numeric_cols, categorical_cols = make_pipeline(
                X_train,
                dataset,
                class_weight,
            )

            with warnings.catch_warnings(record=True) as captured:
                warnings.simplefilter("always")
                model.fit(X_train, y_train)

            fit_warnings = [str(w.message) for w in captured]
            validation_prediction = model.predict(X_valid)

            report = classification_report(
                y_valid,
                validation_prediction,
                output_dict=True,
                zero_division=0,
            )

            macro_f1 = f1_score(
                y_valid,
                validation_prediction,
                average="macro",
                zero_division=0,
            )

            balanced_accuracy = balanced_accuracy_score(
                y_valid, validation_prediction
            )

            accuracy = accuracy_score(
                y_valid, validation_prediction
            )

            classifier_classes = list(
                model.named_steps["classifier"].classes_
            )

            candidate_results[candidate_name] = {
                "accuracy": float(accuracy),
                "balanced_accuracy": float(balanced_accuracy),
                "macro_f1": float(macro_f1),
                "weighted_f1": float(
                    f1_score(
                        y_valid,
                        validation_prediction,
                        average="weighted",
                        zero_division=0,
                    )
                ),
                "classification_report": json_safe(report),
                "confusion_matrix": confusion_matrix(
                    y_valid,
                    validation_prediction,
                    labels=classifier_classes,
                ).tolist(),
                "class_labels": [
                    str(value) for value in classifier_classes
                ],
                "fit_warnings": fit_warnings,
                "class_weight": class_weight,
            }

            candidate_models[candidate_name] = model

            print(
                f"{candidate_name:<12} "
                f"accuracy={accuracy:.4f} | "
                f"balanced_accuracy={balanced_accuracy:.4f} | "
                f"macro_f1={macro_f1:.4f} | "
                f"warnings={len(fit_warnings)}"
            )

        # Select by held-out macro F1, then balanced accuracy.
        # This selects a predictive model, not the most faithful synthetic labels.
        selected_name = max(
            candidate_results,
            key=lambda name: (
                candidate_results[name]["macro_f1"],
                candidate_results[name]["balanced_accuracy"],
                name == "unweighted",
            ),
        )

        selected_validation = candidate_results[selected_name]
        selected_class_weight = candidates[selected_name]

        print(f"\nSelected candidate: {selected_name}")
        print(
            "Selection criterion: validation macro F1, "
            "then balanced accuracy."
        )

        # Refit selected model on all available real reference rows.
        final_model, numeric_cols, categorical_cols = make_pipeline(
            X,
            dataset,
            selected_class_weight,
        )

        with warnings.catch_warnings(record=True) as captured:
            warnings.simplefilter("always")
            final_model.fit(X, y)

        final_fit_warnings = [str(w.message) for w in captured]

        # Predict labels without artificially forcing class proportions.
        predicted_target = final_model.predict(Xs)

        if len(predicted_target) != len(synthetic_features):
            raise AssertionError(
                "Predicted target count differs from synthetic row count."
            )

        restored = synthetic_features.copy()
        restored[target] = predicted_target

        if restored[target].isna().any():
            raise ValueError("Target assignment produced missing values.")

        if len(restored) != len(synthetic_features):
            raise AssertionError(
                "Synthetic row count changed during target restoration."
            )

        if list(restored.columns) != (
            list(synthetic_features.columns) + [target]
        ):
            raise AssertionError(
                "Restored output columns differ from expected feature-plus-target order."
            )

        diagnostics = get_distribution_diagnostics(y, restored[target])
        distance = diagnostics["tv_distance"]

        reference_distribution = (
            y.astype(str).value_counts(normalize=True).to_dict()
        )
        synthetic_distribution = (
            restored[target].astype(str).value_counts(normalize=True).to_dict()
        )

        distribution_rows = distribution_records(
            dataset, target, y, restored[target]
        )
        TARGET_DISTRIBUTION_ROWS.extend(distribution_rows)

        output_csv = SYNTHETIC_DIR / (
            f"{dataset}_synthetic_with_predicted_target.csv"
        )

        restored.to_csv(
            output_csv, index=False, encoding="utf-8"
        )

        file_hash = sha256_file(output_csv)
        dataframe_hash = dataframe_sha256(restored)
        runtime = time.perf_counter() - start_time

        TARGET_RESTORATION_RESULTS[dataset] = restored

        TARGET_CLASSIFICATION_REPORTS[dataset] = {
            "selected_candidate": selected_name,
            "candidate_comparison": candidate_results,
            "selected_validation_report": selected_validation[
                "classification_report"
            ],
            "selected_confusion_matrix": selected_validation[
                "confusion_matrix"
            ],
            "selected_confusion_matrix_labels": selected_validation[
                "class_labels"
            ],
            "final_fit_warnings": final_fit_warnings,
        }

        metadata = {
            "dataset": dataset,
            "target": target,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "master_seed": MASTER_SEED,
            "dataset_seed": dataset_seed,
            "validation_size": VALIDATION_SIZE,
            "candidate_models": list(candidates.keys()),
            "selection_metric": "validation_macro_f1",
            "selected_candidate": selected_name,
            "selected_class_weight": selected_class_weight,
            "candidate_validation_metrics": {
                name: {
                    "accuracy": result["accuracy"],
                    "balanced_accuracy": result["balanced_accuracy"],
                    "macro_f1": result["macro_f1"],
                    "weighted_f1": result["weighted_f1"],
                }
                for name, result in candidate_results.items()
            },
            "reference_rows_before_target_filter": (
                reference_rows_before_filter
            ),
            "reference_rows_used": len(reference),
            "synthetic_rows": len(restored),
            "predictor_count": X.shape[1],
            "numeric_predictor_count": len(numeric_cols),
            "categorical_predictor_count": len(categorical_cols),
            "reference_target_distribution": reference_distribution,
            "synthetic_target_distribution": synthetic_distribution,
            "target_distribution_tv_distance": distance,
            "tv_distance_review_threshold": TV_DISTANCE_REVIEW_THRESHOLD,
            "minority_reference_class": diagnostics["minority_class"],
            "minority_reference_proportion": (
                diagnostics["minority_reference_share"]
            ),
            "minority_synthetic_proportion": (
                diagnostics["minority_synthetic_share"]
            ),
            "classes_with_zero_predictions": (
                diagnostics["classes_with_zero_predictions"]
            ),
            "distribution_review_flag": (
                diagnostics["distribution_review_flag"]
            ),
            "selected_validation_warnings": (
                selected_validation["fit_warnings"]
            ),
            "final_fit_warnings": final_fit_warnings,
            "reference_dataframe_sha256": dataframe_sha256(reference),
            "synthetic_features_dataframe_sha256": (
                dataframe_sha256(synthetic_features)
            ),
            "synthetic_output_dataframe_sha256": dataframe_hash,
            "synthetic_output_file_sha256": file_hash,
            "output_csv": str(output_csv),
            "runtime_seconds": runtime,
            "execution_status": "PASS",
            "target_assignment_method": (
                "Classifier-assigned; target not directly generated by SPP-GAN"
            ),
            "end_to_end_differential_privacy_established": False,
            "scikit_learn_version": sklearn.__version__,
            "numpy_version": np.__version__,
            "pandas_version": pd.__version__,
        }

        TARGET_RESTORATION_METADATA[dataset] = metadata

        TARGET_VALIDATION_ROWS.append(
            {
                "dataset": dataset,
                "target": target,
                "selected_candidate": selected_name,
                "reference_rows": len(reference),
                "synthetic_rows": len(restored),
                "predictor_count": X.shape[1],
                "validation_accuracy": selected_validation["accuracy"],
                "validation_balanced_accuracy": (
                    selected_validation["balanced_accuracy"]
                ),
                "validation_macro_f1": selected_validation["macro_f1"],
                "validation_weighted_f1": (
                    selected_validation["weighted_f1"]
                ),
                "unweighted_validation_macro_f1": (
                    candidate_results["unweighted"]["macro_f1"]
                ),
                "balanced_validation_macro_f1": (
                    candidate_results["balanced"]["macro_f1"]
                ),
                "target_distribution_tv_distance": distance,
                "tv_distance_review_threshold": (
                    TV_DISTANCE_REVIEW_THRESHOLD
                ),
                "minority_reference_class": diagnostics["minority_class"],
                "minority_reference_proportion": (
                    diagnostics["minority_reference_share"]
                ),
                "minority_synthetic_proportion": (
                    diagnostics["minority_synthetic_share"]
                ),
                "zero_prediction_classes": json.dumps(
                    diagnostics["classes_with_zero_predictions"]
                ),
                "distribution_review_flag": (
                    diagnostics["distribution_review_flag"]
                ),
                "validation_warning_count": len(
                    selected_validation["fit_warnings"]
                ),
                "final_fit_warning_count": len(final_fit_warnings),
                "runtime_seconds": runtime,
                "output_csv": str(output_csv),
                "output_sha256": file_hash,
                "execution_status": "PASS",
            }
        )

        print(f"Reference rows           : {len(reference):,}")
        print(f"Synthetic rows           : {len(restored):,}")
        print(f"Target                   : {target}")
        print(f"Predictor count          : {X.shape[1]}")
        print(
            f"Selected validation F1   : "
            f"{selected_validation['macro_f1']:.4f}"
        )
        print(
            f"Selected balanced acc.   : "
            f"{selected_validation['balanced_accuracy']:.4f}"
        )
        print(f"Distribution TV distance: {distance:.4f}")
        print(
            f"Minority class           : "
            f"{diagnostics['minority_class']}"
        )
        print(
            f"Reference minority share : "
            f"{diagnostics['minority_reference_share']:.4%}"
        )
        print(
            f"Synthetic minority share : "
            f"{diagnostics['minority_synthetic_share']:.4%}"
        )
        print(
            "Zero-prediction classes  : "
            f"{diagnostics['classes_with_zero_predictions']}"
        )
        print("\nPredicted target distribution:")
        print(restored[target].value_counts(dropna=False))
        print(f"Saved                    : {output_csv}")
        print(f"SHA-256                  : {file_hash}")
        print(f"Runtime                  : {runtime:.2f}s")
        print("EXECUTION STATUS         : PASS")
        print(
            "FIDELITY DIAGNOSTIC      : "
            + (
                "REVIEW REQUIRED"
                if diagnostics["distribution_review_flag"]
                else "NO AUTOMATIC FLAG"
            )
        )

    except Exception as error:
        TARGET_RESTORATION_ERRORS.append(
            {
                "dataset": dataset,
                "error_type": type(error).__name__,
                "error_message": str(error),
                "traceback": traceback.format_exc(),
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            }
        )

        print(f"STATUS: FAIL — {type(error).__name__}: {error}")
        print(traceback.format_exc())

# =============================================================================
# 13.6 — SAVE VALIDATION, DISTRIBUTION, REPORT AND ERROR ARTIFACTS
# =============================================================================

validation_df = pd.DataFrame(TARGET_VALIDATION_ROWS)
distribution_df = pd.DataFrame(TARGET_DISTRIBUTION_ROWS)
errors_df = pd.DataFrame(TARGET_RESTORATION_ERRORS)

validation_df.to_csv(OUTPUT_VALIDATION, index=False)
distribution_df.to_csv(OUTPUT_DISTRIBUTIONS, index=False)
errors_df.to_csv(OUTPUT_ERRORS, index=False)

with open(OUTPUT_REPORTS, "w", encoding="utf-8") as handle:
    json.dump(
        json_safe(TARGET_CLASSIFICATION_REPORTS),
        handle,
        indent=2,
        ensure_ascii=False,
    )

metadata_payload = {
    "section": "Notebook 13 — Section 13: Target Restoration",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "master_seed": MASTER_SEED,
    "expected_datasets": EXPECTED_DATASETS,
    "successful_datasets": list(TARGET_RESTORATION_RESULTS.keys()),
    "failed_datasets": [
        dataset for dataset in EXPECTED_DATASETS
        if dataset not in TARGET_RESTORATION_RESULTS
    ],
    "dataset_metadata": TARGET_RESTORATION_METADATA,
    "validation_csv": str(OUTPUT_VALIDATION),
    "distribution_comparison_csv": str(OUTPUT_DISTRIBUTIONS),
    "classification_reports_json": str(OUTPUT_REPORTS),
    "errors_csv": str(OUTPUT_ERRORS),
    "tv_distance_review_threshold": TV_DISTANCE_REVIEW_THRESHOLD,
    "research_limitations": [
        "Targets are classifier-assigned, not directly generated by SPP-GAN.",
        "Held-out real-data classifier metrics do not establish synthetic joint-distribution fidelity.",
        "Target distribution similarity does not establish feature-target dependency fidelity.",
        "This section does not establish end-to-end differential privacy.",
    ],
}

with open(OUTPUT_METADATA, "w", encoding="utf-8") as handle:
    json.dump(
        json_safe(metadata_payload),
        handle,
        indent=2,
        ensure_ascii=False,
    )

# =============================================================================
# 13.7 — FINAL ARTIFACT VERIFICATION
# =============================================================================

print("\n" + "=" * 100)
print("SECTION 13 — FINAL VERIFICATION")
print("=" * 100)

verification_rows = []

for dataset in EXPECTED_DATASETS:
    target = DATASET_TARGETS[dataset]

    if dataset not in TARGET_RESTORATION_RESULTS:
        verification_rows.append(
            {
                "dataset": dataset,
                "status": "FAIL",
                "details": json.dumps(
                    {"restoration_result_available": False}
                ),
            }
        )
        print(f"FAIL | {dataset:<18} | no output dataset")
        continue

    restored = TARGET_RESTORATION_RESULTS[dataset]
    original_features = SYNTHETIC_DATAFRAMES[dataset]
    output_csv = SYNTHETIC_DIR / (
        f"{dataset}_synthetic_with_predicted_target.csv"
    )

    checks = {
        "target_present": target in restored.columns,
        "target_complete": (
            target in restored.columns
            and not restored[target].isna().any()
        ),
        "row_count_preserved": (
            len(restored) == len(original_features)
        ),
        "feature_columns_preserved": (
            list(restored.columns[:-1]) == list(original_features.columns)
        ),
        "target_is_last_column": (
            len(restored.columns) > 0
            and restored.columns[-1] == target
        ),
        "output_exists": output_csv.exists(),
    }

    try:
        reloaded = pd.read_csv(output_csv)

        checks["saved_shape_matches"] = (
            reloaded.shape == restored.shape
        )
        checks["saved_columns_match"] = (
            list(reloaded.columns) == list(restored.columns)
        )
        checks["saved_target_complete"] = (
            target in reloaded.columns
            and not reloaded[target].isna().any()
        )
        checks["sha256_matches"] = (
            sha256_file(output_csv)
            == TARGET_RESTORATION_METADATA[dataset][
                "synthetic_output_file_sha256"
            ]
        )
    except Exception as error:
        checks["saved_shape_matches"] = False
        checks["saved_columns_match"] = False
        checks["saved_target_complete"] = False
        checks["sha256_matches"] = False
        checks["reload_error"] = str(error)

    passed = all(
        value for key, value in checks.items()
        if key != "reload_error"
    )

    verification_rows.append(
        {
            "dataset": dataset,
            "status": "PASS" if passed else "FAIL",
            "details": json.dumps(checks, sort_keys=True),
        }
    )

    print(
        f"{'PASS' if passed else 'FAIL'} | {dataset:<18} "
        f"| shape={restored.shape} | target={target} "
        f"| missing={int(restored[target].isna().sum())}"
    )

verification_df = pd.DataFrame(verification_rows)
verification_df.to_csv(OUTPUT_VERIFICATION, index=False)

successful = len(TARGET_RESTORATION_RESULTS)
failed = len(EXPECTED_DATASETS) - successful
verified = (
    int((verification_df["status"] == "PASS").sum())
    if not verification_df.empty else 0
)

review_datasets = [
    dataset
    for dataset, metadata in TARGET_RESTORATION_METADATA.items()
    if metadata["distribution_review_flag"]
]

print("\nExpected datasets       :", len(EXPECTED_DATASETS))
print("Successful restorations :", successful)
print("Failed restorations     :", failed)
print("Verification PASS       :", verified)
print("Validation CSV          :", OUTPUT_VALIDATION)
print("Distribution comparison :", OUTPUT_DISTRIBUTIONS)
print("Classification reports  :", OUTPUT_REPORTS)
print("Metadata JSON           :", OUTPUT_METADATA)
print("Error log               :", OUTPUT_ERRORS)
print("Verification CSV        :", OUTPUT_VERIFICATION)

if failed == 0 and verified == len(EXPECTED_DATASETS):
    SECTION_13_STATUS = "PASS — execution and artifacts verified"
else:
    SECTION_13_STATUS = "FAIL — inspect error and verification logs"

print("\nSECTION 13 STATUS:", SECTION_13_STATUS)
print("FIDELITY REVIEW REQUIRED:", review_datasets)

print(
    "\nRESEARCH LIMITATION: Targets are classifier-assigned. "
    "Classifier validation metrics do not prove synthetic joint-distribution "
    "fidelity or end-to-end differential privacy."
)

# Public output for subsequent Notebook 13 sections.
SYNTHETIC_DATAFRAMES_WITH_TARGET = TARGET_RESTORATION_RESULTS

print("\nAvailable output variable: SYNTHETIC_DATAFRAMES_WITH_TARGET")
print("=" * 100)

# Stop downstream execution if any dataset or artifact failed verification.
if SECTION_13_STATUS.startswith("FAIL"):
    raise RuntimeError(
        "Section 13 verification failed. Resolve errors before proceeding."
    )


NOTEBOOK 13 — SECTION 13: TARGET RESTORATION
Datasets    : ['adult_income', 'bank_marketing', 'diabetes_130us']
Targets     : {'adult_income': 'income', 'bank_marketing': 'y', 'diabetes_130us': 'readmitted'}
Output root : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13
Master seed : 2025
Validation  : 20% held-out real data
TV review threshold: 0.1
adult_income       reference=(34189, 16) synthetic_features=(34189, 14)
bank_marketing     reference=(31647, 18) synthetic_features=(31647, 16)
diabetes_130us     reference=(71236, 49) synthetic_features=(71236, 47)

DATASET: adult_income
unweighted   accuracy=0.8511 | balanced_accuracy=0.7653 | macro_f1=0.7818 | warnings=1
balanced     accuracy=0.8099 | balanced_accuracy=0.8187 | macro_f1=0.7715 | warnings=1

Selected candidate: unweighted
Selection criterion: validation macro F1, then balanced accuracy.
Reference rows           : 34,189
Synthetic rows           : 34,189
Target                   : income
Predictor coun

In [26]:

# =============================================================================
# SECTION 14 — VALIDATE SCHEMA
# SPP-GAN | Validate restored synthetic datasets against reference schemas
# =============================================================================

import json
import pandas as pd
from pathlib import Path

print("=" * 100)
print("SECTION 14 — VALIDATE SCHEMA")
print("=" * 100)

# -----------------------------------------------------------------------------
# 14.1 — CONFIGURATION
# -----------------------------------------------------------------------------

if "REFERENCE_TRAIN" not in globals():
    raise NameError("REFERENCE_TRAIN is missing. Run the earlier data-loading sections.")

if "SYNTHETIC_DATAFRAMES_WITH_TARGET" not in globals():
    raise NameError(
        "SYNTHETIC_DATAFRAMES_WITH_TARGET is missing. "
        "Run the updated Section 13 first."
    )

if "DATASET_TARGETS" not in globals():
    DATASET_TARGETS = {
        "adult_income": "income",
        "bank_marketing": "y",
        "diabetes_130us": "readmitted",
    }

if "NB13_VALIDATION_ROOT" not in globals():
    NB13_VALIDATION_ROOT = Path(NB13_ROOT) / "validation"

NB13_VALIDATION_ROOT = Path(NB13_VALIDATION_ROOT)
NB13_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_DATASETS = list(DATASET_TARGETS)

# This identifier is intentionally not generated by the model.
IDENTIFIER_COLUMNS = {"__original_row_id__"}

# -----------------------------------------------------------------------------
# 14.2 — SCHEMA VALIDATION FUNCTION
# -----------------------------------------------------------------------------

def validate_schema(dataset, dataframe):
    reference = REFERENCE_TRAIN.get(dataset)

    if reference is None:
        return {
            "dataset": dataset,
            "schema_status": "NO_REFERENCE",
            "expected_columns": None,
            "actual_columns": len(dataframe.columns),
            "missing_columns": None,
            "unexpected_columns": None,
            "duplicate_columns": None,
            "target_present": False,
            "identifier_policy": "Not evaluated",
            "schema_pass": False,
        }

    target = DATASET_TARGETS[dataset]

    expected_columns = list(reference.columns)
    actual_columns = list(dataframe.columns)

    # A source identifier is not expected to be synthesized.
    expected_model_columns = [
        column for column in expected_columns
        if column not in IDENTIFIER_COLUMNS
    ]

    missing = [
        column for column in expected_model_columns
        if column not in actual_columns
    ]

    unexpected = [
        column for column in actual_columns
        if column not in expected_model_columns
    ]

    duplicate_columns = [
        str(column)
        for column in dataframe.columns[dataframe.columns.duplicated()].tolist()
    ]

    target_present = target in actual_columns

    # Verify column order as well as column membership.
    expected_order = expected_model_columns
    actual_order = actual_columns
    column_order_matches = actual_order == expected_order

    schema_pass = (
        not missing
        and not unexpected
        and not duplicate_columns
        and target_present
        and column_order_matches
    )

    return {
        "dataset": dataset,
        "schema_status": "PASS" if schema_pass else "FAIL",
        "expected_columns": len(expected_model_columns),
        "actual_columns": len(actual_columns),
        "missing_columns": json.dumps(missing),
        "unexpected_columns": json.dumps(unexpected),
        "duplicate_columns": json.dumps(duplicate_columns),
        "target_column": target,
        "target_present": target_present,
        "identifier_policy": (
            "Excluded from expected synthetic schema: "
            "__original_row_id__ is not generated"
        ),
        "column_order_matches": column_order_matches,
        "schema_pass": schema_pass,
    }

# -----------------------------------------------------------------------------
# 14.3 — VALIDATE ALL RESTORED DATASETS
# -----------------------------------------------------------------------------

SCHEMA_VALIDATION_ROWS = []

for dataset in EXPECTED_DATASETS:
    if dataset not in SYNTHETIC_DATAFRAMES_WITH_TARGET:
        SCHEMA_VALIDATION_ROWS.append({
            "dataset": dataset,
            "schema_status": "MISSING_OUTPUT",
            "expected_columns": None,
            "actual_columns": None,
            "missing_columns": None,
            "unexpected_columns": None,
            "duplicate_columns": None,
            "target_column": DATASET_TARGETS[dataset],
            "target_present": False,
            "identifier_policy": "Not evaluated",
            "column_order_matches": False,
            "schema_pass": False,
        })
        continue

    synthetic_df = SYNTHETIC_DATAFRAMES_WITH_TARGET[dataset]
    result = validate_schema(dataset, synthetic_df)
    SCHEMA_VALIDATION_ROWS.append(result)

SCHEMA_VALIDATION_DF = pd.DataFrame(SCHEMA_VALIDATION_ROWS)

SCHEMA_VALIDATION_PATH = (
    NB13_VALIDATION_ROOT / "schema_validation.csv"
)

SCHEMA_VALIDATION_DF.to_csv(
    SCHEMA_VALIDATION_PATH,
    index=False,
)

# -----------------------------------------------------------------------------
# 14.4 — DISPLAY RESULTS
# -----------------------------------------------------------------------------

display(SCHEMA_VALIDATION_DF)

print("\n" + "=" * 100)
print("SECTION 14 — FINAL SUMMARY")
print("=" * 100)

total_datasets = len(SCHEMA_VALIDATION_DF)
passed_datasets = int(SCHEMA_VALIDATION_DF["schema_pass"].sum())
failed_datasets = total_datasets - passed_datasets

print(f"Expected datasets : {len(EXPECTED_DATASETS)}")
print(f"Schema PASS       : {passed_datasets}")
print(f"Schema FAIL       : {failed_datasets}")
print(f"Validation file   : {SCHEMA_VALIDATION_PATH}")

if failed_datasets == 0:
    SECTION_14_STATUS = "PASS — restored synthetic schemas verified"
else:
    SECTION_14_STATUS = "FAIL — inspect missing/unexpected columns and order"

print("\nSECTION 14 STATUS:", SECTION_14_STATUS)
print(
    "\nNOTE: __original_row_id__ is excluded intentionally because it is a source "
    "identifier, not a generated predictor. Target columns must be present."
)
print("=" * 100)


SECTION 14 — VALIDATE SCHEMA


,dataset,schema_status,expected_columns,actual_columns,missing_columns,unexpected_columns,duplicate_columns,target_column,target_present,identifier_policy,column_order_matches,schema_pass
0,adult_income,PASS,15,15,[],[],[],income,True,Excluded from expected synthetic schema: __ori...,True,True
1,bank_marketing,PASS,17,17,[],[],[],y,True,Excluded from expected synthetic schema: __ori...,True,True
2,diabetes_130us,PASS,48,48,[],[],[],readmitted,True,Excluded from expected synthetic schema: __ori...,True,True



SECTION 14 — FINAL SUMMARY
Expected datasets : 3
Schema PASS       : 3
Schema FAIL       : 0
Validation file   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/schema_validation.csv

SECTION 14 STATUS: PASS — restored synthetic schemas verified

NOTE: __original_row_id__ is excluded intentionally because it is a source identifier, not a generated predictor. Target columns must be present.


In [27]:

# ==================================================================================================
# SECTION 15 — RANGE AND DOMAIN VALIDATION
# Notebook 13 — SPP-GAN Synthetic Data Generation
# ==================================================================================================
# Purpose:
#   1. Validate synthetic numeric values against the real training-data ranges.
#   2. Check known feature-specific domain constraints.
#   3. Detect missing, non-numeric, non-finite, and fractional values.
#   4. Save detailed reports for reproducibility and research auditing.
#   5. Preserve the original synthetic data without silent clipping or correction.
#
# Required existing objects:
#   SYNTHETIC_DATAFRAMES_WITH_TARGET
#   REFERENCE_TRAIN
#   DATASET_IDS
#   NB13_ROOT
#
# Outputs:
#   validation/range_validation.csv
#   validation/domain_validation.csv
#   validation/range_validation_summary.csv
#   validation/section_15_validation_metadata.json
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import warnings

import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 15 — RANGE AND DOMAIN VALIDATION")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Validate required inputs
# --------------------------------------------------------------------------------------------------

_REQUIRED_GLOBALS = [
    "SYNTHETIC_DATAFRAMES_WITH_TARGET",
    "REFERENCE_TRAIN",
    "DATASET_IDS",
    "NB13_ROOT",
]

_missing_globals = [name for name in _REQUIRED_GLOBALS if name not in globals()]
if _missing_globals:
    raise NameError(
        "Section 15 cannot run. Missing required objects: "
        + ", ".join(_missing_globals)
        + ". Run the preceding Notebook 13 sections first."
    )

if not isinstance(SYNTHETIC_DATAFRAMES_WITH_TARGET, dict):
    raise TypeError("SYNTHETIC_DATAFRAMES_WITH_TARGET must be a dictionary.")

if not isinstance(REFERENCE_TRAIN, dict):
    raise TypeError(
        "REFERENCE_TRAIN must be a dictionary keyed by dataset ID. "
        "Check the reference-data loading section."
    )

DATASET_IDS = list(DATASET_IDS)
NB13_ROOT = Path(NB13_ROOT)

VALIDATION_ROOT = NB13_ROOT / "validation"
VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

RANGE_REPORT_PATH = VALIDATION_ROOT / "range_validation.csv"
DOMAIN_REPORT_PATH = VALIDATION_ROOT / "domain_validation.csv"
SUMMARY_REPORT_PATH = VALIDATION_ROOT / "range_validation_summary.csv"
METADATA_PATH = VALIDATION_ROOT / "section_15_validation_metadata.json"

# --------------------------------------------------------------------------------------------------
# 2. Feature-specific domain rules
# --------------------------------------------------------------------------------------------------
# These are explicit semantic checks, not replacements for the observed reference ranges.
# A rule is applied only when the feature exists in both datasets.
#
# Rule format:
#   (minimum, maximum, allowed_special_values, description)
#
# None means no bound on that side.
# Special values are allowed only when explicitly listed.

DOMAIN_RULES = {
    "adult_income": {
        "age": (0, None, [], "Age must be non-negative."),
        "education_num": (1, 16, [], "Education level code must be between 1 and 16."),
        "capital_gain": (0, None, [], "Capital gain must be non-negative."),
        "capital_loss": (0, None, [], "Capital loss must be non-negative."),
        "hours_per_week": (0, 168, [], "Hours per week must be between 0 and 168."),
    },
    "bank_marketing": {
        "age": (0, None, [], "Age must be non-negative."),
        "day": (1, 31, [], "Day of month must be between 1 and 31."),
        "duration": (0, None, [], "Call duration must be non-negative."),
        "campaign": (1, None, [], "Campaign contact count must be at least 1."),
        "pdays": (-1, None, [-1], "Pdays must be -1 (not previously contacted) or non-negative."),
        "previous": (0, None, [], "Previous contact count must be non-negative."),
    },
    "diabetes_130us": {
        "admission_type_id": (1, 8, [], "Admission type code must be between 1 and 8."),
        "discharge_disposition_id": (1, 28, [], "Discharge disposition code must be between 1 and 28."),
        "admission_source_id": (1, 25, [], "Admission source code must be between 1 and 25."),
        "time_in_hospital": (1, 14, [], "Length of stay must be between 1 and 14 days."),
        "num_lab_procedures": (0, None, [], "Lab procedure count must be non-negative."),
        "num_procedures": (0, None, [], "Procedure count must be non-negative."),
        "num_medications": (0, None, [], "Medication count must be non-negative."),
        "number_outpatient": (0, None, [], "Outpatient visit count must be non-negative."),
        "number_emergency": (0, None, [], "Emergency visit count must be non-negative."),
        "number_inpatient": (0, None, [], "Inpatient visit count must be non-negative."),
        "number_diagnoses": (1, 16, [], "Diagnosis count must be between 1 and 16."),
    },
}

# Common count/code columns expected to be integer-valued.
INTEGER_FEATURES = {
    "adult_income": {
        "age", "fnlwgt", "education_num", "capital_gain", "capital_loss",
        "hours_per_week",
    },
    "bank_marketing": {
        "age", "balance", "day", "duration", "campaign", "pdays", "previous",
    },
    "diabetes_130us": {
        "admission_type_id", "discharge_disposition_id", "admission_source_id",
        "time_in_hospital", "num_lab_procedures", "num_procedures",
        "num_medications", "number_outpatient", "number_emergency",
        "number_inpatient", "number_diagnoses",
    },
}

# --------------------------------------------------------------------------------------------------
# 3. Helpers for retrieving and validating data
# --------------------------------------------------------------------------------------------------

def _get_dataframe(registry, dataset_id, registry_name):
    """Retrieve a DataFrame from a dataset-keyed registry."""
    if dataset_id not in registry:
        raise KeyError(
            f"{registry_name} has no entry for dataset '{dataset_id}'. "
            f"Available keys: {list(registry.keys())}"
        )

    df = registry[dataset_id]

    if not isinstance(df, pd.DataFrame):
        raise TypeError(
            f"{registry_name}['{dataset_id}'] must be a pandas DataFrame; "
            f"received {type(df).__name__}."
        )

    if df.empty:
        warnings.warn(f"{registry_name}['{dataset_id}'] is empty.")

    return df


def _numeric_series(series):
    """Convert a copy of a series to numeric values; preserve the source DataFrame."""
    return pd.to_numeric(series, errors="coerce")


def _count_outside(values, minimum=None, maximum=None, allowed_special=None):
    """Count values outside bounds, excluding missing/non-finite values and allowed specials."""
    allowed_special = allowed_special or []

    valid = values.notna() & np.isfinite(values)
    checked = values[valid]

    if allowed_special:
        checked = checked[~checked.isin(allowed_special)]

    below = int((checked < minimum).sum()) if minimum is not None else 0
    above = int((checked > maximum).sum()) if maximum is not None else 0

    return below, above, int(below + above), int(len(checked))


# --------------------------------------------------------------------------------------------------
# 4. Run range and domain checks
# --------------------------------------------------------------------------------------------------

range_records = []
domain_records = []
dataset_summaries = []

print("\n" + "-" * 100)
print("VALIDATING SYNTHETIC DATASETS")
print("-" * 100)

for dataset_id in DATASET_IDS:
    reference_df = _get_dataframe(REFERENCE_TRAIN, dataset_id, "REFERENCE_TRAIN")
    synthetic_df = _get_dataframe(
        SYNTHETIC_DATAFRAMES_WITH_TARGET,
        dataset_id,
        "SYNTHETIC_DATAFRAMES_WITH_TARGET",
    )

    # Exclude row identifiers from numerical validation.
    identifier_columns = {
        col for col in reference_df.columns
        if str(col).strip().lower() in {
            "__original_row_id__", "original_row_id", "row_id", "record_id"
        }
    }

    # Only common columns can be compared directly.
    common_columns = [
        col for col in reference_df.columns
        if col in synthetic_df.columns and col not in identifier_columns
    ]

    numeric_columns = [
        col for col in common_columns
        if pd.api.types.is_numeric_dtype(reference_df[col])
    ]

    dataset_range_count = 0
    dataset_range_pass = 0
    dataset_range_review = 0
    dataset_domain_count = 0
    dataset_domain_pass = 0
    dataset_domain_review = 0

    # ----------------------------------------------------------------------------------------------
    # 4A. Observed reference-range validation
    # ----------------------------------------------------------------------------------------------

    for column in numeric_columns:
        ref_numeric = _numeric_series(reference_df[column])
        syn_numeric = _numeric_series(synthetic_df[column])

        ref_finite = ref_numeric[ref_numeric.notna() & np.isfinite(ref_numeric)]
        syn_finite = syn_numeric[syn_numeric.notna() & np.isfinite(syn_numeric)]

        ref_min = float(ref_finite.min()) if not ref_finite.empty else np.nan
        ref_max = float(ref_finite.max()) if not ref_finite.empty else np.nan
        syn_min = float(syn_finite.min()) if not syn_finite.empty else np.nan
        syn_max = float(syn_finite.max()) if not syn_finite.empty else np.nan

        below_count = (
            int((syn_finite < ref_min).sum()) if not np.isnan(ref_min) else 0
        )
        above_count = (
            int((syn_finite > ref_max).sum()) if not np.isnan(ref_max) else 0
        )

        missing_count = int(synthetic_df[column].isna().sum())
        nonnumeric_count = int(
            (synthetic_df[column].notna() & syn_numeric.isna()).sum()
        )
        nonfinite_count = int(
            (syn_numeric.notna() & ~np.isfinite(syn_numeric)).sum()
        )

        is_integer_feature = column in INTEGER_FEATURES.get(dataset_id, set())
        fractional_count = (
            int(
                (
                    syn_finite.notna()
                    & ~np.isclose(syn_finite, np.round(syn_finite), atol=1e-8)
                ).sum()
            )
            if is_integer_feature
            else 0
        )

        reasons = []
        if np.isnan(ref_min) or np.isnan(ref_max):
            reasons.append("REFERENCE_HAS_NO_FINITE_VALUES")
        if below_count:
            reasons.append("BELOW_REFERENCE_MIN")
        if above_count:
            reasons.append("ABOVE_REFERENCE_MAX")
        if missing_count:
            reasons.append("MISSING_VALUES")
        if nonnumeric_count:
            reasons.append("NON_NUMERIC_VALUES")
        if nonfinite_count:
            reasons.append("NON_FINITE_VALUES")
        if fractional_count:
            reasons.append("FRACTIONAL_INTEGER_VALUES")

        status = "PASS" if not reasons else "REVIEW"

        range_records.append({
            "dataset_id": dataset_id,
            "column": column,
            "reference_dtype": str(reference_df[column].dtype),
            "synthetic_dtype": str(synthetic_df[column].dtype),
            "reference_rows": int(len(reference_df)),
            "synthetic_rows": int(len(synthetic_df)),
            "reference_min": ref_min,
            "reference_max": ref_max,
            "synthetic_min": syn_min,
            "synthetic_max": syn_max,
            "below_reference_min_count": below_count,
            "above_reference_max_count": above_count,
            "out_of_reference_range_count": below_count + above_count,
            "missing_count": missing_count,
            "nonnumeric_count": nonnumeric_count,
            "nonfinite_count": nonfinite_count,
            "fractional_integer_count": fractional_count,
            "integer_feature_check": bool(is_integer_feature),
            "status": status,
            "review_reasons": ";".join(reasons),
        })

        dataset_range_count += 1
        if status == "PASS":
            dataset_range_pass += 1
        else:
            dataset_range_review += 1

    # ----------------------------------------------------------------------------------------------
    # 4B. Explicit semantic domain validation
    # ----------------------------------------------------------------------------------------------

    dataset_rules = DOMAIN_RULES.get(dataset_id, {})

    for column, rule in dataset_rules.items():
        if column not in reference_df.columns or column not in synthetic_df.columns:
            domain_records.append({
                "dataset_id": dataset_id,
                "column": column,
                "rule_min": rule[0],
                "rule_max": rule[1],
                "allowed_special_values": json.dumps(rule[2]),
                "rule_description": rule[3],
                "synthetic_min": np.nan,
                "synthetic_max": np.nan,
                "below_domain_min_count": np.nan,
                "above_domain_max_count": np.nan,
                "out_of_domain_count": np.nan,
                "missing_count": np.nan,
                "nonnumeric_count": np.nan,
                "nonfinite_count": np.nan,
                "status": "NOT_CHECKED",
                "review_reasons": "COLUMN_MISSING_FROM_REFERENCE_OR_SYNTHETIC_DATA",
            })
            dataset_domain_count += 1
            dataset_domain_review += 1
            continue

        minimum, maximum, allowed_special, description = rule
        raw_series = synthetic_df[column]
        values = _numeric_series(raw_series)
        finite_values = values[values.notna() & np.isfinite(values)]

        below, above, outside_count, checked_count = _count_outside(
            finite_values,
            minimum=minimum,
            maximum=maximum,
            allowed_special=allowed_special,
        )

        missing_count = int(raw_series.isna().sum())
        nonnumeric_count = int((raw_series.notna() & values.isna()).sum())
        nonfinite_count = int(
            (values.notna() & ~np.isfinite(values)).sum()
        )

        reasons = []
        if below:
            reasons.append("BELOW_DOMAIN_MIN")
        if above:
            reasons.append("ABOVE_DOMAIN_MAX")
        if missing_count:
            reasons.append("MISSING_VALUES")
        if nonnumeric_count:
            reasons.append("NON_NUMERIC_VALUES")
        if nonfinite_count:
            reasons.append("NON_FINITE_VALUES")

        status = "PASS" if not reasons else "REVIEW"

        domain_records.append({
            "dataset_id": dataset_id,
            "column": column,
            "rule_min": minimum,
            "rule_max": maximum,
            "allowed_special_values": json.dumps(allowed_special),
            "rule_description": description,
            "synthetic_min": (
                float(finite_values.min()) if not finite_values.empty else np.nan
            ),
            "synthetic_max": (
                float(finite_values.max()) if not finite_values.empty else np.nan
            ),
            "below_domain_min_count": below,
            "above_domain_max_count": above,
            "out_of_domain_count": outside_count,
            "missing_count": missing_count,
            "nonnumeric_count": nonnumeric_count,
            "nonfinite_count": nonfinite_count,
            "status": status,
            "review_reasons": ";".join(reasons),
        })

        dataset_domain_count += 1
        if status == "PASS":
            dataset_domain_pass += 1
        else:
            dataset_domain_review += 1

    dataset_summaries.append({
        "dataset_id": dataset_id,
        "reference_rows": int(len(reference_df)),
        "synthetic_rows": int(len(synthetic_df)),
        "numeric_columns_checked": dataset_range_count,
        "range_pass_columns": dataset_range_pass,
        "range_review_columns": dataset_range_review,
        "domain_rules_checked": dataset_domain_count,
        "domain_pass_rules": dataset_domain_pass,
        "domain_review_rules": dataset_domain_review,
        "range_status": "PASS" if dataset_range_review == 0 else "REVIEW",
        "domain_status": "PASS" if dataset_domain_review == 0 else "REVIEW",
    })

    print(
        f"{dataset_id}: "
        f"range PASS={dataset_range_pass}, REVIEW={dataset_range_review}; "
        f"domain PASS={dataset_domain_pass}, REVIEW={dataset_domain_review}"
    )

# --------------------------------------------------------------------------------------------------
# 5. Build and save validation reports
# --------------------------------------------------------------------------------------------------

range_validation_df = pd.DataFrame(range_records)
domain_validation_df = pd.DataFrame(domain_records)
range_validation_summary_df = pd.DataFrame(dataset_summaries)

range_validation_df.to_csv(RANGE_REPORT_PATH, index=False)
domain_validation_df.to_csv(DOMAIN_REPORT_PATH, index=False)
range_validation_summary_df.to_csv(SUMMARY_REPORT_PATH, index=False)

range_review_rows = (
    int((range_validation_df["status"] != "PASS").sum())
    if not range_validation_df.empty else 0
)
domain_review_rows = (
    int((domain_validation_df["status"] != "PASS").sum())
    if not domain_validation_df.empty else 0
)

metadata = {
    "section": 15,
    "section_name": "Range and Domain Validation",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(NB13_ROOT.parent.parent.parent),
    "notebook_13_root": str(NB13_ROOT),
    "dataset_ids": DATASET_IDS,
    "range_validation_rows": int(len(range_validation_df)),
    "range_pass_rows": int(
        (range_validation_df["status"] == "PASS").sum()
    ) if not range_validation_df.empty else 0,
    "range_review_rows": range_review_rows,
    "domain_validation_rows": int(len(domain_validation_df)),
    "domain_pass_rows": int(
        (domain_validation_df["status"] == "PASS").sum()
    ) if not domain_validation_df.empty else 0,
    "domain_review_rows": domain_review_rows,
    "overall_status": (
        "PASS"
        if range_review_rows == 0 and domain_review_rows == 0
        else "REVIEW_REQUIRED"
    ),
    "data_modified": False,
    "correction_applied": False,
    "notes": [
        "This section validates but does not modify synthetic datasets.",
        "Reference min/max checks are empirical comparisons, not proof of semantic validity.",
        "Domain rules are explicit checks and should be reviewed against dataset documentation.",
        "A REVIEW result must be investigated before claiming valid synthetic data.",
    ],
    "artifacts": {
        "range_validation": str(RANGE_REPORT_PATH),
        "domain_validation": str(DOMAIN_REPORT_PATH),
        "summary": str(SUMMARY_REPORT_PATH),
    },
}

with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

# --------------------------------------------------------------------------------------------------
# 6. Print final report
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 15 — FINAL VALIDATION SUMMARY")
print("=" * 100)

print(f"Datasets checked                 : {len(DATASET_IDS)}")
print(f"Numeric range checks             : {len(range_validation_df)}")
print(f"Range checks requiring review    : {range_review_rows}")
print(f"Domain rules checked             : {len(domain_validation_df)}")
print(f"Domain rules requiring review    : {domain_review_rows}")
print(f"Overall status                   : {metadata['overall_status']}")
print("\nSaved artifacts:")
print(f"  Range report    : {RANGE_REPORT_PATH}")
print(f"  Domain report   : {DOMAIN_REPORT_PATH}")
print(f"  Summary         : {SUMMARY_REPORT_PATH}")
print(f"  Metadata        : {METADATA_PATH}")

print("\nPer-dataset summary:")
display(range_validation_summary_df)

if range_review_rows or domain_review_rows:
    print(
        "\nWARNING: Validation issues remain. Do not label the synthetic data "
        "as fully range/domain valid."
    )
    print(
        "Investigate the generator output, transformed-feature ordering, and "
        "inverse-transformation logic before applying any correction."
    )
else:
    print(
        "\nAll configured range and domain checks passed. This does not establish "
        "distributional fidelity, downstream utility, or privacy."
    )

print("\nSECTION 15 EXECUTION COMPLETED.")


SECTION 15 — RANGE AND DOMAIN VALIDATION

----------------------------------------------------------------------------------------------------
VALIDATING SYNTHETIC DATASETS
----------------------------------------------------------------------------------------------------
adult_income: range PASS=6, REVIEW=0; domain PASS=5, REVIEW=0
bank_marketing: range PASS=7, REVIEW=0; domain PASS=6, REVIEW=0
diabetes_130us: range PASS=11, REVIEW=0; domain PASS=11, REVIEW=0

SECTION 15 — FINAL VALIDATION SUMMARY
Datasets checked                 : 3
Numeric range checks             : 24
Range checks requiring review    : 0
Domain rules checked             : 22
Domain rules requiring review    : 0
Overall status                   : PASS

Saved artifacts:
  Range report    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/range_validation.csv
  Domain report   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/domain_validation.csv
  Sum

,dataset_id,reference_rows,synthetic_rows,numeric_columns_checked,range_pass_columns,range_review_columns,domain_rules_checked,domain_pass_rules,domain_review_rules,range_status,domain_status
0,adult_income,34189,34189,6,6,0,5,5,0,PASS,PASS
1,bank_marketing,31647,31647,7,7,0,6,6,0,PASS,PASS
2,diabetes_130us,71236,71236,11,11,0,11,11,0,PASS,PASS



All configured range and domain checks passed. This does not establish distributional fidelity, downstream utility, or privacy.

SECTION 15 EXECUTION COMPLETED.


In [33]:

# ==================================================================================================
# SECTION 16 — VALIDATE AND DIAGNOSE MISSINGNESS
# Notebook 13 — SPP-GAN Synthetic Data Generation
# ==================================================================================================
# Objectives:
#   1. Validate feature schemas.
#   2. Compare actual null rates between reference and synthetic datasets.
#   3. Diagnose possible placeholder representations without modifying values.
#   4. Exclude confirmed bookkeeping identifiers from feature-level validation.
#   5. Save detailed results, dataset summaries, and reproducibility metadata.
#
# IMPORTANT:
#   - This section diagnoses; it does not repair missingness.
#   - Do not add artificial nulls to synthetic records to force a PASS.
#   - Do not silently impute or recode placeholder strings here.
#   - Passing missingness validation does not establish fidelity, utility, or privacy.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd

# --------------------------------------------------------------------------------------------------
# 1. Configuration
# --------------------------------------------------------------------------------------------------

MISSINGNESS_RATE_TOLERANCE = 0.01

TARGET_COLUMNS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

# Include __original_row_id__ only if it is bookkeeping metadata rather than
# an intended feature in the research model.
IDENTIFIER_COLUMNS = {
    "adult_income": [
        "id", "record_id", "__original_row_id__"
    ],
    "bank_marketing": [
        "id", "record_id", "__original_row_id__"
    ],
    "diabetes_130us": [
        "encounter_id", "patient_nbr", "id",
        "record_id", "__original_row_id__"
    ],
}

# Diagnostic candidates only. These strings are not automatically treated as nulls.
MISSINGNESS_CANDIDATE_TOKENS = [
    "?", "", "unknown", "none", "null", "nan", "na", "n/a"
]

# --------------------------------------------------------------------------------------------------
# 2. Validate prerequisites
# --------------------------------------------------------------------------------------------------

required_names = [
    "SYNTHETIC_DATAFRAMES_WITH_TARGET",
    "REFERENCE_TRAIN",
    "DATASET_IDS",
    "NB13_VALIDATION_ROOT",
]

missing_names = [name for name in required_names if name not in globals()]

if missing_names:
    raise NameError(
        "Missing Section 16 prerequisites: "
        + ", ".join(missing_names)
        + ". Run the relevant Notebook 13 loading and generation sections first."
    )

if not isinstance(REFERENCE_TRAIN, dict):
    raise TypeError("REFERENCE_TRAIN must be a dictionary keyed by dataset ID.")

if not isinstance(SYNTHETIC_DATAFRAMES_WITH_TARGET, dict):
    raise TypeError(
        "SYNTHETIC_DATAFRAMES_WITH_TARGET must be a dictionary keyed by dataset ID."
    )

DATASET_IDS = list(DATASET_IDS)

if not DATASET_IDS:
    raise ValueError("DATASET_IDS is empty.")

for dataset_id in DATASET_IDS:
    if dataset_id not in REFERENCE_TRAIN:
        raise KeyError(f"Reference training data missing: {dataset_id}")
    if dataset_id not in SYNTHETIC_DATAFRAMES_WITH_TARGET:
        raise KeyError(f"Synthetic dataset missing: {dataset_id}")

    if not isinstance(REFERENCE_TRAIN[dataset_id], pd.DataFrame):
        raise TypeError(f"{dataset_id}: reference data must be a DataFrame.")

    if not isinstance(
        SYNTHETIC_DATAFRAMES_WITH_TARGET[dataset_id], pd.DataFrame
    ):
        raise TypeError(f"{dataset_id}: synthetic data must be a DataFrame.")

# --------------------------------------------------------------------------------------------------
# 3. Resolve output directory without duplicating validation/
# --------------------------------------------------------------------------------------------------

NB13_VALIDATION_ROOT = Path(NB13_VALIDATION_ROOT)

if NB13_VALIDATION_ROOT.name.lower() == "validation":
    VALIDATION_DIR = NB13_VALIDATION_ROOT
else:
    VALIDATION_DIR = NB13_VALIDATION_ROOT / "validation"

VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------------------------------------
# 4. Helper functions
# --------------------------------------------------------------------------------------------------

def candidate_token_counts(series):
    """Count candidate placeholder strings without modifying the input."""
    values = series.dropna().astype("string").str.strip().str.casefold()
    counts = {}

    for token in MISSINGNESS_CANDIDATE_TOKENS:
        normalized_token = token.casefold()
        count = int(values.eq(normalized_token).sum())

        if count:
            counts["<blank>" if token == "" else token] = count

    return counts


def null_rate(series):
    """Return actual pandas-null count and rate."""
    count = int(series.isna().sum())
    rate = float(series.isna().mean()) if len(series) else np.nan
    return count, rate


def json_safe_list(values):
    """Convert column names to stable JSON-compatible strings."""
    return [str(value) for value in values]


def get_feature_columns(dataset_id, reference_df, synthetic_df):
    """
    Identify shared feature columns, missing synthetic columns,
    and unexpected synthetic columns.

    Targets and configured identifiers are excluded from feature-level
    missingness comparison. Their presence is still reflected in the
    full DataFrame column counts and schema report.
    """
    excluded = set(IDENTIFIER_COLUMNS.get(dataset_id, []))
    target = TARGET_COLUMNS.get(dataset_id)

    if target:
        excluded.add(target)

    reference_features = [
        col for col in reference_df.columns if col not in excluded
    ]
    synthetic_features = [
        col for col in synthetic_df.columns if col not in excluded
    ]

    reference_set = set(reference_features)
    synthetic_set = set(synthetic_features)

    missing_from_synthetic = [
        col for col in reference_features if col not in synthetic_set
    ]
    unexpected_in_synthetic = [
        col for col in synthetic_features if col not in reference_set
    ]
    common_features = [
        col for col in reference_features if col in synthetic_set
    ]

    return (
        common_features,
        missing_from_synthetic,
        unexpected_in_synthetic,
    )


def check_full_schema(dataset_id, reference_df, synthetic_df):
    """
    Report full-schema differences separately from feature-level comparison.

    A configured identifier absent from synthetic data is not automatically
    treated as a model-feature failure. Any other column difference is reported.
    """
    configured_identifiers = set(IDENTIFIER_COLUMNS.get(dataset_id, []))

    reference_columns = set(reference_df.columns)
    synthetic_columns = set(synthetic_df.columns)

    missing_columns = reference_columns - synthetic_columns
    unexpected_columns = synthetic_columns - reference_columns

    missing_non_identifier = missing_columns - configured_identifiers
    unexpected_non_identifier = unexpected_columns - configured_identifiers

    full_schema_status = (
        "PASS"
        if not missing_non_identifier and not unexpected_non_identifier
        else "SCHEMA_MISMATCH"
    )

    return {
        "reference_columns": json_safe_list(reference_df.columns),
        "synthetic_columns": json_safe_list(synthetic_df.columns),
        "missing_columns": sorted(map(str, missing_columns)),
        "unexpected_columns": sorted(map(str, unexpected_columns)),
        "missing_non_identifier_columns": sorted(
            map(str, missing_non_identifier)
        ),
        "unexpected_non_identifier_columns": sorted(
            map(str, unexpected_non_identifier)
        ),
        "full_schema_status": full_schema_status,
    }


# --------------------------------------------------------------------------------------------------
# 5. Run missingness validation
# --------------------------------------------------------------------------------------------------

detail_rows = []
dataset_summary_rows = []
dataset_metadata = {}

print("=" * 100)
print("SECTION 16 — VALIDATE AND DIAGNOSE MISSINGNESS")
print("=" * 100)
print(f"Datasets registered       : {len(DATASET_IDS)}")
print(f"Null-rate tolerance       : {MISSINGNESS_RATE_TOLERANCE:.4f}")
print("Source DataFrames modified: No")

for dataset_id in DATASET_IDS:

    reference_df = REFERENCE_TRAIN[dataset_id]
    synthetic_df = SYNTHETIC_DATAFRAMES_WITH_TARGET[dataset_id]

    reference_rows = len(reference_df)
    synthetic_rows = len(synthetic_df)

    schema_info = check_full_schema(
        dataset_id,
        reference_df,
        synthetic_df,
    )

    (
        common_features,
        missing_model_columns,
        unexpected_model_columns,
    ) = get_feature_columns(
        dataset_id,
        reference_df,
        synthetic_df,
    )

    exact_matches = 0
    within_tolerance = 0
    rate_reviews = 0
    schema_mismatch_rows = 0
    not_comparable_rows = 0
    excluded_identifier_rows = 0

    # Report missing expected model features.
    for col in missing_model_columns:
        detail_rows.append({
            "dataset_id": dataset_id,
            "column": str(col),
            "column_role": "feature",
            "reference_rows": reference_rows,
            "synthetic_rows": synthetic_rows,
            "reference_null_count": int(reference_df[col].isna().sum()),
            "reference_null_rate": (
                float(reference_df[col].isna().mean())
                if reference_rows else np.nan
            ),
            "reference_candidate_tokens": candidate_token_counts(
                reference_df[col]
            ),
            "synthetic_null_count": np.nan,
            "synthetic_null_rate": np.nan,
            "synthetic_candidate_tokens": {},
            "absolute_null_rate_difference": np.nan,
            "signed_null_rate_difference": np.nan,
            "status": "SCHEMA_MISMATCH",
            "diagnostic": "MISSING_MODEL_FEATURE",
            "review_reason": "REFERENCE_FEATURE_ABSENT_FROM_SYNTHETIC",
        })
        schema_mismatch_rows += 1

    # Report unexpected synthetic model features.
    for col in unexpected_model_columns:
        detail_rows.append({
            "dataset_id": dataset_id,
            "column": str(col),
            "column_role": "feature",
            "reference_rows": reference_rows,
            "synthetic_rows": synthetic_rows,
            "reference_null_count": np.nan,
            "reference_null_rate": np.nan,
            "reference_candidate_tokens": {},
            "synthetic_null_count": int(synthetic_df[col].isna().sum()),
            "synthetic_null_rate": (
                float(synthetic_df[col].isna().mean())
                if synthetic_rows else np.nan
            ),
            "synthetic_candidate_tokens": candidate_token_counts(
                synthetic_df[col]
            ),
            "absolute_null_rate_difference": np.nan,
            "signed_null_rate_difference": np.nan,
            "status": "SCHEMA_MISMATCH",
            "diagnostic": "UNEXPECTED_SYNTHETIC_FEATURE",
            "review_reason": "SYNTHETIC_FEATURE_ABSENT_FROM_REFERENCE",
        })
        schema_mismatch_rows += 1

    # Compare actual null rates for common model features.
    for col in common_features:

        reference_series = reference_df[col]
        synthetic_series = synthetic_df[col]

        ref_null_count, ref_null_rate = null_rate(reference_series)
        syn_null_count, syn_null_rate = null_rate(synthetic_series)

        ref_tokens = candidate_token_counts(reference_series)
        syn_tokens = candidate_token_counts(synthetic_series)

        if reference_rows == 0 or synthetic_rows == 0:
            absolute_difference = np.nan
            signed_difference = np.nan
            status = "NOT_COMPARABLE"
            diagnostic = "EMPTY_DATAFRAME"
            review_reason = "ZERO_ROWS"
            not_comparable_rows += 1

        else:
            signed_difference = syn_null_rate - ref_null_rate
            absolute_difference = abs(signed_difference)

            if absolute_difference <= 1e-12:
                status = "PASS"
                diagnostic = "EXACT_NULL_RATE_MATCH"
                review_reason = ""
                exact_matches += 1

            elif absolute_difference <= MISSINGNESS_RATE_TOLERANCE:
                status = "PASS_WITHIN_TOLERANCE"
                diagnostic = "NULL_RATE_WITHIN_CONFIGURED_TOLERANCE"
                review_reason = ""
                within_tolerance += 1

            else:
                status = "REVIEW"
                diagnostic = "INVESTIGATE_UPSTREAM_NULL_HANDLING"
                review_reason = "NULL_RATE_DIFFERENCE_EXCEEDS_TOLERANCE"
                rate_reviews += 1

        detail_rows.append({
            "dataset_id": dataset_id,
            "column": str(col),
            "column_role": "feature",
            "reference_rows": reference_rows,
            "synthetic_rows": synthetic_rows,
            "reference_null_count": ref_null_count,
            "reference_null_rate": ref_null_rate,
            "reference_candidate_tokens": ref_tokens,
            "synthetic_null_count": syn_null_count,
            "synthetic_null_rate": syn_null_rate,
            "synthetic_candidate_tokens": syn_tokens,
            "absolute_null_rate_difference": absolute_difference,
            "signed_null_rate_difference": signed_difference,
            "status": status,
            "diagnostic": diagnostic,
            "review_reason": review_reason,
        })

    # Report identifier columns explicitly as excluded bookkeeping metadata.
    for col in IDENTIFIER_COLUMNS.get(dataset_id, []):
        if col in reference_df.columns or col in synthetic_df.columns:
            detail_rows.append({
                "dataset_id": dataset_id,
                "column": str(col),
                "column_role": "identifier",
                "reference_rows": reference_rows,
                "synthetic_rows": synthetic_rows,
                "reference_null_count": (
                    int(reference_df[col].isna().sum())
                    if col in reference_df.columns else np.nan
                ),
                "reference_null_rate": (
                    float(reference_df[col].isna().mean())
                    if col in reference_df.columns and reference_rows
                    else np.nan
                ),
                "reference_candidate_tokens": (
                    candidate_token_counts(reference_df[col])
                    if col in reference_df.columns else {}
                ),
                "synthetic_null_count": (
                    int(synthetic_df[col].isna().sum())
                    if col in synthetic_df.columns else np.nan
                ),
                "synthetic_null_rate": (
                    float(synthetic_df[col].isna().mean())
                    if col in synthetic_df.columns and synthetic_rows
                    else np.nan
                ),
                "synthetic_candidate_tokens": (
                    candidate_token_counts(synthetic_df[col])
                    if col in synthetic_df.columns else {}
                ),
                "absolute_null_rate_difference": np.nan,
                "signed_null_rate_difference": np.nan,
                "status": "EXCLUDED_IDENTIFIER",
                "diagnostic": "BOOKKEEPING_IDENTIFIER_EXCLUDED",
                "review_reason": "",
            })
            excluded_identifier_rows += 1

    if schema_info["full_schema_status"] == "SCHEMA_MISMATCH":
        dataset_status = "SCHEMA_MISMATCH"
    elif rate_reviews or not_comparable_rows:
        dataset_status = "REVIEW_REQUIRED"
    else:
        dataset_status = "PASS"

    dataset_summary_rows.append({
        "dataset_id": dataset_id,
        "reference_rows": reference_rows,
        "synthetic_rows": synthetic_rows,
        "reference_columns": reference_df.shape[1],
        "synthetic_columns": synthetic_df.shape[1],
        "missing_model_columns": json.dumps(missing_model_columns),
        "unexpected_model_columns": json.dumps(unexpected_model_columns),
        "full_schema_missing_columns": json.dumps(
            schema_info["missing_columns"]
        ),
        "full_schema_unexpected_columns": json.dumps(
            schema_info["unexpected_columns"]
        ),
        "exact_matches": exact_matches,
        "within_tolerance": within_tolerance,
        "rate_reviews": rate_reviews,
        "schema_mismatch_rows": schema_mismatch_rows,
        "not_comparable_rows": not_comparable_rows,
        "excluded_identifier_rows": excluded_identifier_rows,
        "schema_status": schema_info["full_schema_status"],
        "overall_status": dataset_status,
    })

    dataset_metadata[dataset_id] = {
        "reference_rows": reference_rows,
        "synthetic_rows": synthetic_rows,
        "reference_columns": schema_info["reference_columns"],
        "synthetic_columns": schema_info["synthetic_columns"],
        "missing_columns": schema_info["missing_columns"],
        "unexpected_columns": schema_info["unexpected_columns"],
        "missing_model_columns": json_safe_list(missing_model_columns),
        "unexpected_model_columns": json_safe_list(
            unexpected_model_columns
        ),
        "schema_status": schema_info["full_schema_status"],
        "overall_status": dataset_status,
    }

    print(
        f"{dataset_id}: exact={exact_matches}, "
        f"within tolerance={within_tolerance}, "
        f"review={rate_reviews}, "
        f"model schema mismatches={schema_mismatch_rows}, "
        f"status={dataset_status}"
    )

# --------------------------------------------------------------------------------------------------
# 6. Build final result DataFrames
# --------------------------------------------------------------------------------------------------

MISSINGNESS_VALIDATION_DF = pd.DataFrame(detail_rows)
MISSINGNESS_SUMMARY_DF = pd.DataFrame(dataset_summary_rows)

if MISSINGNESS_VALIDATION_DF.empty:
    raise RuntimeError("No validation rows were generated.")

total_exact = int(
    (MISSINGNESS_VALIDATION_DF["status"] == "PASS").sum()
)
total_within_tolerance = int(
    (
        MISSINGNESS_VALIDATION_DF["status"]
        == "PASS_WITHIN_TOLERANCE"
    ).sum()
)
total_reviews = int(
    (MISSINGNESS_VALIDATION_DF["status"] == "REVIEW").sum()
)
total_schema_mismatches = int(
    (
        MISSINGNESS_VALIDATION_DF["status"]
        == "SCHEMA_MISMATCH"
    ).sum()
)
total_not_comparable = int(
    (
        MISSINGNESS_VALIDATION_DF["status"]
        == "NOT_COMPARABLE"
    ).sum()
)
total_excluded_identifiers = int(
    (
        MISSINGNESS_VALIDATION_DF["status"]
        == "EXCLUDED_IDENTIFIER"
    ).sum()
)

has_schema_mismatch = (
    MISSINGNESS_SUMMARY_DF["schema_status"]
    == "SCHEMA_MISMATCH"
).any()

if has_schema_mismatch:
    overall_status = "SCHEMA_MISMATCH"
elif total_reviews or total_not_comparable:
    overall_status = "REVIEW_REQUIRED"
else:
    overall_status = "PASS"

# --------------------------------------------------------------------------------------------------
# 7. Save artifacts
# --------------------------------------------------------------------------------------------------

detail_path = VALIDATION_DIR / "missingness_validation.csv"
summary_path = VALIDATION_DIR / "missingness_validation_summary.csv"
metadata_path = VALIDATION_DIR / "missingness_validation_metadata.json"

detail_export = MISSINGNESS_VALIDATION_DF.copy()

for column in [
    "reference_candidate_tokens",
    "synthetic_candidate_tokens",
]:
    detail_export[column] = detail_export[column].apply(
        lambda value: json.dumps(
            value if isinstance(value, dict) else {},
            sort_keys=True,
        )
    )

detail_export.to_csv(detail_path, index=False)
MISSINGNESS_SUMMARY_DF.to_csv(summary_path, index=False)

metadata = {
    "notebook": "Notebook 13 — SPP-GAN Synthetic Data Generation",
    "section": "16 — Validate and Diagnose Missingness",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "missingness_rate_tolerance": MISSINGNESS_RATE_TOLERANCE,
    "source_dataframes_modified": False,
    "candidate_tokens_are_diagnostic_only": True,
    "bookkeeping_identifiers_excluded_from_feature_comparison": True,
    "datasets_checked": len(DATASET_IDS),
    "detail_rows": len(MISSINGNESS_VALIDATION_DF),
    "exact_matches": total_exact,
    "within_tolerance": total_within_tolerance,
    "missingness_differences_to_review": total_reviews,
    "schema_mismatch_rows": total_schema_mismatches,
    "not_comparable_rows": total_not_comparable,
    "excluded_identifier_rows": total_excluded_identifiers,
    "overall_status": overall_status,
    "datasets": dataset_metadata,
    "artifacts": {
        "detail_csv": str(detail_path),
        "summary_csv": str(summary_path),
        "metadata_json": str(metadata_path),
    },
    "interpretation": (
        "Feature-level null-rate diagnostic only. "
        "This section does not repair missingness and does not establish "
        "overall synthetic-data fidelity, downstream utility, or privacy."
    ),
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

# --------------------------------------------------------------------------------------------------
# 8. Final report
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 16 — FINAL SUMMARY")
print("=" * 100)
print(f"Datasets checked                  : {len(DATASET_IDS)}")
print(f"Detailed validation rows          : {len(MISSINGNESS_VALIDATION_DF)}")
print(f"Exact matches                     : {total_exact}")
print(f"Within tolerance                  : {total_within_tolerance}")
print(f"Missingness differences to review : {total_reviews}")
print(f"Schema mismatch rows              : {total_schema_mismatches}")
print(f"Non-comparable rows               : {total_not_comparable}")
print(f"Excluded identifier rows          : {total_excluded_identifiers}")
print(f"Overall status                    : {overall_status}")

print("\nSaved artifacts:")
print(f"  Detail   : {detail_path}")
print(f"  Summary  : {summary_path}")
print(f"  Metadata : {metadata_path}")

print("\nDataset summary:")
display(MISSINGNESS_SUMMARY_DF)

review_df = MISSINGNESS_VALIDATION_DF[
    MISSINGNESS_VALIDATION_DF["status"] == "REVIEW"
].copy()

if not review_df.empty:
    review_df = review_df.sort_values(
        "absolute_null_rate_difference",
        ascending=False,
    )

    print("\nHighest-priority missingness differences:")
    display(
        review_df[
            [
                "dataset_id",
                "column",
                "reference_null_rate",
                "synthetic_null_rate",
                "absolute_null_rate_difference",
                "diagnostic",
                "review_reason",
            ]
        ].head(15)
    )
else:
    print("\nNo feature null-rate differences exceeded the configured tolerance.")

review_tokens = MISSINGNESS_VALIDATION_DF[
    MISSINGNESS_VALIDATION_DF["status"] == "REVIEW"
]

tokens_found = False

for col in [
    "reference_candidate_tokens",
    "synthetic_candidate_tokens",
]:
    if col in review_tokens.columns:
        tokens_found |= review_tokens[col].apply(
            lambda value: bool(value) if isinstance(value, dict) else False
        ).any()

print("\nRepresentation diagnostics:")
if tokens_found:
    print(
        "Candidate placeholder tokens were found. "
        "Inspect their semantics and upstream handling before changing data."
    )
else:
    print(
        "No configured candidate tokens were detected in reviewed columns. "
        "Inspect preprocessing, encoding, generation, and inverse transformation."
    )

if overall_status == "PASS":
    print(
        "\nMISSINGNESS CHECK PASSED. This result applies only to the checks "
        "implemented in this section."
    )
else:
    print(
        "\nREVIEW REQUIRED. Investigate flagged columns and schema differences. "
        "Do not change synthetic values merely to force a PASS."
    )

print("\nSECTION 16 EXECUTION COMPLETED.")


SECTION 16 — VALIDATE AND DIAGNOSE MISSINGNESS
Datasets registered       : 3
Null-rate tolerance       : 0.0100
Source DataFrames modified: No
adult_income: exact=11, within tolerance=0, review=3, model schema mismatches=0, status=REVIEW_REQUIRED
bank_marketing: exact=16, within tolerance=0, review=0, model schema mismatches=0, status=PASS
diabetes_130us: exact=38, within tolerance=2, review=7, model schema mismatches=0, status=REVIEW_REQUIRED

SECTION 16 — FINAL SUMMARY
Datasets checked                  : 3
Detailed validation rows          : 80
Exact matches                     : 65
Within tolerance                  : 2
Missingness differences to review : 10
Schema mismatch rows              : 0
Non-comparable rows               : 0
Excluded identifier rows          : 3
Overall status                    : REVIEW_REQUIRED

Saved artifacts:
  Detail   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/missingness_validation.csv
  Summary  : /content/driv

,dataset_id,reference_rows,synthetic_rows,reference_columns,synthetic_columns,missing_model_columns,unexpected_model_columns,full_schema_missing_columns,full_schema_unexpected_columns,exact_matches,within_tolerance,rate_reviews,schema_mismatch_rows,not_comparable_rows,excluded_identifier_rows,schema_status,overall_status
0,adult_income,34189,34189,16,15,[],[],"[""__original_row_id__""]",[],11,0,3,0,0,1,PASS,REVIEW_REQUIRED
1,bank_marketing,31647,31647,18,17,[],[],"[""__original_row_id__""]",[],16,0,0,0,0,1,PASS,PASS
2,diabetes_130us,71236,71236,49,48,[],[],"[""__original_row_id__""]",[],38,2,7,0,0,1,PASS,REVIEW_REQUIRED



Highest-priority missingness differences:


,dataset_id,column,reference_null_rate,synthetic_null_rate,absolute_null_rate_difference,diagnostic,review_reason
35,diabetes_130us,weight,0.969187,0.0,0.969187,INVESTIGATE_UPSTREAM_NULL_HANDLING,NULL_RATE_DIFFERENCE_EXCEEDS_TOLERANCE
52,diabetes_130us,max_glu_serum,0.947316,0.0,0.947316,INVESTIGATE_UPSTREAM_NULL_HANDLING,NULL_RATE_DIFFERENCE_EXCEEDS_TOLERANCE
53,diabetes_130us,A1Cresult,0.834171,0.0,0.834171,INVESTIGATE_UPSTREAM_NULL_HANDLING,NULL_RATE_DIFFERENCE_EXCEEDS_TOLERANCE
41,diabetes_130us,medical_specialty,0.490510,0.0,0.490510,INVESTIGATE_UPSTREAM_NULL_HANDLING,NULL_RATE_DIFFERENCE_EXCEEDS_TOLERANCE
40,diabetes_130us,payer_code,0.396499,0.0,0.396499,INVESTIGATE_UPSTREAM_NULL_HANDLING,NULL_RATE_DIFFERENCE_EXCEEDS_TOLERANCE
6,adult_income,occupation,0.058411,0.0,0.058411,INVESTIGATE_UPSTREAM_NULL_HANDLING,NULL_RATE_DIFFERENCE_EXCEEDS_TOLERANCE
1,adult_income,workclass,0.058264,0.0,0.058264,INVESTIGATE_UPSTREAM_NULL_HANDLING,NULL_RATE_DIFFERENCE_EXCEEDS_TOLERANCE
32,diabetes_130us,race,0.021941,0.0,0.021941,INVESTIGATE_UPSTREAM_NULL_HANDLING,NULL_RATE_DIFFERENCE_EXCEEDS_TOLERANCE
13,adult_income,native_country,0.017842,0.0,0.017842,INVESTIGATE_UPSTREAM_NULL_HANDLING,NULL_RATE_DIFFERENCE_EXCEEDS_TOLERANCE
50,diabetes_130us,diag_3,0.014080,0.0,0.014080,INVESTIGATE_UPSTREAM_NULL_HANDLING,NULL_RATE_DIFFERENCE_EXCEEDS_TOLERANCE



Representation diagnostics:
No configured candidate tokens were detected in reviewed columns. Inspect preprocessing, encoding, generation, and inverse transformation.

REVIEW REQUIRED. Investigate flagged columns and schema differences. Do not change synthetic values merely to force a PASS.

SECTION 16 EXECUTION COMPLETED.


In [35]:

# ==================================================================================================
# SECTION 17 — VALIDATE INVALID VALUES
# Notebook 13 — SPP-GAN Synthetic Data Generation
# ==================================================================================================
# Objectives:
#   1. Detect non-finite numeric values.
#   2. Detect non-numeric values in reference-numeric columns.
#   3. Detect unseen categorical values.
#   4. Detect fractional values in integer-valued features.
#   5. Report genuine schema mismatches separately from excluded identifiers.
#   6. Save auditable validation results and metadata.
#
# IMPORTANT:
#   - Reference and synthetic DataFrames are never modified.
#   - Missingness rates are assessed in Section 16.
#   - Numeric range/domain validation is assessed in Section 15.
#   - Bookkeeping identifiers are excluded only from feature validation.
#   - A PASS does not establish distributional fidelity, utility, or privacy.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 17 — VALIDATE INVALID VALUES")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Validate prerequisites
# --------------------------------------------------------------------------------------------------

_REQUIRED = [
    "SYNTHETIC_DATAFRAMES_WITH_TARGET",
    "REFERENCE_TRAIN",
    "DATASET_IDS",
    "NB13_VALIDATION_ROOT",
]

_missing = [name for name in _REQUIRED if name not in globals()]

if _missing:
    raise NameError(
        "Missing required objects: "
        + ", ".join(_missing)
        + ". Run the preceding Notebook 13 sections first."
    )

if not isinstance(SYNTHETIC_DATAFRAMES_WITH_TARGET, dict):
    raise TypeError(
        "SYNTHETIC_DATAFRAMES_WITH_TARGET must be a dictionary."
    )

if not isinstance(REFERENCE_TRAIN, dict):
    raise TypeError("REFERENCE_TRAIN must be a dictionary.")

DATASET_IDS = list(DATASET_IDS)

if not DATASET_IDS:
    raise ValueError("DATASET_IDS is empty.")

for dataset in DATASET_IDS:
    if dataset not in REFERENCE_TRAIN:
        raise KeyError(f"Reference data not found for '{dataset}'.")

    if dataset not in SYNTHETIC_DATAFRAMES_WITH_TARGET:
        raise KeyError(
            f"Synthetic data with target not found for '{dataset}'. "
            "Run the preceding data-generation sections first."
        )

    if not isinstance(REFERENCE_TRAIN[dataset], pd.DataFrame):
        raise TypeError(
            f"Reference data for '{dataset}' must be a pandas DataFrame."
        )

    if not isinstance(
        SYNTHETIC_DATAFRAMES_WITH_TARGET[dataset], pd.DataFrame
    ):
        raise TypeError(
            f"Synthetic data for '{dataset}' must be a pandas DataFrame."
        )

# --------------------------------------------------------------------------------------------------
# 2. Resolve validation output directory
# --------------------------------------------------------------------------------------------------

NB13_VALIDATION_ROOT = Path(NB13_VALIDATION_ROOT)

# Avoid accidentally creating validation/validation/.
if NB13_VALIDATION_ROOT.name.lower() == "validation":
    VALIDATION_DIR = NB13_VALIDATION_ROOT
else:
    VALIDATION_DIR = NB13_VALIDATION_ROOT / "validation"

VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

INVALID_VALUES_PATH = (
    VALIDATION_DIR / "invalid_values_validation.csv"
)
INVALID_SUMMARY_PATH = (
    VALIDATION_DIR / "invalid_values_validation_summary.csv"
)
INVALID_METADATA_PATH = (
    VALIDATION_DIR / "invalid_values_validation_metadata.json"
)

# --------------------------------------------------------------------------------------------------
# 3. Configuration
# --------------------------------------------------------------------------------------------------

# These columns are bookkeeping identifiers, not synthetic model features.
# Exclude them from feature-schema comparison and value validation.
#
# IMPORTANT:
# Keep __original_row_id__ here only if it is genuinely bookkeeping metadata.
# If it is an intended model feature, remove it from this configuration.
IDENTIFIER_COLUMNS = {
    "id",
    "record_id",
    "row_id",
    "original_row_id",
    "__original_row_id__",
    "encounter_id",
    "patient_nbr",
}

# Dataset-specific integer-valued features.
# A feature should appear here only if its intended values are integers.
INTEGER_FEATURES = {
    "adult_income": {
        "age",
        "fnlwgt",
        "education_num",
        "capital_gain",
        "capital_loss",
        "hours_per_week",
    },
    "bank_marketing": {
        "age",
        "balance",
        "day",
        "duration",
        "campaign",
        "pdays",
        "previous",
    },
    "diabetes_130us": {
        "admission_type_id",
        "discharge_disposition_id",
        "admission_source_id",
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    },
}

FRACTIONAL_VALUE_ATOL = 1e-8


def normalize_column_name(column):
    """Normalize a column name for identifier matching."""
    return str(column).strip().lower()


def is_identifier_column(column):
    """Return True for configured bookkeeping identifier columns."""
    return normalize_column_name(column) in IDENTIFIER_COLUMNS


def safe_json_list(values):
    """Convert column labels into JSON-compatible strings."""
    return [str(value) for value in values]


# --------------------------------------------------------------------------------------------------
# 4. Column-level validation function
# --------------------------------------------------------------------------------------------------

def validate_invalid_values(dataset, synthetic_df, reference_df):
    """
    Validate synthetic feature values against the reference training data.

    Identifier columns are reported as EXCLUDED_IDENTIFIER and do not count
    as feature-schema mismatches when they are absent from synthetic data.

    The function never modifies the input DataFrames.
    """

    records = []

    reference_columns = list(reference_df.columns)
    synthetic_columns = list(synthetic_df.columns)

    reference_column_set = set(reference_columns)
    synthetic_column_set = set(synthetic_columns)

    missing_columns = [
        col for col in reference_columns
        if col not in synthetic_column_set
    ]

    unexpected_columns = [
        col for col in synthetic_columns
        if col not in reference_column_set
    ]

    # ----------------------------------------------------------------------------------------------
    # 4A. Validate reference columns
    # ----------------------------------------------------------------------------------------------

    for column in reference_columns:

        source = reference_df[column]
        is_identifier = is_identifier_column(column)

        # Exclude bookkeeping identifiers from feature-schema checks.
        # Their absence from synthetic data is not a feature-schema mismatch.
        if is_identifier:
            synthetic_has_column = column in synthetic_df.columns

            records.append({
                "dataset_id": dataset,
                "column": str(column),
                "column_role": "identifier",
                "reference_dtype": str(source.dtype),
                "synthetic_dtype": (
                    str(synthetic_df[column].dtype)
                    if synthetic_has_column
                    else None
                ),
                "synthetic_missing_count": (
                    int(synthetic_df[column].isna().sum())
                    if synthetic_has_column
                    else np.nan
                ),
                "non_finite_numeric_count": np.nan,
                "nonnumeric_in_numeric_column_count": np.nan,
                "unseen_category_count": np.nan,
                "fractional_integer_value_count": np.nan,
                "invalid_value_count": np.nan,
                "schema_status": "EXCLUDED_IDENTIFIER",
                "value_status": "NOT_CHECKED",
                "overall_status": "EXCLUDED_IDENTIFIER",
                "review_reasons": (
                    "BOOKKEEPING_IDENTIFIER_PRESENT_BUT_EXCLUDED"
                    if synthetic_has_column
                    else "BOOKKEEPING_IDENTIFIER_ABSENT_BUT_EXCLUDED"
                ),
            })
            continue

        # Genuine missing feature or target column.
        if column not in synthetic_df.columns:
            records.append({
                "dataset_id": dataset,
                "column": str(column),
                "column_role": "feature_or_target",
                "reference_dtype": str(source.dtype),
                "synthetic_dtype": None,
                "synthetic_missing_count": np.nan,
                "non_finite_numeric_count": np.nan,
                "nonnumeric_in_numeric_column_count": np.nan,
                "unseen_category_count": np.nan,
                "fractional_integer_value_count": np.nan,
                "invalid_value_count": np.nan,
                "schema_status": "MISSING_COLUMN",
                "value_status": "NOT_CHECKED",
                "overall_status": "SCHEMA_MISMATCH",
                "review_reasons": "MODEL_COLUMN_MISSING_FROM_SYNTHETIC",
            })
            continue

        series = synthetic_df[column]

        missing_count = int(series.isna().sum())
        non_finite = 0
        nonnumeric_count = 0
        unseen_categories = 0
        fractional_count = 0

        reference_is_numeric = pd.api.types.is_numeric_dtype(
            source.dtype
        )
        reference_is_bool = pd.api.types.is_bool_dtype(
            source.dtype
        )

        # ------------------------------------------------------------------------------------------
        # 4B. Numeric value checks
        # ------------------------------------------------------------------------------------------

        if reference_is_numeric and not reference_is_bool:

            numeric = pd.to_numeric(series, errors="coerce")

            # Non-null values that cannot be parsed as numeric.
            nonnumeric_mask = series.notna() & numeric.isna()
            nonnumeric_count = int(nonnumeric_mask.sum())

            numeric_array = numeric.to_numpy(
                dtype=float,
                na_value=np.nan,
            )

            finite_mask = np.isfinite(numeric_array)

            # Infinite positive/negative values and other non-finite,
            # non-missing numeric representations.
            non_finite_mask = (
                numeric.notna().to_numpy() & ~finite_mask
            )
            non_finite = int(non_finite_mask.sum())

            # Check only finite values for fractional values in designated
            # integer-valued features.
            integer_feature_names = {
                str(name).strip().lower()
                for name in INTEGER_FEATURES.get(dataset, set())
            }

            if normalize_column_name(column) in integer_feature_names:

                finite_values = numeric_array[finite_mask]

                if finite_values.size:
                    fractional_mask = ~np.isclose(
                        finite_values,
                        np.round(finite_values),
                        atol=FRACTIONAL_VALUE_ATOL,
                        rtol=0.0,
                    )
                    fractional_count = int(fractional_mask.sum())

        # ------------------------------------------------------------------------------------------
        # 4C. Categorical value checks
        # ------------------------------------------------------------------------------------------

        elif (
            pd.api.types.is_object_dtype(source.dtype)
            or pd.api.types.is_string_dtype(source.dtype)
            or isinstance(source.dtype, pd.CategoricalDtype)
            or reference_is_bool
        ):

            allowed_categories = set(
                source.dropna().astype(str).unique()
            )

            observed_categories = series.dropna().astype(str)

            unseen_categories = int(
                (~observed_categories.isin(allowed_categories)).sum()
            )

        # ------------------------------------------------------------------------------------------
        # 4D. Determine value status
        # ------------------------------------------------------------------------------------------

        reasons = []

        if non_finite:
            reasons.append("NON_FINITE_NUMERIC_VALUES")

        if nonnumeric_count:
            reasons.append("NON_NUMERIC_VALUES_IN_NUMERIC_COLUMN")

        if unseen_categories:
            reasons.append("UNSEEN_CATEGORIES")

        if fractional_count:
            reasons.append("FRACTIONAL_INTEGER_VALUES")

        invalid_count = (
            non_finite
            + nonnumeric_count
            + unseen_categories
            + fractional_count
        )

        value_status = "REVIEW" if reasons else "PASS"

        records.append({
            "dataset_id": dataset,
            "column": str(column),
            "column_role": "feature_or_target",
            "reference_dtype": str(source.dtype),
            "synthetic_dtype": str(series.dtype),
            "synthetic_missing_count": missing_count,
            "non_finite_numeric_count": non_finite,
            "nonnumeric_in_numeric_column_count": nonnumeric_count,
            "unseen_category_count": unseen_categories,
            "fractional_integer_value_count": fractional_count,
            "invalid_value_count": invalid_count,
            "schema_status": "PASS",
            "value_status": value_status,
            "overall_status": value_status,
            "review_reasons": ";".join(reasons) if reasons else "NONE",
        })

    # ----------------------------------------------------------------------------------------------
    # 4E. Report unexpected synthetic columns
    # ----------------------------------------------------------------------------------------------

    for column in unexpected_columns:

        series = synthetic_df[column]

        # Unexpected bookkeeping identifiers are excluded consistently.
        if is_identifier_column(column):
            records.append({
                "dataset_id": dataset,
                "column": str(column),
                "column_role": "identifier",
                "reference_dtype": None,
                "synthetic_dtype": str(series.dtype),
                "synthetic_missing_count": int(series.isna().sum()),
                "non_finite_numeric_count": np.nan,
                "nonnumeric_in_numeric_column_count": np.nan,
                "unseen_category_count": np.nan,
                "fractional_integer_value_count": np.nan,
                "invalid_value_count": np.nan,
                "schema_status": "EXCLUDED_IDENTIFIER",
                "value_status": "NOT_CHECKED",
                "overall_status": "EXCLUDED_IDENTIFIER",
                "review_reasons": (
                    "UNEXPECTED_BOOKKEEPING_IDENTIFIER_EXCLUDED"
                ),
            })
            continue

        # Unexpected non-identifier columns remain genuine schema issues.
        records.append({
            "dataset_id": dataset,
            "column": str(column),
            "column_role": "unexpected_feature",
            "reference_dtype": None,
            "synthetic_dtype": str(series.dtype),
            "synthetic_missing_count": int(series.isna().sum()),
            "non_finite_numeric_count": np.nan,
            "nonnumeric_in_numeric_column_count": np.nan,
            "unseen_category_count": np.nan,
            "fractional_integer_value_count": np.nan,
            "invalid_value_count": np.nan,
            "schema_status": "UNEXPECTED_COLUMN",
            "value_status": "NOT_CHECKED",
            "overall_status": "SCHEMA_MISMATCH",
            "review_reasons": "UNEXPECTED_SYNTHETIC_FEATURE",
        })

    return records


# --------------------------------------------------------------------------------------------------
# 5. Run validation across all datasets
# --------------------------------------------------------------------------------------------------

INVALID_VALUE_ROWS = []
INVALID_SUMMARY_ROWS = []
DATASET_SCHEMA_DETAILS = {}

for dataset in DATASET_IDS:

    reference_df = REFERENCE_TRAIN[dataset]
    synthetic_df = SYNTHETIC_DATAFRAMES_WITH_TARGET[dataset]

    records = validate_invalid_values(
        dataset=dataset,
        synthetic_df=synthetic_df,
        reference_df=reference_df,
    )

    INVALID_VALUE_ROWS.extend(records)

    dataset_df = pd.DataFrame(records)

    genuine_value_reviews = int(
        (dataset_df["value_status"] == "REVIEW").sum()
    )

    genuine_schema_mismatches = int(
        (dataset_df["schema_status"].isin([
            "MISSING_COLUMN",
            "UNEXPECTED_COLUMN",
        ])).sum()
    )

    excluded_identifier_rows = int(
        (
            dataset_df["schema_status"]
            == "EXCLUDED_IDENTIFIER"
        ).sum()
    )

    passed_feature_columns = int(
        (
            (dataset_df["value_status"] == "PASS")
            & (dataset_df["schema_status"] == "PASS")
        ).sum()
    )

    if genuine_schema_mismatches:
        dataset_status = "SCHEMA_MISMATCH"
    elif genuine_value_reviews:
        dataset_status = "REVIEW_REQUIRED"
    else:
        dataset_status = "PASS"

    INVALID_SUMMARY_ROWS.append({
        "dataset_id": dataset,
        "reference_rows": int(len(reference_df)),
        "synthetic_rows": int(len(synthetic_df)),
        "reference_columns": int(len(reference_df.columns)),
        "synthetic_columns": int(len(synthetic_df.columns)),
        "feature_columns_passed": passed_feature_columns,
        "columns_with_invalid_values": genuine_value_reviews,
        "genuine_schema_mismatch_columns": genuine_schema_mismatches,
        "excluded_identifier_rows": excluded_identifier_rows,
        "overall_status": dataset_status,
    })

    DATASET_SCHEMA_DETAILS[dataset] = {
        "reference_columns": safe_json_list(reference_df.columns),
        "synthetic_columns": safe_json_list(synthetic_df.columns),
        "missing_columns": safe_json_list([
            col for col in reference_df.columns
            if col not in synthetic_df.columns
            and not is_identifier_column(col)
        ]),
        "unexpected_columns": safe_json_list([
            col for col in synthetic_df.columns
            if col not in reference_df.columns
            and not is_identifier_column(col)
        ]),
        "excluded_reference_identifiers": safe_json_list([
            col for col in reference_df.columns
            if is_identifier_column(col)
        ]),
        "excluded_synthetic_identifiers": safe_json_list([
            col for col in synthetic_df.columns
            if is_identifier_column(col)
        ]),
        "overall_status": dataset_status,
    }

    print(
        f"{dataset}: "
        f"PASS={passed_feature_columns}, "
        f"value REVIEW={genuine_value_reviews}, "
        f"genuine schema mismatches={genuine_schema_mismatches}, "
        f"excluded identifiers={excluded_identifier_rows}, "
        f"status={dataset_status}"
    )

# --------------------------------------------------------------------------------------------------
# 6. Build final result DataFrames
# --------------------------------------------------------------------------------------------------

INVALID_VALUES_DF = pd.DataFrame(INVALID_VALUE_ROWS)
INVALID_VALUES_SUMMARY_DF = pd.DataFrame(INVALID_SUMMARY_ROWS)

if INVALID_VALUES_DF.empty:
    raise RuntimeError("Section 17 generated no validation rows.")

total_value_reviews = int(
    (INVALID_VALUES_DF["value_status"] == "REVIEW").sum()
)

total_schema_mismatches = int(
    INVALID_VALUES_DF["schema_status"].isin([
        "MISSING_COLUMN",
        "UNEXPECTED_COLUMN",
    ]).sum()
)

total_excluded_identifiers = int(
    (
        INVALID_VALUES_DF["schema_status"]
        == "EXCLUDED_IDENTIFIER"
    ).sum()
)

if total_schema_mismatches:
    overall_status = "SCHEMA_MISMATCH"
elif total_value_reviews:
    overall_status = "REVIEW_REQUIRED"
else:
    overall_status = "PASS"

# --------------------------------------------------------------------------------------------------
# 7. Save CSV artifacts
# --------------------------------------------------------------------------------------------------

INVALID_VALUES_DF.to_csv(
    INVALID_VALUES_PATH,
    index=False,
)

INVALID_VALUES_SUMMARY_DF.to_csv(
    INVALID_SUMMARY_PATH,
    index=False,
)

# --------------------------------------------------------------------------------------------------
# 8. Save metadata
# --------------------------------------------------------------------------------------------------

metadata = {
    "section": 17,
    "section_name": "Validate Invalid Values",
    "notebook": "Notebook 13 — SPP-GAN Synthetic Data Generation",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_ids": safe_json_list(DATASET_IDS),
    "detail_rows": int(len(INVALID_VALUES_DF)),
    "value_review_columns": total_value_reviews,
    "genuine_schema_mismatch_columns": total_schema_mismatches,
    "excluded_identifier_rows": total_excluded_identifiers,
    "overall_status": overall_status,
    "data_modified": False,
    "identifier_policy": (
        "Configured bookkeeping identifiers are excluded from feature "
        "schema and value validation. This is appropriate only when those "
        "columns are metadata rather than intended model features."
    ),
    "integer_value_tolerance": FRACTIONAL_VALUE_ATOL,
    "notes": [
        "Missingness rates are assessed separately in Section 16.",
        "Numeric range and semantic domain constraints are assessed in Section 15.",
        "Only configured integer-valued features are checked for fractional values.",
        "Unexpected non-identifier synthetic columns remain schema mismatches.",
        "PASS does not establish distributional fidelity, downstream utility, or privacy.",
    ],
    "datasets": DATASET_SCHEMA_DETAILS,
    "artifacts": {
        "detail_csv": str(INVALID_VALUES_PATH),
        "summary_csv": str(INVALID_SUMMARY_PATH),
        "metadata_json": str(INVALID_METADATA_PATH),
    },
}

with open(INVALID_METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

# --------------------------------------------------------------------------------------------------
# 9. Display final report
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 17 — FINAL SUMMARY")
print("=" * 100)

print(f"Datasets checked                    : {len(DATASET_IDS)}")
print(f"Detailed validation rows            : {len(INVALID_VALUES_DF)}")
print(f"Columns with invalid values         : {total_value_reviews}")
print(f"Genuine schema mismatch columns     : {total_schema_mismatches}")
print(f"Excluded identifier rows            : {total_excluded_identifiers}")
print(f"Overall status                       : {overall_status}")

print("\nSaved artifacts:")
print(f"  Detail   : {INVALID_VALUES_PATH}")
print(f"  Summary  : {INVALID_SUMMARY_PATH}")
print(f"  Metadata : {INVALID_METADATA_PATH}")

print("\nPer-dataset summary:")
display(INVALID_VALUES_SUMMARY_DF)

# Show only genuine feature-value reviews and genuine schema mismatches.
review_mask = (
    (INVALID_VALUES_DF["value_status"] == "REVIEW")
    | INVALID_VALUES_DF["schema_status"].isin([
        "MISSING_COLUMN",
        "UNEXPECTED_COLUMN",
    ])
)

review_columns = INVALID_VALUES_DF.loc[review_mask].copy()

print("\nColumns requiring review:")
if review_columns.empty:
    print("No invalid feature values or genuine schema mismatches detected.")
else:
    display(review_columns.head(50))

print("\nExcluded identifiers:")
excluded_df = INVALID_VALUES_DF[
    INVALID_VALUES_DF["schema_status"] == "EXCLUDED_IDENTIFIER"
]

if excluded_df.empty:
    print("No configured bookkeeping identifiers were found.")
else:
    display(
        excluded_df[
            [
                "dataset_id",
                "column",
                "reference_dtype",
                "synthetic_dtype",
                "schema_status",
                "review_reasons",
            ]
        ]
    )

if overall_status == "PASS":
    print(
        "\nSECTION 17 PASSED: no invalid values or genuine schema mismatches "
        "were detected by the checks implemented here."
    )
else:
    print(
        "\nSECTION 17 REQUIRES REVIEW: inspect the reported feature-value "
        "issues or genuine schema mismatches."
    )

print("\nSECTION 17 EXECUTION COMPLETED.")


SECTION 17 — VALIDATE INVALID VALUES
adult_income: PASS=15, value REVIEW=0, genuine schema mismatches=0, excluded identifiers=1, status=PASS
bank_marketing: PASS=17, value REVIEW=0, genuine schema mismatches=0, excluded identifiers=1, status=PASS
diabetes_130us: PASS=48, value REVIEW=0, genuine schema mismatches=0, excluded identifiers=1, status=PASS

SECTION 17 — FINAL SUMMARY
Datasets checked                    : 3
Detailed validation rows            : 83
Columns with invalid values         : 0
Genuine schema mismatch columns     : 0
Excluded identifier rows            : 3
Overall status                       : PASS

Saved artifacts:
  Detail   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/invalid_values_validation.csv
  Summary  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/invalid_values_validation_summary.csv
  Metadata : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/invali

,dataset_id,reference_rows,synthetic_rows,reference_columns,synthetic_columns,feature_columns_passed,columns_with_invalid_values,genuine_schema_mismatch_columns,excluded_identifier_rows,overall_status
0,adult_income,34189,34189,16,15,15,0,0,1,PASS
1,bank_marketing,31647,31647,18,17,17,0,0,1,PASS
2,diabetes_130us,71236,71236,49,48,48,0,0,1,PASS



Columns requiring review:
No invalid feature values or genuine schema mismatches detected.

Excluded identifiers:


,dataset_id,column,reference_dtype,synthetic_dtype,schema_status,review_reasons
0,adult_income,__original_row_id__,int64,None,EXCLUDED_IDENTIFIER,BOOKKEEPING_IDENTIFIER_ABSENT_BUT_EXCLUDED
16,bank_marketing,__original_row_id__,int64,None,EXCLUDED_IDENTIFIER,BOOKKEEPING_IDENTIFIER_ABSENT_BUT_EXCLUDED
34,diabetes_130us,__original_row_id__,int64,None,EXCLUDED_IDENTIFIER,BOOKKEEPING_IDENTIFIER_ABSENT_BUT_EXCLUDED



SECTION 17 PASSED: no invalid values or genuine schema mismatches were detected by the checks implemented here.

SECTION 17 EXECUTION COMPLETED.


In [36]:

# ==================================================================================================
# SECTION 18 — DUPLICATE/EXACT-MATCH CHARACTERIZATION
# Notebook 13 — SPP-GAN Synthetic Data Generation
# ==================================================================================================
# Purpose:
#   1. Measure duplicate rows within each synthetic dataset.
#   2. Measure synthetic rows that exactly match reference training records.
#   3. Exclude original row identifiers from comparisons.
#   4. Report the comparison scope and any schema differences.
#   5. Save summary, row-level match details, and metadata.
#
# Important:
#   Exact-match analysis is a diagnostic, not a complete privacy evaluation.
#   No synthetic or reference data is modified.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 18 — DUPLICATE/EXACT-MATCH CHARACTERIZATION")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Validate inputs
# --------------------------------------------------------------------------------------------------

_REQUIRED = [
    "SYNTHETIC_DATAFRAMES_WITH_TARGET",
    "REFERENCE_TRAIN",
    "DATASET_IDS",
    "NB13_VALIDATION_ROOT",
]

_missing = [name for name in _REQUIRED if name not in globals()]
if _missing:
    raise NameError(
        "Missing required objects: " + ", ".join(_missing)
        + ". Run the preceding Notebook 13 sections first."
    )

if not isinstance(SYNTHETIC_DATAFRAMES_WITH_TARGET, dict):
    raise TypeError("SYNTHETIC_DATAFRAMES_WITH_TARGET must be a dictionary.")

if not isinstance(REFERENCE_TRAIN, dict):
    raise TypeError("REFERENCE_TRAIN must be a dictionary.")

DATASET_IDS = list(DATASET_IDS)
NB13_VALIDATION_ROOT = Path(NB13_VALIDATION_ROOT)
NB13_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

SUMMARY_PATH = (
    NB13_VALIDATION_ROOT / "duplicate_exact_match_characterization.csv"
)
ROW_MATCH_PATH = (
    NB13_VALIDATION_ROOT / "duplicate_exact_match_row_details.csv"
)
METADATA_PATH = (
    NB13_VALIDATION_ROOT / "duplicate_exact_match_metadata.json"
)

IDENTIFIER_COLUMNS = {
    "__original_row_id__",
    "original_row_id",
    "row_id",
    "record_id",
}

# --------------------------------------------------------------------------------------------------
# 2. Normalize values consistently for exact row comparison
# --------------------------------------------------------------------------------------------------

def _normalized_frame(dataframe, columns):
    """
    Create a comparison-only copy with consistent string representation.
    Missing values are represented by a reserved token.
    """
    normalized = dataframe.loc[:, columns].copy()

    for column in columns:
        normalized[column] = (
            normalized[column]
            .astype("string")
            .fillna("<NA>")
            .astype(str)
        )

    return normalized


def _row_signatures(dataframe, columns):
    """
    Create row signatures for efficient candidate matching.
    Hashes are used to find candidates; exact normalized rows confirm matches.
    """
    normalized = _normalized_frame(dataframe, columns)
    hashes = pd.util.hash_pandas_object(normalized, index=False)
    return normalized, hashes


# --------------------------------------------------------------------------------------------------
# 3. Dataset-level characterization
# --------------------------------------------------------------------------------------------------

SUMMARY_ROWS = []
ROW_DETAIL_ROWS = []

for dataset in DATASET_IDS:
    if dataset not in REFERENCE_TRAIN:
        raise KeyError(f"Reference training data missing for '{dataset}'.")

    if dataset not in SYNTHETIC_DATAFRAMES_WITH_TARGET:
        raise KeyError(
            f"Synthetic data with target missing for '{dataset}'. "
            "Run Section 13 first."
        )

    reference = REFERENCE_TRAIN[dataset]
    synthetic = SYNTHETIC_DATAFRAMES_WITH_TARGET[dataset]

    if not isinstance(reference, pd.DataFrame):
        raise TypeError(f"Reference data for '{dataset}' is not a DataFrame.")

    if not isinstance(synthetic, pd.DataFrame):
        raise TypeError(f"Synthetic data for '{dataset}' is not a DataFrame.")

    # Exclude row identifiers, which should not be copied into synthetic records.
    reference_columns = [
        col for col in reference.columns
        if str(col).strip().lower() not in IDENTIFIER_COLUMNS
    ]
    synthetic_columns = [
        col for col in synthetic.columns
        if str(col).strip().lower() not in IDENTIFIER_COLUMNS
    ]

    shared_columns = [
        col for col in reference_columns if col in synthetic_columns
    ]
    missing_synthetic_columns = [
        col for col in reference_columns if col not in synthetic_columns
    ]
    unexpected_synthetic_columns = [
        col for col in synthetic_columns if col not in reference_columns
    ]

    # Within-synthetic duplicate detection on all substantive synthetic columns.
    duplicate_mask = synthetic[synthetic_columns].duplicated(keep="first")
    duplicate_count = int(duplicate_mask.sum())
    duplicate_rate = (
        float(duplicate_count / len(synthetic))
        if len(synthetic) else np.nan
    )

    # A full-record comparison is possible only when substantive schemas match.
    schemas_match = (
        set(reference_columns) == set(synthetic_columns)
        and len(reference_columns) == len(synthetic_columns)
    )

    comparison_columns = (
        reference_columns.copy()
        if schemas_match
        else shared_columns.copy()
    )

    if not comparison_columns:
        SUMMARY_ROWS.append({
            "dataset_id": dataset,
            "reference_rows": len(reference),
            "synthetic_rows": len(synthetic),
            "duplicate_rows": duplicate_count,
            "duplicate_rate": duplicate_rate,
            "real_exact_match_count": np.nan,
            "real_exact_match_rate": np.nan,
            "comparison_columns_count": 0,
            "comparison_scope": "NO_SHARED_COLUMNS",
            "schema_match": schemas_match,
            "missing_synthetic_columns": json.dumps(missing_synthetic_columns),
            "unexpected_synthetic_columns": json.dumps(unexpected_synthetic_columns),
            "exact_match_status": "NOT_COMPUTED",
        })
        continue

    # Deduplicate the reference before matching, so a reference record is represented once.
    reference_compare = reference[comparison_columns].drop_duplicates().reset_index(drop=True)
    synthetic_compare = synthetic[comparison_columns].reset_index(drop=True)

    reference_normalized, reference_hashes = _row_signatures(
        reference_compare, comparison_columns
    )
    synthetic_normalized, synthetic_hashes = _row_signatures(
        synthetic_compare, comparison_columns
    )

    # Group reference rows by hash. Confirm candidate matches using full normalized row equality.
    reference_hash_groups = {}
    for ref_idx, hash_value in enumerate(reference_hashes.tolist()):
        reference_hash_groups.setdefault(hash_value, []).append(ref_idx)

    exact_match_mask = np.zeros(len(synthetic_compare), dtype=bool)

    for syn_idx, hash_value in enumerate(synthetic_hashes.tolist()):
        candidate_indices = reference_hash_groups.get(hash_value, [])

        if not candidate_indices:
            continue

        synthetic_row = synthetic_normalized.iloc[syn_idx]

        for ref_idx in candidate_indices:
            reference_row = reference_normalized.iloc[ref_idx]

            if synthetic_row.equals(reference_row):
                exact_match_mask[syn_idx] = True
                break

    exact_match_count = int(exact_match_mask.sum())
    exact_match_rate = (
        float(exact_match_count / len(synthetic_compare))
        if len(synthetic_compare) else np.nan
    )

    comparison_scope = (
        "FULL_RECORD"
        if schemas_match
        else "SHARED_COLUMNS_ONLY"
    )

    # Save details for each synthetic row that exactly matches a reference row.
    for syn_idx in np.flatnonzero(exact_match_mask):
        ROW_DETAIL_ROWS.append({
            "dataset_id": dataset,
            "synthetic_row_position": int(syn_idx),
            "match_type": "EXACT_MATCH",
            "comparison_scope": comparison_scope,
            "comparison_columns_count": len(comparison_columns),
        })

    SUMMARY_ROWS.append({
        "dataset_id": dataset,
        "reference_rows": int(len(reference)),
        "synthetic_rows": int(len(synthetic)),
        "duplicate_rows": duplicate_count,
        "duplicate_rate": duplicate_rate,
        "real_exact_match_count": exact_match_count,
        "real_exact_match_rate": exact_match_rate,
        "comparison_columns_count": len(comparison_columns),
        "comparison_scope": comparison_scope,
        "schema_match": schemas_match,
        "missing_synthetic_columns": json.dumps(missing_synthetic_columns),
        "unexpected_synthetic_columns": json.dumps(unexpected_synthetic_columns),
        "exact_match_status": (
            "COMPUTED_FULL_RECORD"
            if schemas_match
            else "COMPUTED_SHARED_COLUMNS_ONLY"
        ),
    })

    print(
        f"{dataset}: duplicates={duplicate_count}, "
        f"exact matches={exact_match_count}, "
        f"scope={comparison_scope}"
    )

# --------------------------------------------------------------------------------------------------
# 4. Save reports and metadata
# --------------------------------------------------------------------------------------------------

DUPLICATE_DF = pd.DataFrame(SUMMARY_ROWS)
DUPLICATE_ROW_DETAILS_DF = pd.DataFrame(
    ROW_DETAIL_ROWS,
    columns=[
        "dataset_id",
        "synthetic_row_position",
        "match_type",
        "comparison_scope",
        "comparison_columns_count",
    ],
)

DUPLICATE_DF.to_csv(SUMMARY_PATH, index=False)
DUPLICATE_ROW_DETAILS_DF.to_csv(ROW_MATCH_PATH, index=False)

metadata = {
    "section": 18,
    "section_name": "Duplicate/Exact-Match Characterization",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_ids": DATASET_IDS,
    "within_synthetic_duplicate_rows": int(
        DUPLICATE_DF["duplicate_rows"].sum()
    ) if not DUPLICATE_DF.empty else 0,
    "exact_match_rows": int(
        DUPLICATE_DF["real_exact_match_count"].fillna(0).sum()
    ) if not DUPLICATE_DF.empty else 0,
    "data_modified": False,
    "hash_collisions_verified_with_row_comparison": True,
    "interpretation": (
        "Exact-match rates describe record-level equality under the reported "
        "comparison scope. They are not a complete privacy guarantee. "
        "When schemas differ, matches are computed only over shared columns."
    ),
    "artifacts": {
        "summary": str(SUMMARY_PATH),
        "matching_rows": str(ROW_MATCH_PATH),
    },
}

with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

# --------------------------------------------------------------------------------------------------
# 5. Display results
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 18 — FINAL SUMMARY")
print("=" * 100)

display(DUPLICATE_DF)

print("\nSaved artifacts:")
print(f"  Summary       : {SUMMARY_PATH}")
print(f"  Matched rows  : {ROW_MATCH_PATH}")
print(f"  Metadata      : {METADATA_PATH}")

print(
    "\nNOTE: Zero exact matches does not establish privacy. "
    "Membership-inference, attribute-inference, and other privacy evaluations "
    "remain separate research requirements."
)

print("\nSECTION 18 EXECUTION COMPLETED.")


SECTION 18 — DUPLICATE/EXACT-MATCH CHARACTERIZATION
adult_income: duplicates=0, exact matches=0, scope=FULL_RECORD
bank_marketing: duplicates=0, exact matches=0, scope=FULL_RECORD
diabetes_130us: duplicates=0, exact matches=0, scope=FULL_RECORD

SECTION 18 — FINAL SUMMARY


,dataset_id,reference_rows,synthetic_rows,duplicate_rows,duplicate_rate,real_exact_match_count,real_exact_match_rate,comparison_columns_count,comparison_scope,schema_match,missing_synthetic_columns,unexpected_synthetic_columns,exact_match_status
0,adult_income,34189,34189,0,0.0,0,0.0,15,FULL_RECORD,True,[],[],COMPUTED_FULL_RECORD
1,bank_marketing,31647,31647,0,0.0,0,0.0,17,FULL_RECORD,True,[],[],COMPUTED_FULL_RECORD
2,diabetes_130us,71236,71236,0,0.0,0,0.0,48,FULL_RECORD,True,[],[],COMPUTED_FULL_RECORD



Saved artifacts:
  Summary       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/duplicate_exact_match_characterization.csv
  Matched rows  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/duplicate_exact_match_row_details.csv
  Metadata      : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/duplicate_exact_match_metadata.json

NOTE: Zero exact matches does not establish privacy. Membership-inference, attribute-inference, and other privacy evaluations remain separate research requirements.

SECTION 18 EXECUTION COMPLETED.


In [37]:

# ==================================================================================================
# SECTION 19 — VALIDATE SAMPLE SIZE
# Notebook 13 — SPP-GAN Synthetic Data Generation
# ==================================================================================================
# Purpose:
#   1. Validate final synthetic row counts against the expected sample size.
#   2. Use SAMPLE_SIZE_OVERRIDE when explicitly configured.
#   3. Otherwise, use the corresponding reference training-data row count.
#   4. Save detailed results, summary, and metadata.
#   5. Never resize or modify the synthetic data silently.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 19 — VALIDATE SAMPLE SIZE")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Validate required inputs
# --------------------------------------------------------------------------------------------------

_REQUIRED = [
    "SYNTHETIC_DATAFRAMES_WITH_TARGET",
    "REFERENCE_TRAIN",
    "DATASET_IDS",
    "SAMPLE_SIZE_OVERRIDE",
    "NB13_VALIDATION_ROOT",
]

_missing = [name for name in _REQUIRED if name not in globals()]
if _missing:
    raise NameError(
        "Missing required objects: " + ", ".join(_missing)
        + ". Run the preceding Notebook 13 sections first."
    )

if not isinstance(SYNTHETIC_DATAFRAMES_WITH_TARGET, dict):
    raise TypeError("SYNTHETIC_DATAFRAMES_WITH_TARGET must be a dictionary.")

if not isinstance(REFERENCE_TRAIN, dict):
    raise TypeError("REFERENCE_TRAIN must be a dictionary.")

DATASET_IDS = list(DATASET_IDS)
NB13_VALIDATION_ROOT = Path(NB13_VALIDATION_ROOT)
NB13_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)

SAMPLE_SIZE_PATH = (
    NB13_VALIDATION_ROOT / "sample_size_validation.csv"
)
SAMPLE_SIZE_SUMMARY_PATH = (
    NB13_VALIDATION_ROOT / "sample_size_validation_summary.csv"
)
SAMPLE_SIZE_METADATA_PATH = (
    NB13_VALIDATION_ROOT / "sample_size_validation_metadata.json"
)

# --------------------------------------------------------------------------------------------------
# 2. Validate the sample-size override
# --------------------------------------------------------------------------------------------------

if SAMPLE_SIZE_OVERRIDE is not None:
    if (
        isinstance(SAMPLE_SIZE_OVERRIDE, (bool, np.bool_))
        or not isinstance(SAMPLE_SIZE_OVERRIDE, (int, np.integer))
        or int(SAMPLE_SIZE_OVERRIDE) <= 0
    ):
        raise ValueError(
            "SAMPLE_SIZE_OVERRIDE must be None or a positive integer."
        )

# --------------------------------------------------------------------------------------------------
# 3. Resolve expected sample size
# --------------------------------------------------------------------------------------------------

def expected_sample_count(dataset):
    """Resolve the expected synthetic row count for a dataset."""

    if SAMPLE_SIZE_OVERRIDE is not None:
        return int(SAMPLE_SIZE_OVERRIDE), "SAMPLE_SIZE_OVERRIDE"

    if dataset not in REFERENCE_TRAIN:
        raise RuntimeError(
            f"{dataset}: reference training data is unavailable; "
            "cannot determine expected sample size."
        )

    reference = REFERENCE_TRAIN[dataset]

    if not isinstance(reference, pd.DataFrame):
        raise TypeError(
            f"REFERENCE_TRAIN['{dataset}'] must be a pandas DataFrame."
        )

    expected = len(reference)

    if expected <= 0:
        raise RuntimeError(
            f"{dataset}: reference training dataset has no rows."
        )

    return int(expected), "REFERENCE_TRAIN_ROW_COUNT"

# --------------------------------------------------------------------------------------------------
# 4. Validate each final synthetic dataset
# --------------------------------------------------------------------------------------------------

SAMPLE_SIZE_ROWS = []

for dataset in DATASET_IDS:
    if dataset not in SYNTHETIC_DATAFRAMES_WITH_TARGET:
        raise KeyError(
            f"{dataset}: final synthetic data with target was not found. "
            "Run Section 13 first."
        )

    dataframe = SYNTHETIC_DATAFRAMES_WITH_TARGET[dataset]

    if not isinstance(dataframe, pd.DataFrame):
        raise TypeError(
            f"SYNTHETIC_DATAFRAMES_WITH_TARGET['{dataset}'] "
            "must be a pandas DataFrame."
        )

    expected, expected_source = expected_sample_count(dataset)
    actual = len(dataframe)
    difference = actual - expected

    SAMPLE_SIZE_ROWS.append({
        "dataset_id": dataset,
        "expected_rows": expected,
        "actual_rows": int(actual),
        "row_count_difference": int(difference),
        "expected_count_source": expected_source,
        "sample_size_status": "PASS" if actual == expected else "FAIL",
    })

SAMPLE_SIZE_DF = pd.DataFrame(SAMPLE_SIZE_ROWS)

# --------------------------------------------------------------------------------------------------
# 5. Save detailed and summary reports
# --------------------------------------------------------------------------------------------------

SAMPLE_SIZE_DF.to_csv(SAMPLE_SIZE_PATH, index=False)

pass_count = int((SAMPLE_SIZE_DF["sample_size_status"] == "PASS").sum())
fail_count = int((SAMPLE_SIZE_DF["sample_size_status"] == "FAIL").sum())

SAMPLE_SIZE_SUMMARY_DF = pd.DataFrame([{
    "datasets_checked": int(len(SAMPLE_SIZE_DF)),
    "datasets_passed": pass_count,
    "datasets_failed": fail_count,
    "total_expected_rows": int(SAMPLE_SIZE_DF["expected_rows"].sum()),
    "total_actual_rows": int(SAMPLE_SIZE_DF["actual_rows"].sum()),
    "overall_status": "PASS" if fail_count == 0 else "FAIL",
}])

SAMPLE_SIZE_SUMMARY_DF.to_csv(
    SAMPLE_SIZE_SUMMARY_PATH,
    index=False,
)

metadata = {
    "section": 19,
    "section_name": "Validate Sample Size",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_ids": DATASET_IDS,
    "sample_size_override": (
        int(SAMPLE_SIZE_OVERRIDE)
        if SAMPLE_SIZE_OVERRIDE is not None else None
    ),
    "datasets_checked": int(len(SAMPLE_SIZE_DF)),
    "datasets_passed": pass_count,
    "datasets_failed": fail_count,
    "overall_status": "PASS" if fail_count == 0 else "FAIL",
    "data_modified": False,
    "artifacts": {
        "detail": str(SAMPLE_SIZE_PATH),
        "summary": str(SAMPLE_SIZE_SUMMARY_PATH),
    },
}

with open(SAMPLE_SIZE_METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, ensure_ascii=False)

# --------------------------------------------------------------------------------------------------
# 6. Display results
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 19 — FINAL SUMMARY")
print("=" * 100)

print(f"Datasets checked     : {len(SAMPLE_SIZE_DF)}")
print(f"Datasets passed      : {pass_count}")
print(f"Datasets failed      : {fail_count}")
print(f"Overall status       : {metadata['overall_status']}")

print("\nPer-dataset results:")
display(SAMPLE_SIZE_DF)

print("\nSummary:")
display(SAMPLE_SIZE_SUMMARY_DF)

print("\nSaved artifacts:")
print(f"  Detail   : {SAMPLE_SIZE_PATH}")
print(f"  Summary  : {SAMPLE_SIZE_SUMMARY_PATH}")
print(f"  Metadata : {SAMPLE_SIZE_METADATA_PATH}")

if fail_count:
    print(
        "\nWARNING: One or more datasets have incorrect sample sizes. "
        "Investigate the generation configuration before proceeding."
    )
else:
    print(
        "\nAll final synthetic datasets match their expected row counts. "
        "This validates sample size only, not data quality or privacy."
    )

print("\nSECTION 19 EXECUTION COMPLETED.")


SECTION 19 — VALIDATE SAMPLE SIZE

SECTION 19 — FINAL SUMMARY
Datasets checked     : 3
Datasets passed      : 3
Datasets failed      : 0
Overall status       : PASS

Per-dataset results:


,dataset_id,expected_rows,actual_rows,row_count_difference,expected_count_source,sample_size_status
0,adult_income,34189,34189,0,REFERENCE_TRAIN_ROW_COUNT,PASS
1,bank_marketing,31647,31647,0,REFERENCE_TRAIN_ROW_COUNT,PASS
2,diabetes_130us,71236,71236,0,REFERENCE_TRAIN_ROW_COUNT,PASS



Summary:


,datasets_checked,datasets_passed,datasets_failed,total_expected_rows,total_actual_rows,overall_status
0,3,3,0,137072,137072,PASS



Saved artifacts:
  Detail   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/sample_size_validation.csv
  Summary  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/sample_size_validation_summary.csv
  Metadata : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/sample_size_validation_metadata.json

All final synthetic datasets match their expected row counts. This validates sample size only, not data quality or privacy.

SECTION 19 EXECUTION COMPLETED.


In [38]:

# ==================================================================================================
# SECTION 20 — RECORD GENERATION RUNTIME
# Notebook 13 — SPP-GAN Synthetic Data Generation
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 20 — RECORD GENERATION RUNTIME")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Validate inputs
# --------------------------------------------------------------------------------------------------

_REQUIRED = [
    "SYNTHETIC_DATAFRAMES_WITH_TARGET",
    "DATASET_IDS",
    "GENERATION_RUNTIME",
    "NB13_METADATA_ROOT",
]

_missing = [name for name in _REQUIRED if name not in globals()]
if _missing:
    raise NameError(
        "Missing required objects: " + ", ".join(_missing)
    )

if not isinstance(GENERATION_RUNTIME, dict):
    raise TypeError("GENERATION_RUNTIME must be a dictionary.")

NB13_METADATA_ROOT = Path(NB13_METADATA_ROOT)
NB13_METADATA_ROOT.mkdir(parents=True, exist_ok=True)

RUNTIME_REPORT_PATH = (
    NB13_METADATA_ROOT / "generation_runtime_summary.csv"
)
RUNTIME_SUMMARY_PATH = (
    NB13_METADATA_ROOT / "generation_runtime_overall_summary.csv"
)
RUNTIME_METADATA_PATH = (
    NB13_METADATA_ROOT / "generation_runtime_metadata.json"
)

# --------------------------------------------------------------------------------------------------
# 2. Normalize runtime records
# --------------------------------------------------------------------------------------------------
# Supported key formats:
#   "adult_income"             -> one runtime measurement
#   ("adult_income", 1)        -> dataset and repetition
#
# If several repetitions exist, report their mean, minimum, maximum, and count.
# Do not treat repeated measurements as independent datasets.

runtime_by_dataset = {dataset: [] for dataset in DATASET_IDS}
unrecognized_runtime_keys = []

for key, raw_seconds in GENERATION_RUNTIME.items():

    if isinstance(key, tuple) and len(key) >= 1:
        dataset = key[0]
        repetition = key[1] if len(key) >= 2 else None
    else:
        dataset = key
        repetition = None

    if dataset not in runtime_by_dataset:
        unrecognized_runtime_keys.append(str(key))
        continue

    try:
        seconds = float(raw_seconds)
    except (TypeError, ValueError):
        continue

    if not np.isfinite(seconds) or seconds < 0:
        continue

    runtime_by_dataset[dataset].append({
        "repetition": repetition,
        "seconds": seconds,
        "original_key": str(key),
    })

# --------------------------------------------------------------------------------------------------
# 3. Build per-dataset runtime records
# --------------------------------------------------------------------------------------------------

RUNTIME_ROWS = []

for dataset in DATASET_IDS:

    if dataset not in SYNTHETIC_DATAFRAMES_WITH_TARGET:
        raise KeyError(
            f"Synthetic dataset '{dataset}' not found."
        )

    dataframe = SYNTHETIC_DATAFRAMES_WITH_TARGET[dataset]
    measurements = runtime_by_dataset.get(dataset, [])

    if not isinstance(dataframe, pd.DataFrame):
        raise TypeError(
            f"Synthetic dataset '{dataset}' must be a pandas DataFrame."
        )

    seconds_values = [item["seconds"] for item in measurements]
    generated_rows = int(len(dataframe))

    if seconds_values:
        # The first measurement is the recorded runtime for the single-run case.
        # With multiple repetitions, mean/min/max are reported transparently.
        mean_seconds = float(np.mean(seconds_values))
        min_seconds = float(np.min(seconds_values))
        max_seconds = float(np.max(seconds_values))

        runtime_status = "RECORDED"
        seconds_per_1000 = (
            mean_seconds / generated_rows * 1000
            if generated_rows > 0
            else np.nan
        )
    else:
        mean_seconds = np.nan
        min_seconds = np.nan
        max_seconds = np.nan
        seconds_per_1000 = np.nan
        runtime_status = "MISSING_OR_INVALID"

    RUNTIME_ROWS.append({
        "dataset_id": dataset,
        "generation_runtime_seconds": mean_seconds,
        "runtime_min_seconds": min_seconds,
        "runtime_max_seconds": max_seconds,
        "runtime_measurement_count": len(seconds_values),
        "generated_rows": generated_rows,
        "seconds_per_1000_rows": seconds_per_1000,
        "runtime_status": runtime_status,
        "runtime_source": (
            "GENERATION_RUNTIME_TUPLE_OR_DATASET_KEY"
            if seconds_values
            else "NO_VALID_RECORDED_MEASUREMENT"
        ),
    })

RUNTIME_DF = pd.DataFrame(RUNTIME_ROWS)

# --------------------------------------------------------------------------------------------------
# 4. Save detailed report
# --------------------------------------------------------------------------------------------------

RUNTIME_DF.to_csv(RUNTIME_REPORT_PATH, index=False)

recorded = RUNTIME_DF[
    RUNTIME_DF["runtime_status"] == "RECORDED"
]
missing_count = int(
    (RUNTIME_DF["runtime_status"] != "RECORDED").sum()
)

total_recorded_runtime = (
    float(recorded["generation_runtime_seconds"].sum())
    if not recorded.empty
    else np.nan
)

RUNTIME_SUMMARY_DF = pd.DataFrame([{
    "datasets_checked": len(RUNTIME_DF),
    "datasets_with_recorded_runtime": len(recorded),
    "datasets_with_missing_runtime": missing_count,
    "total_generated_rows": int(RUNTIME_DF["generated_rows"].sum()),
    "sum_of_dataset_mean_runtimes_seconds": total_recorded_runtime,
    "overall_status": (
        "PASS" if missing_count == 0 else "REVIEW_REQUIRED"
    ),
}])

RUNTIME_SUMMARY_DF.to_csv(
    RUNTIME_SUMMARY_PATH,
    index=False,
)

# --------------------------------------------------------------------------------------------------
# 5. Save metadata
# --------------------------------------------------------------------------------------------------

runtime_metadata = {
    "section": 20,
    "section_name": "Record Generation Runtime",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_ids": list(DATASET_IDS),
    "runtime_key_format": "dataset or (dataset, repetition)",
    "datasets_with_recorded_runtime": int(len(recorded)),
    "datasets_with_missing_runtime": missing_count,
    "overall_status": (
        "PASS" if missing_count == 0 else "REVIEW_REQUIRED"
    ),
    "unrecognized_runtime_keys": unrecognized_runtime_keys,
    "data_modified": False,
    "artifacts": {
        "detail": str(RUNTIME_REPORT_PATH),
        "summary": str(RUNTIME_SUMMARY_PATH),
    },
}

with open(RUNTIME_METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(runtime_metadata, file, indent=2, ensure_ascii=False)

# --------------------------------------------------------------------------------------------------
# 6. Display results
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 20 — FINAL SUMMARY")
print("=" * 100)

print(f"Datasets checked             : {len(RUNTIME_DF)}")
print(f"Runtime measurements present : {len(recorded)}")
print(f"Runtime measurements missing : {missing_count}")
print(f"Overall status               : {runtime_metadata['overall_status']}")

print("\nPer-dataset runtime:")
display(RUNTIME_DF)

print("\nOverall summary:")
display(RUNTIME_SUMMARY_DF)

print("\nSaved artifacts:")
print(f"  Detail   : {RUNTIME_REPORT_PATH}")
print(f"  Summary  : {RUNTIME_SUMMARY_PATH}")
print(f"  Metadata : {RUNTIME_METADATA_PATH}")

if missing_count:
    print("\nWARNING: Some runtime measurements remain missing.")
else:
    print(
        "\nAll datasets have recorded generation runtimes. "
        "These measurements were retrieved from the existing runtime dictionary."
    )

print("\nSECTION 20 EXECUTION COMPLETED.")


SECTION 20 — RECORD GENERATION RUNTIME

SECTION 20 — FINAL SUMMARY
Datasets checked             : 3
Runtime measurements present : 3
Runtime measurements missing : 0
Overall status               : PASS

Per-dataset runtime:


,dataset_id,generation_runtime_seconds,runtime_min_seconds,runtime_max_seconds,runtime_measurement_count,generated_rows,seconds_per_1000_rows,runtime_status,runtime_source
0,adult_income,11.550080,11.550080,11.550080,1,34189,0.337830,RECORDED,GENERATION_RUNTIME_TUPLE_OR_DATASET_KEY
1,bank_marketing,9.987916,9.987916,9.987916,1,31647,0.315604,RECORDED,GENERATION_RUNTIME_TUPLE_OR_DATASET_KEY
2,diabetes_130us,57.020966,57.020966,57.020966,1,71236,0.800452,RECORDED,GENERATION_RUNTIME_TUPLE_OR_DATASET_KEY



Overall summary:


,datasets_checked,datasets_with_recorded_runtime,datasets_with_missing_runtime,total_generated_rows,sum_of_dataset_mean_runtimes_seconds,overall_status
0,3,3,0,137072,78.558962,PASS



Saved artifacts:
  Detail   : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/metadata/generation_runtime_summary.csv
  Summary  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/metadata/generation_runtime_overall_summary.csv
  Metadata : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/metadata/generation_runtime_metadata.json

All datasets have recorded generation runtimes. These measurements were retrieved from the existing runtime dictionary.

SECTION 20 EXECUTION COMPLETED.


In [39]:

# =============================================================================
# SECTION 21 — MODEL ARTIFACT FINGERPRINTS
# =============================================================================

import hashlib
import json
from pathlib import Path
import pandas as pd

MODEL_FINGERPRINT_ROWS = []

for dataset, repetitions in MODEL_ARTIFACTS.items():
    if isinstance(repetitions, dict):
        items = repetitions.items()
    else:
        items = [("1", repetitions)]

    for repetition, model_value in items:
        model_path = None
        model_hash = None
        status = "NOT_FOUND"

        # Model metadata may contain a path or a nested dictionary.
        candidates = (
            list(model_value.values())
            if isinstance(model_value, dict)
            else [model_value]
        )

        for candidate in candidates:
            if not isinstance(candidate, (str, Path)):
                continue

            candidate_path = Path(candidate).expanduser()

            if candidate_path.is_file():
                model_path = candidate_path.resolve()
                break

        if model_path is not None:
            try:
                digest = hashlib.sha256()

                with model_path.open("rb") as handle:
                    for chunk in iter(
                        lambda: handle.read(1024 * 1024), b""
                    ):
                        digest.update(chunk)

                model_hash = digest.hexdigest()
                status = "HASHED"

            except OSError as exc:
                status = f"HASH_ERROR: {type(exc).__name__}"

        MODEL_FINGERPRINT_ROWS.append({
            "dataset": dataset,
            "repetition": str(repetition),
            "model_path": str(model_path) if model_path else None,
            "model_sha256": model_hash,
            "model_fingerprint_status": status,
        })

MODEL_FINGERPRINT_DF = pd.DataFrame(MODEL_FINGERPRINT_ROWS)

model_fingerprint_path = (
    Path(NB13_METADATA_ROOT) / "model_fingerprints.csv"
)
model_fingerprint_path.parent.mkdir(parents=True, exist_ok=True)
MODEL_FINGERPRINT_DF.to_csv(model_fingerprint_path, index=False)

print("=" * 100)
print("SECTION 21 — MODEL FINGERPRINT SUMMARY")
print("=" * 100)
print("Model entries checked :", len(MODEL_FINGERPRINT_DF))
print(
    "Model files hashed    :",
    int((MODEL_FINGERPRINT_DF["model_fingerprint_status"] == "HASHED").sum()),
)
print(
    "Model files unresolved:",
    int((MODEL_FINGERPRINT_DF["model_fingerprint_status"] != "HASHED").sum()),
)
print("Saved artifact        :", model_fingerprint_path)

display(MODEL_FINGERPRINT_DF)


SECTION 21 — MODEL FINGERPRINT SUMMARY
Model entries checked : 15
Model files hashed    : 15
Model files unresolved: 0
Saved artifact        : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/metadata/model_fingerprints.csv


,dataset,repetition,model_path,model_sha256,model_fingerprint_status
0,adult_income,1,/content/drive/MyDrive/SPP_GAN_Research/result...,3147dc0dd607f0b242d35daed9e4c7a17830f9589fe29b...,HASHED
1,adult_income,2,/content/drive/MyDrive/SPP_GAN_Research/result...,60e8d6409ddd8f5f2e9a82cd758829a736e46c4921913e...,HASHED
2,adult_income,3,/content/drive/MyDrive/SPP_GAN_Research/result...,48e62c40e789f84822f4c26b75137464996fbff3274b8e...,HASHED
3,adult_income,4,/content/drive/MyDrive/SPP_GAN_Research/result...,183431ff4d80d2d1228f9adad8470986db81c1d5b664d3...,HASHED
4,adult_income,5,/content/drive/MyDrive/SPP_GAN_Research/result...,aa1828fd4b41c1450c7c447f1c4206c1a95213bf9fe0fa...,HASHED
5,bank_marketing,1,/content/drive/MyDrive/SPP_GAN_Research/result...,5bb3cc6fcea00cb21f417257963552a16c5e7faaa149ba...,HASHED
6,bank_marketing,2,/content/drive/MyDrive/SPP_GAN_Research/result...,e9055c31fab00a8be54c818a8cde714572680908ae9c75...,HASHED
7,bank_marketing,3,/content/drive/MyDrive/SPP_GAN_Research/result...,797dd47e7c97106cce5d02bc52da26bbc9408567a37877...,HASHED
8,bank_marketing,4,/content/drive/MyDrive/SPP_GAN_Research/result...,fed615ef19110561dddae97a6d67d63950dcb67adece08...,HASHED
9,bank_marketing,5,/content/drive/MyDrive/SPP_GAN_Research/result...,9ef74c66d380fbbbb5e33c900c00e23f74dcd7c76e9005...,HASHED


In [40]:

# =============================================================================
# SECTION 22 — SAVE SYNTHETIC DATASETS
# =============================================================================

from pathlib import Path
import gc
import json
import pandas as pd

print("=" * 100)
print("SECTION 22 — SAVE SYNTHETIC DATASETS")
print("=" * 100)

# -----------------------------------------------------------------------------
# 1. Select final synthetic datasets
# -----------------------------------------------------------------------------

if (
    "SYNTHETIC_DATAFRAMES_WITH_TARGET" not in globals()
    or not SYNTHETIC_DATAFRAMES_WITH_TARGET
):
    raise RuntimeError(
        "Final target-restored synthetic datasets are unavailable. "
        "Run Section 13 before saving."
    )

FINAL_SYNTHETIC_DATAFRAMES = SYNTHETIC_DATAFRAMES_WITH_TARGET

NB13_SYNTHETIC_ROOT = Path(NB13_SYNTHETIC_ROOT)
NB13_SYNTHETIC_ROOT.mkdir(parents=True, exist_ok=True)

SAVED_DATASET_PATHS = {}
SAVE_VALIDATION_ROWS = []

# -----------------------------------------------------------------------------
# 2. Save and verify each final dataset
# -----------------------------------------------------------------------------

for dataset, dataframe in FINAL_SYNTHETIC_DATAFRAMES.items():

    csv_path = (
        NB13_SYNTHETIC_ROOT
        / f"{dataset}_sppgan_synthetic.csv"
    )

    dataframe.to_csv(csv_path, index=False)

    if not csv_path.is_file() or csv_path.stat().st_size == 0:
        raise RuntimeError(
            f"{dataset}: CSV output is missing or empty."
        )

    # Reload for independent structural verification.
    reloaded = pd.read_csv(csv_path, low_memory=False)

    row_match = len(reloaded) == len(dataframe)
    column_match = list(reloaded.columns) == list(dataframe.columns)

    if not row_match:
        raise RuntimeError(
            f"{dataset}: saved CSV row count mismatch."
        )

    if not column_match:
        raise RuntimeError(
            f"{dataset}: saved CSV column mismatch."
        )

    # Confirm the target column is present.
    target_column = (
        DATASET_CONFIG[dataset].get("target")
        if "DATASET_CONFIG" in globals()
        and isinstance(DATASET_CONFIG.get(dataset), dict)
        else None
    )

    if target_column is None:
        target_column = {
            "adult_income": "income",
            "bank_marketing": "y",
            "diabetes_130us": "readmitted",
        }.get(dataset)

    target_present = (
        target_column in reloaded.columns
        if target_column is not None
        else False
    )

    if not target_present:
        raise RuntimeError(
            f"{dataset}: expected target column "
            f"{target_column!r} is absent from the saved dataset."
        )

    # Optional Parquet export.
    parquet_path = None
    parquet_status = "DISABLED"

    if SAVE_PARQUET:
        parquet_path = (
            NB13_SYNTHETIC_ROOT
            / f"{dataset}_sppgan_synthetic.parquet"
        )

        try:
            dataframe.to_parquet(parquet_path, index=False)

            if not parquet_path.is_file() or parquet_path.stat().st_size == 0:
                raise RuntimeError("Parquet file is missing or empty.")

            parquet_status = "SAVED"

        except Exception as exc:
            parquet_status = f"FAILED: {type(exc).__name__}"
            print(
                f"WARNING: Parquet not saved for {dataset}: {exc}"
            )

    SAVED_DATASET_PATHS[dataset] = csv_path

    SAVE_VALIDATION_ROWS.append({
        "dataset": dataset,
        "rows": len(reloaded),
        "columns": len(reloaded.columns),
        "target_column": target_column,
        "target_present": target_present,
        "row_count_match": row_match,
        "column_match": column_match,
        "csv_size_bytes": csv_path.stat().st_size,
        "csv_path": str(csv_path),
        "parquet_status": parquet_status,
        "save_status": "PASS",
    })

    print(
        f"PASS | {dataset}: {reloaded.shape} "
        f"| target={target_column} "
        f"| CSV={csv_path}"
    )

    del reloaded
    gc.collect()

# -----------------------------------------------------------------------------
# 3. Save verification report
# -----------------------------------------------------------------------------

SAVE_VALIDATION_DF = pd.DataFrame(SAVE_VALIDATION_ROWS)

validation_path = (
    Path(NB13_METADATA_ROOT)
    / "synthetic_dataset_save_validation.csv"
)
validation_path.parent.mkdir(parents=True, exist_ok=True)

SAVE_VALIDATION_DF.to_csv(validation_path, index=False)

overall_status = (
    "PASS"
    if len(SAVE_VALIDATION_DF) == len(FINAL_SYNTHETIC_DATAFRAMES)
    and SAVE_VALIDATION_DF["save_status"].eq("PASS").all()
    else "REVIEW"
)

summary = {
    "datasets_expected": len(FINAL_SYNTHETIC_DATAFRAMES),
    "datasets_saved": len(SAVED_DATASET_PATHS),
    "total_rows_saved": int(SAVE_VALIDATION_DF["rows"].sum()),
    "all_targets_present": bool(
        SAVE_VALIDATION_DF["target_present"].all()
    ),
    "overall_status": overall_status,
    "dataset_scope": "FINAL_SYNTHETIC_DATA_WITH_TARGET",
    "validation_report": str(validation_path),
}

summary_path = (
    Path(NB13_METADATA_ROOT)
    / "synthetic_dataset_save_summary.json"
)

with summary_path.open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)

# -----------------------------------------------------------------------------
# 4. Final summary
# -----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 22 — FINAL SUMMARY")
print("=" * 100)

print("Datasets saved       :", summary["datasets_saved"])
print("Total rows saved     :", summary["total_rows_saved"])
print("All targets present  :", summary["all_targets_present"])
print("Overall status       :", summary["overall_status"])
print("Validation report    :", validation_path)
print("Summary metadata     :", summary_path)

display(SAVE_VALIDATION_DF)

print("\nSECTION 22 EXECUTION COMPLETED.")


SECTION 22 — SAVE SYNTHETIC DATASETS
PASS | adult_income: (34189, 15) | target=income | CSV=/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/synthetic_datasets/adult_income_sppgan_synthetic.csv
PASS | bank_marketing: (31647, 17) | target=y | CSV=/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/synthetic_datasets/bank_marketing_sppgan_synthetic.csv
PASS | diabetes_130us: (71236, 48) | target=readmitted | CSV=/content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/synthetic_datasets/diabetes_130us_sppgan_synthetic.csv

SECTION 22 — FINAL SUMMARY
Datasets saved       : 3
Total rows saved     : 137072
All targets present  : True
Overall status       : PASS
Validation report    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/metadata/synthetic_dataset_save_validation.csv
Summary metadata     : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/metadata/synthetic_dataset_save_summary.json


,dataset,rows,columns,target_column,target_present,row_count_match,column_match,csv_size_bytes,csv_path,parquet_status,save_status
0,adult_income,34189,15,income,True,True,True,3823288,/content/drive/MyDrive/SPP_GAN_Research/result...,DISABLED,PASS
1,bank_marketing,31647,17,y,True,True,True,2624562,/content/drive/MyDrive/SPP_GAN_Research/result...,DISABLED,PASS
2,diabetes_130us,71236,48,readmitted,True,True,True,12945043,/content/drive/MyDrive/SPP_GAN_Research/result...,DISABLED,PASS



SECTION 22 EXECUTION COMPLETED.


In [41]:

# =============================================================================
# SECTION 23 — SAVE GENERATION METADATA
# =============================================================================

import json
import platform
import hashlib
from datetime import datetime, timezone
from pathlib import Path

print("=" * 100)
print("SECTION 23 — SAVE GENERATION METADATA")
print("=" * 100)

# -----------------------------------------------------------------------------
# 1. Select final synthetic datasets
# -----------------------------------------------------------------------------

if (
    "SYNTHETIC_DATAFRAMES_WITH_TARGET" not in globals()
    or not SYNTHETIC_DATAFRAMES_WITH_TARGET
):
    raise RuntimeError(
        "Final synthetic datasets are unavailable. Run Section 13 first."
    )

FINAL_METADATA_DATAFRAMES = SYNTHETIC_DATAFRAMES_WITH_TARGET

# -----------------------------------------------------------------------------
# 2. Robust file fingerprint helper
# -----------------------------------------------------------------------------

def metadata_file_sha256(path):
    if not path:
        return None

    try:
        path = Path(path)
        if not path.is_file():
            return None

        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for chunk in iter(
                lambda: handle.read(1024 * 1024), b""
            ):
                digest.update(chunk)

        return digest.hexdigest()

    except (OSError, TypeError, ValueError):
        return None


# -----------------------------------------------------------------------------
# 3. Normalize generation runtime lookup
# -----------------------------------------------------------------------------

def get_dataset_runtime(dataset):
    runtime_registry = globals().get("GENERATION_RUNTIME", {})

    values = []

    if isinstance(runtime_registry, dict):
        for key, value in runtime_registry.items():
            matches = (
                key == dataset
                or (
                    isinstance(key, tuple)
                    and len(key) >= 1
                    and key[0] == dataset
                )
            )

            if matches:
                try:
                    runtime = float(value)
                    if np.isfinite(runtime) and runtime >= 0:
                        values.append(runtime)
                except (TypeError, ValueError):
                    continue

    if values:
        return {
            "runtime_seconds": float(np.mean(values)),
            "runtime_measurement_count": len(values),
            "runtime_min_seconds": float(np.min(values)),
            "runtime_max_seconds": float(np.max(values)),
        }

    return {
        "runtime_seconds": None,
        "runtime_measurement_count": 0,
        "runtime_min_seconds": None,
        "runtime_max_seconds": None,
    }


# -----------------------------------------------------------------------------
# 4. Build dataset-level generation metadata
# -----------------------------------------------------------------------------

GENERATION_METADATA = {
    "framework": "SPP-GAN",
    "notebook": str(globals().get("NOTEBOOK_NAME", "Notebook 13")),
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "master_seed": MASTER_SEED,
    "generation_repetitions": GENERATION_REPETITIONS,
    "generation_batch_size": GENERATION_BATCH_SIZE,
    "device": str(DEVICE),
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "torch_version": torch.__version__,
    "cuda_available": bool(torch.cuda.is_available()),
    "configuration_path": str(GENERATION_CONFIG_PATH),
    "dataset_scope": "FINAL_SYNTHETIC_DATA_WITH_TARGET",
    "datasets": {},
}

for dataset, dataframe in FINAL_METADATA_DATAFRAMES.items():

    # Each dataset can have multiple repetition-specific model artifacts.
    artifact_registry = globals().get("MODEL_ARTIFACTS", {})
    dataset_artifacts = artifact_registry.get(dataset, {})

    model_artifacts = []

    if isinstance(dataset_artifacts, dict):
        model_items = dataset_artifacts.items()
    elif isinstance(dataset_artifacts, (str, Path)):
        model_items = [("1", dataset_artifacts)]
    else:
        model_items = []

    for repetition, artifact in model_items:
        candidate_paths = (
            list(artifact.values())
            if isinstance(artifact, dict)
            else [artifact]
        )

        for candidate in candidate_paths:
            if not isinstance(candidate, (str, Path)):
                continue

            candidate_path = Path(candidate).expanduser()

            if candidate_path.is_file():
                model_artifacts.append({
                    "repetition": str(repetition),
                    "model_path": str(candidate_path.resolve()),
                    "model_sha256": metadata_file_sha256(candidate_path),
                })
                break

    runtime_info = get_dataset_runtime(dataset)

    target_column = TARGET_COLUMNS.get(dataset)

    if not target_column:
        target_column = {
            "adult_income": "income",
            "bank_marketing": "y",
            "diabetes_130us": "readmitted",
        }.get(dataset)

    csv_path = SAVED_DATASET_PATHS.get(dataset)

    preprocessor_path = (
        PREPROCESSOR_PATHS.get(dataset)
        if "PREPROCESSOR_PATHS" in globals()
        else None
    )

    schema_path = (
        SCHEMA_PATHS.get(dataset)
        if "SCHEMA_PATHS" in globals()
        else None
    )

    GENERATION_METADATA["datasets"][dataset] = {
        "rows": int(len(dataframe)),
        "columns": [str(column) for column in dataframe.columns],
        "column_count": int(len(dataframe.columns)),
        "target_column": target_column,
        "target_present": (
            target_column in dataframe.columns
            if target_column else False
        ),
        "model_artifacts": model_artifacts,
        "model_artifact_count": len(model_artifacts),
        "synthetic_dataframe_sha256": dataframe_sha256(dataframe),
        "seed": generation_seed(dataset, 0),
        **runtime_info,
        "synthetic_csv": str(csv_path) if csv_path else None,
        "synthetic_csv_sha256": metadata_file_sha256(csv_path),
        "preprocessor_path": (
            str(preprocessor_path) if preprocessor_path else None
        ),
        "schema_path": str(schema_path) if schema_path else None,
    }

# -----------------------------------------------------------------------------
# 5. Save metadata and verify JSON integrity
# -----------------------------------------------------------------------------

GENERATION_METADATA_PATH = (
    Path(NB13_METADATA_ROOT)
    / "sppgan_synthetic_generation_metadata.json"
)
GENERATION_METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)

with GENERATION_METADATA_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        GENERATION_METADATA,
        handle,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

# Reload to confirm the metadata is valid JSON.
with GENERATION_METADATA_PATH.open("r", encoding="utf-8") as handle:
    RELOADED_GENERATION_METADATA = json.load(handle)

# -----------------------------------------------------------------------------
# 6. Final summary
# -----------------------------------------------------------------------------

METADATA_SUMMARY_ROWS = []

for dataset, metadata in RELOADED_GENERATION_METADATA["datasets"].items():
    METADATA_SUMMARY_ROWS.append({
        "dataset": dataset,
        "rows": metadata["rows"],
        "columns": metadata["column_count"],
        "target_column": metadata["target_column"],
        "target_present": metadata["target_present"],
        "model_artifacts_found": metadata["model_artifact_count"],
        "synthetic_hash_present": bool(
            metadata["synthetic_dataframe_sha256"]
        ),
        "csv_hash_present": bool(metadata["synthetic_csv_sha256"]),
        "runtime_seconds": metadata["runtime_seconds"],
        "runtime_measurement_count": metadata[
            "runtime_measurement_count"
        ],
    })

METADATA_SUMMARY_DF = pd.DataFrame(METADATA_SUMMARY_ROWS)

print("\nSECTION 23 — FINAL SUMMARY")
print("=" * 100)
print("Datasets documented :", len(METADATA_SUMMARY_DF))
print(
    "Targets present     :",
    int(METADATA_SUMMARY_DF["target_present"].sum()),
    "/",
    len(METADATA_SUMMARY_DF),
)
print(
    "Model artifacts     :",
    int(METADATA_SUMMARY_DF["model_artifacts_found"].sum()),
)
print(
    "Synthetic hashes    :",
    int(METADATA_SUMMARY_DF["synthetic_hash_present"].sum()),
    "/",
    len(METADATA_SUMMARY_DF),
)
print("JSON verified       :", GENERATION_METADATA_PATH.is_file())
print("Metadata saved      :", GENERATION_METADATA_PATH)

display(METADATA_SUMMARY_DF)

print("\nSECTION 23 EXECUTION COMPLETED.")


SECTION 23 — SAVE GENERATION METADATA

SECTION 23 — FINAL SUMMARY
Datasets documented : 3
Targets present     : 3 / 3
Model artifacts     : 15
Synthetic hashes    : 3 / 3
JSON verified       : True
Metadata saved      : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/metadata/sppgan_synthetic_generation_metadata.json


,dataset,rows,columns,target_column,target_present,model_artifacts_found,synthetic_hash_present,csv_hash_present,runtime_seconds,runtime_measurement_count
0,adult_income,34189,15,income,True,5,True,True,11.550080,1
1,bank_marketing,31647,17,y,True,5,True,True,9.987916,1
2,diabetes_130us,71236,48,readmitted,True,5,True,True,57.020966,1



SECTION 23 EXECUTION COMPLETED.


In [42]:
# =============================================================================
# SECTION 24 — SAVE GENERATION MANIFEST
# =============================================================================

import json
from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 100)
print("SECTION 24 — SAVE GENERATION MANIFEST")
print("=" * 100)

# -----------------------------------------------------------------------------
# 1. Select final synthetic datasets
# -----------------------------------------------------------------------------

if (
    "SYNTHETIC_DATAFRAMES_WITH_TARGET" not in globals()
    or not SYNTHETIC_DATAFRAMES_WITH_TARGET
):
    raise RuntimeError(
        "Final target-restored datasets are unavailable. "
        "Run Section 13 before generating the manifest."
    )

FINAL_MANIFEST_DATAFRAMES = SYNTHETIC_DATAFRAMES_WITH_TARGET

# -----------------------------------------------------------------------------
# 2. Resolve model artifacts and runtime measurements
# -----------------------------------------------------------------------------

def manifest_runtime_info(dataset):
    registry = globals().get("GENERATION_RUNTIME", {})
    values = []

    if isinstance(registry, dict):
        for key, value in registry.items():
            if (
                key == dataset
                or (
                    isinstance(key, tuple)
                    and len(key) > 0
                    and key[0] == dataset
                )
            ):
                try:
                    value = float(value)
                    if np.isfinite(value) and value >= 0:
                        values.append(value)
                except (TypeError, ValueError):
                    pass

    if not values:
        return {
            "runtime_seconds": None,
            "runtime_measurement_count": 0,
        }

    return {
        "runtime_seconds": float(np.mean(values)),
        "runtime_measurement_count": len(values),
    }


def manifest_model_artifacts(dataset):
    registry = globals().get("MODEL_ARTIFACTS", {})
    artifact_info = registry.get(dataset, {})
    artifacts = []

    if isinstance(artifact_info, dict):
        items = artifact_info.items()
    elif isinstance(artifact_info, (str, Path)):
        items = [("1", artifact_info)]
    else:
        items = []

    for repetition, value in items:
        candidates = (
            list(value.values())
            if isinstance(value, dict)
            else [value]
        )

        for candidate in candidates:
            if not isinstance(candidate, (str, Path)):
                continue

            path = Path(candidate).expanduser()

            if path.is_file():
                artifacts.append({
                    "repetition": str(repetition),
                    "model_path": str(path.resolve()),
                    "model_sha256": sha256_file(path),
                })
                break

    return artifacts


# -----------------------------------------------------------------------------
# 3. Build manifest rows
# -----------------------------------------------------------------------------

MANIFEST_ROWS = []

for dataset, dataframe in FINAL_MANIFEST_DATAFRAMES.items():

    csv_path = SAVED_DATASET_PATHS.get(dataset)

    if not csv_path or not Path(csv_path).is_file():
        raise RuntimeError(
            f"{dataset}: final synthetic CSV is missing. "
            "Run Section 22 first."
        )

    target_column = TARGET_COLUMNS.get(dataset)

    if not target_column:
        target_column = {
            "adult_income": "income",
            "bank_marketing": "y",
            "diabetes_130us": "readmitted",
        }.get(dataset)

    if not target_column or target_column not in dataframe.columns:
        raise RuntimeError(
            f"{dataset}: target column is missing from the final dataframe."
        )

    runtime_info = manifest_runtime_info(dataset)
    model_artifacts = manifest_model_artifacts(dataset)

    MANIFEST_ROWS.append({
        "dataset": dataset,
        "synthetic_csv": str(Path(csv_path).resolve()),
        "synthetic_csv_sha256": metadata_file_sha256(csv_path),
        "rows": int(len(dataframe)),
        "columns": int(len(dataframe.columns)),
        "column_names_json": json.dumps(
            [str(column) for column in dataframe.columns],
            ensure_ascii=False,
        ),
        "target_column": target_column,
        "target_present": True,
        "model_artifact_count": len(model_artifacts),
        "model_artifacts_json": json.dumps(
            model_artifacts,
            ensure_ascii=False,
        ),
        "synthetic_sha256": dataframe_sha256(dataframe),
        "seed": generation_seed(dataset, 0),
        "runtime_seconds": runtime_info["runtime_seconds"],
        "runtime_measurement_count": (
            runtime_info["runtime_measurement_count"]
        ),
        "status": "GENERATED",
    })

# -----------------------------------------------------------------------------
# 4. Save CSV and JSON manifests
# -----------------------------------------------------------------------------

GENERATION_MANIFEST_DF = pd.DataFrame(MANIFEST_ROWS)

NB13_MANIFEST_ROOT = Path(NB13_MANIFEST_ROOT)
NB13_MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

MANIFEST_CSV_PATH = (
    NB13_MANIFEST_ROOT / "synthetic_generation_manifest.csv"
)
MANIFEST_JSON_PATH = (
    NB13_MANIFEST_ROOT / "synthetic_generation_manifest.json"
)

GENERATION_MANIFEST_DF.to_csv(
    MANIFEST_CSV_PATH,
    index=False,
)

with MANIFEST_JSON_PATH.open("w", encoding="utf-8") as handle:
    json.dump(
        MANIFEST_ROWS,
        handle,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

# -----------------------------------------------------------------------------
# 5. Verify saved manifests
# -----------------------------------------------------------------------------

CSV_MANIFEST_CHECK = pd.read_csv(MANIFEST_CSV_PATH)

with MANIFEST_JSON_PATH.open("r", encoding="utf-8") as handle:
    JSON_MANIFEST_CHECK = json.load(handle)

if len(CSV_MANIFEST_CHECK) != len(FINAL_MANIFEST_DATAFRAMES):
    raise RuntimeError("CSV manifest dataset count mismatch.")

if len(JSON_MANIFEST_CHECK) != len(FINAL_MANIFEST_DATAFRAMES):
    raise RuntimeError("JSON manifest dataset count mismatch.")

if not CSV_MANIFEST_CHECK["target_present"].all():
    raise RuntimeError("At least one manifest target check failed.")

# -----------------------------------------------------------------------------
# 6. Final summary
# -----------------------------------------------------------------------------

print("\nSECTION 24 — FINAL SUMMARY")
print("=" * 100)
print("Datasets documented :", len(GENERATION_MANIFEST_DF))
print("Total synthetic rows:", int(GENERATION_MANIFEST_DF["rows"].sum()))
print("Model artifacts     :", int(
    GENERATION_MANIFEST_DF["model_artifact_count"].sum()
))
print("Targets verified    :", int(
    GENERATION_MANIFEST_DF["target_present"].sum()
))
print("CSV manifest        :", MANIFEST_CSV_PATH)
print("JSON manifest       :", MANIFEST_JSON_PATH)
print("CSV reload verified :", len(CSV_MANIFEST_CHECK))
print("JSON reload verified:", len(JSON_MANIFEST_CHECK))

display(
    GENERATION_MANIFEST_DF[
        [
            "dataset",
            "rows",
            "columns",
            "target_column",
            "model_artifact_count",
            "synthetic_sha256",
            "runtime_seconds",
            "status",
        ]
    ]
)

print("\nSECTION 24 EXECUTION COMPLETED.")


SECTION 24 — SAVE GENERATION MANIFEST

SECTION 24 — FINAL SUMMARY
Datasets documented : 3
Total synthetic rows: 137072
Model artifacts     : 15
Targets verified    : 3
CSV manifest        : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/manifests/synthetic_generation_manifest.csv
JSON manifest       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/manifests/synthetic_generation_manifest.json
CSV reload verified : 3
JSON reload verified: 3


,dataset,rows,columns,target_column,model_artifact_count,synthetic_sha256,runtime_seconds,status
0,adult_income,34189,15,income,5,493cd5c363761c70ca97b7bc536ec4fa6907758b8c77f6...,11.550080,GENERATED
1,bank_marketing,31647,17,y,5,c9cc07fa48b139d544608c44c6c5c689960f4c545f06a3...,9.987916,GENERATED
2,diabetes_130us,71236,48,readmitted,5,0dfe0dcdf829f3e549f8c1ae9c8cd014ef5e12f039b339...,57.020966,GENERATED



SECTION 24 EXECUTION COMPLETED.


In [44]:
# ==================================================================================================
# SECTION 25 — FINAL VERIFICATION AND STRUCTURAL QUALITY GATE
# SPP-GAN Synthetic Data Generation | Notebook 13
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd

print("=" * 100)
print("SECTION 25 — FINAL VERIFICATION AND STRUCTURAL QUALITY GATE")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Configure project paths
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(
    globals().get(
        "PROJECT_ROOT",
        "/content/drive/MyDrive/SPP_GAN_Research"
    )
)

NB13_ROOT = PROJECT_ROOT / "results" / "notebooks" / "notebook_13"

NB13_VALIDATION_ROOT = Path(
    globals().get(
        "NB13_VALIDATION_ROOT",
        NB13_ROOT / "validation"
    )
)

NB13_SYNTHETIC_ROOT = Path(
    globals().get(
        "NB13_SYNTHETIC_ROOT",
        NB13_ROOT / "synthetic_datasets"
    )
)

NB13_VALIDATION_ROOT.mkdir(parents=True, exist_ok=True)
NB13_SYNTHETIC_ROOT.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------------------------------------
# 2. Recover validation reports from memory or saved CSV files
# --------------------------------------------------------------------------------------------------

REPORT_CANDIDATES = {
    "SAMPLE_SIZE_DF": [
        "sample_size_validation.csv",
        "sample_size_validation_summary.csv",
    ],
    "INVALID_VALUES_DF": [
        "invalid_values_validation.csv",
        "invalid_value_validation.csv",
    ],
    "SCHEMA_VALIDATION_DF": [
        "schema_validation.csv",
        "schema_validation_summary.csv",
        "synthetic_schema_validation.csv",
    ],
    "RANGE_VALIDATION_DF": [
        "range_validation.csv",
        "range_validation_summary.csv",
        "synthetic_range_validation.csv",
        "numeric_range_validation.csv",
    ],
    "DOMAIN_VALIDATION_DF": [
        "domain_validation.csv",
        "domain_validation_summary.csv",
        "synthetic_domain_validation.csv",
    ],
    "MISSINGNESS_DF": [
        "missingness_validation.csv",
        "missingness_validation_summary.csv",
        "synthetic_missingness_validation.csv",
    ],
}


def load_validation_report(variable_name, candidate_filenames):
    existing = globals().get(variable_name)

    if isinstance(existing, pd.DataFrame):
        print(
            f"✓ {variable_name}: in-memory report "
            f"({len(existing):,} rows)"
        )
        return existing

    search_roots = [
        NB13_VALIDATION_ROOT,
        NB13_ROOT / "metadata",
        NB13_ROOT / "manifests",
        NB13_ROOT,
    ]

    candidate_names = {
        filename.lower() for filename in candidate_filenames
    }

    matches = []

    for root in search_roots:
        if not root.exists():
            continue

        try:
            for path in root.rglob("*.csv"):
                if path.name.lower() in candidate_names:
                    matches.append(path)
        except OSError:
            continue

    matches = sorted(
        set(matches),
        key=lambda path: (
            0 if NB13_VALIDATION_ROOT in path.parents else 1,
            len(str(path)),
        ),
    )

    for path in matches:
        try:
            report = pd.read_csv(path, low_memory=False)

            if not report.empty:
                globals()[variable_name] = report
                print(
                    f"✓ {variable_name}: recovered from {path} "
                    f"({len(report):,} rows)"
                )
                return report

        except Exception as exc:
            print(
                f"⚠ Could not read {path}: "
                f"{type(exc).__name__}: {exc}"
            )

    print(f"⚠ {variable_name}: not found in memory or saved CSV files.")
    return None


for report_name, filenames in REPORT_CANDIDATES.items():
    load_validation_report(report_name, filenames)

# --------------------------------------------------------------------------------------------------
# 3. Validate required objects
# --------------------------------------------------------------------------------------------------

REQUIRED_OBJECTS = [
    "SYNTHETIC_DATAFRAMES_WITH_TARGET",
    "DATASET_IDS",
    "TARGET_COLUMNS",
    "SAMPLE_SIZE_DF",
    "INVALID_VALUES_DF",
    "SCHEMA_VALIDATION_DF",
    "RANGE_VALIDATION_DF",
    "DOMAIN_VALIDATION_DF",
    "MISSINGNESS_DF",
]

missing_objects = [
    name
    for name in REQUIRED_OBJECTS
    if name not in globals() or globals()[name] is None
]

if missing_objects:
    raise NameError(
        "Section 25 cannot proceed. Missing required objects/reports: "
        + ", ".join(missing_objects)
        + ". Rerun the upstream sections that create these reports. "
        "Do not create empty placeholder DataFrames."
    )

if not isinstance(SYNTHETIC_DATAFRAMES_WITH_TARGET, dict):
    raise TypeError(
        "SYNTHETIC_DATAFRAMES_WITH_TARGET must be a dictionary."
    )

for report_name in REPORT_CANDIDATES:
    if not isinstance(globals()[report_name], pd.DataFrame):
        raise TypeError(
            f"{report_name} must be a pandas DataFrame."
        )

DATASET_IDS = list(DATASET_IDS)
TARGET_COLUMNS = dict(TARGET_COLUMNS)

if not DATASET_IDS:
    raise ValueError("DATASET_IDS is empty.")

if len({str(dataset).strip().lower() for dataset in DATASET_IDS}) != len(DATASET_IDS):
    raise ValueError("DATASET_IDS contains duplicate dataset identifiers.")

FINAL_SYNTHETIC_FILES = {
    dataset: NB13_SYNTHETIC_ROOT / f"{dataset}_sppgan_synthetic.csv"
    for dataset in DATASET_IDS
}

# --------------------------------------------------------------------------------------------------
# 4. Helper functions
# --------------------------------------------------------------------------------------------------

def normalize_id(value):
    if pd.isna(value):
        return None
    return str(value).strip().lower()


def find_column(df, candidates):
    if not isinstance(df, pd.DataFrame):
        return None

    lookup = {
        str(column).strip().lower(): column
        for column in df.columns
    }

    for candidate in candidates:
        key = candidate.strip().lower()

        if key in lookup:
            return lookup[key]

    return None


def dataset_rows(report, dataset):
    if not isinstance(report, pd.DataFrame) or report.empty:
        return pd.DataFrame()

    dataset_col = find_column(report, ["dataset", "dataset_id"])

    if dataset_col is None:
        return pd.DataFrame()

    mask = report[dataset_col].map(normalize_id).eq(
        normalize_id(dataset)
    )

    return report.loc[mask].copy()


def status_bool(value):
    if pd.isna(value):
        return None

    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    if isinstance(value, (int, float, np.integer, np.floating)):
        if value == 1:
            return True
        if value == 0:
            return False
        return None

    value = str(value).strip().upper()

    if value in {
        "PASS",
        "PASSED",
        "TRUE",
        "VALID",
        "OK",
        "SUCCESS",
        "PASS_NO_REPORTED_REVIEW_FLAGS",
    }:
        return True

    if value in {
        "FAIL",
        "FAILED",
        "FALSE",
        "INVALID",
        "ERROR",
    }:
        return False

    return None


def report_status(report, dataset, status_columns):
    rows = dataset_rows(report, dataset)

    if rows.empty:
        return None

    status_col = find_column(rows, status_columns)

    if status_col is None:
        return None

    values = [status_bool(value) for value in rows[status_col]]

    if not values or any(value is None for value in values):
        return None

    return bool(all(values))


def report_review_count(report, dataset, status_columns):
    rows = dataset_rows(report, dataset)

    if rows.empty:
        return None

    status_col = find_column(rows, status_columns)

    if status_col is None:
        return None

    statuses = rows[status_col].astype(str).str.strip().str.upper()

    return int(
        statuses.isin([
            "REVIEW",
            "WARN",
            "WARNING",
            "FAIL",
            "FAILED",
            "INVALID",
            "ERROR",
            "REVIEW_REQUIRED",
        ]).sum()
    )


def report_count_sum(report, dataset, count_columns):
    rows = dataset_rows(report, dataset)

    if rows.empty:
        return None

    wanted = {name.lower() for name in count_columns}

    count_cols = [
        column
        for column in rows.columns
        if str(column).strip().lower() in wanted
    ]

    if not count_cols:
        return None

    numeric = rows[count_cols].apply(pd.to_numeric, errors="coerce")

    if numeric.isna().any().any():
        return None

    values = numeric.to_numpy(dtype=float)

    if not np.isfinite(values).all():
        return None

    return int(values.sum())


# --------------------------------------------------------------------------------------------------
# 5. Schema verification
# --------------------------------------------------------------------------------------------------

def get_schema_pass(dataset):
    result = report_status(
        SCHEMA_VALIDATION_DF,
        dataset,
        [
            "schema_pass",
            "schema_status",
            "validation_status",
            "status",
            "overall_status",
        ],
    )

    if result is not None:
        return result

    prior = globals().get("FINAL_VERIFICATION_DF")

    if isinstance(prior, pd.DataFrame) and not prior.empty:
        rows = dataset_rows(prior, dataset)
        column = find_column(rows, ["schema_pass"])

        if not rows.empty and column is not None:
            values = [status_bool(value) for value in rows[column]]

            if values and all(value is not None for value in values):
                return bool(all(values))

    return None


# --------------------------------------------------------------------------------------------------
# 6. Sample-size verification
# --------------------------------------------------------------------------------------------------

def get_sample_size_pass(dataset, in_memory_rows):
    rows = dataset_rows(SAMPLE_SIZE_DF, dataset)

    if len(rows) != 1:
        return None

    row = rows.iloc[0]

    status_col = find_column(rows, ["sample_size_status", "status"])
    expected_col = find_column(rows, ["expected_rows", "expected_count"])
    actual_col = find_column(rows, ["actual_rows", "generated_rows"])
    difference_col = find_column(rows, ["row_count_difference"])

    if status_col is None or expected_col is None or actual_col is None:
        return None

    status = status_bool(row[status_col])

    expected = pd.to_numeric(
        pd.Series([row[expected_col]]), errors="coerce"
    ).iloc[0]

    actual = pd.to_numeric(
        pd.Series([row[actual_col]]), errors="coerce"
    ).iloc[0]

    if pd.isna(expected) or pd.isna(actual):
        return None

    if difference_col is not None:
        difference = pd.to_numeric(
            pd.Series([row[difference_col]]), errors="coerce"
        ).iloc[0]

        if pd.isna(difference) or difference != 0:
            return False

    counts_match = bool(
        expected == actual
        and actual == in_memory_rows
    )

    if status is False or not counts_match:
        return False

    return True if status is True and counts_match else None


# --------------------------------------------------------------------------------------------------
# 7. Invalid-value verification
# --------------------------------------------------------------------------------------------------

def get_invalid_value_pass(dataset):
    rows = dataset_rows(INVALID_VALUES_DF, dataset)

    if rows.empty:
        return None, "NO_INVALID_VALUE_REPORT_ROWS"

    role_col = find_column(rows, ["column_role"])

    if role_col is not None:
        identifier_mask = (
            rows[role_col].astype(str).str.strip().str.lower()
            == "identifier"
        )

        identifier_rows = rows.loc[identifier_mask]
        value_rows = rows.loc[~identifier_mask].copy()
    else:
        identifier_rows = pd.DataFrame()
        value_rows = rows.copy()

    if value_rows.empty:
        return None, "NO_FEATURE_OR_TARGET_ROWS_TO_VALIDATE"

    status_col = find_column(
        value_rows,
        ["value_status", "invalid_status", "validation_status"],
    )

    if status_col is None:
        return None, "VALUE_STATUS_COLUMN_MISSING"

    results = [status_bool(value) for value in value_rows[status_col]]

    if not results or any(value is None for value in results):
        return None, "UNKNOWN_VALUE_STATUS"

    if not all(results):
        return False, "FEATURE_OR_TARGET_VALUE_CHECK_FAILED"

    invalid_count_col = find_column(
        value_rows,
        ["invalid_value_count", "invalid_count", "violation_count"],
    )

    if invalid_count_col is None:
        return None, "INVALID_VALUE_COUNT_COLUMN_MISSING"

    invalid_counts = pd.to_numeric(
        value_rows[invalid_count_col], errors="coerce"
    )

    if invalid_counts.isna().any():
        return None, "INVALID_VALUE_COUNTS_INCOMPLETE"

    if not np.isfinite(invalid_counts.to_numpy(dtype=float)).all():
        return None, "INVALID_VALUE_COUNTS_NONFINITE"

    if (invalid_counts != 0).any():
        return False, "NONZERO_INVALID_VALUE_COUNT"

    component_columns = [
        "non_finite_numeric_count",
        "nonnumeric_in_numeric_column_count",
        "unseen_category_count",
        "fractional_integer_value_count",
    ]

    for candidate in component_columns:
        column = find_column(value_rows, [candidate])

        if column is None:
            continue

        counts = pd.to_numeric(value_rows[column], errors="coerce")

        if counts.isna().any():
            return None, f"{candidate}_INCOMPLETE"

        if not np.isfinite(counts.to_numpy(dtype=float)).all():
            return None, f"{candidate}_NONFINITE"

        if (counts != 0).any():
            return False, f"{candidate}_NONZERO"

    identifier_note = "NO_IDENTIFIER_ROWS"

    if not identifier_rows.empty:
        overall_col = find_column(identifier_rows, ["overall_status"])

        if overall_col is not None:
            statuses = (
                identifier_rows[overall_col]
                .astype(str).str.strip().str.upper()
            )

            if statuses.eq("SCHEMA_MISMATCH").any():
                identifier_note = "IDENTIFIER_SCHEMA_MISMATCH_REPORTED_SEPARATELY"
            else:
                identifier_note = "IDENTIFIER_ROWS_RECORDED_SEPARATELY"

    return True, f"FEATURE_TARGET_VALUE_CHECKS_PASS; {identifier_note}"


# --------------------------------------------------------------------------------------------------
# 8. Verify final synthetic datasets and saved CSV files
# --------------------------------------------------------------------------------------------------

verification_rows = []

for dataset in DATASET_IDS:
    print("\n" + "-" * 100)
    print(f"VERIFYING DATASET: {dataset}")
    print("-" * 100)

    target_col = TARGET_COLUMNS.get(dataset)
    csv_path = FINAL_SYNTHETIC_FILES[dataset]

    if dataset not in SYNTHETIC_DATAFRAMES_WITH_TARGET:
        verification_rows.append({
            "dataset": dataset,
            "synthetic_rows": None,
            "synthetic_columns": None,
            "target_column": target_col,
            "target_present": False,
            "schema_pass": None,
            "sample_size_pass": None,
            "invalid_values_pass": None,
            "invalid_values_note": "FINAL_DATAFRAME_MISSING",
            "saved_file_pass": False,
            "saved_rows_match": False,
            "saved_columns_match": False,
            "saved_target_present": False,
            "saved_column_order_match": False,
            "range_report_available": False,
            "range_review_count": None,
            "domain_report_available": False,
            "domain_review_count": None,
            "domain_violation_count": None,
            "missingness_report_available": False,
            "missingness_review_count": None,
            "required_evidence_complete": False,
            "quality_evidence_complete": False,
            "structural_verification_status": "REVIEW",
            "quality_review_status": "REVIEW",
            "saved_csv_path": str(csv_path),
            "saved_file_error": "FINAL_DATAFRAME_MISSING",
        })
        continue

    synthetic_df = SYNTHETIC_DATAFRAMES_WITH_TARGET[dataset]

    if not isinstance(synthetic_df, pd.DataFrame):
        raise TypeError(
            f"{dataset}: final synthetic object is not a DataFrame."
        )

    synthetic_rows = len(synthetic_df)
    synthetic_columns = len(synthetic_df.columns)

    target_present = (
        target_col in synthetic_df.columns
        if target_col else False
    )

    schema_pass = get_schema_pass(dataset)
    sample_pass = get_sample_size_pass(dataset, synthetic_rows)
    invalid_pass, invalid_note = get_invalid_value_pass(dataset)

    saved_file_pass = False
    saved_rows_match = False
    saved_columns_match = False
    saved_target_present = False
    saved_column_order_match = False
    saved_file_error = None

    if csv_path.exists():
        try:
            saved_df = pd.read_csv(csv_path, low_memory=False)

            saved_rows_match = len(saved_df) == synthetic_rows
            saved_columns_match = len(saved_df.columns) == synthetic_columns

            saved_target_present = (
                target_col in saved_df.columns
                if target_col else False
            )

            saved_column_order_match = (
                list(saved_df.columns) == list(synthetic_df.columns)
            )

            saved_file_pass = bool(
                saved_rows_match
                and saved_columns_match
                and saved_target_present
                and saved_column_order_match
            )

        except Exception as exc:
            saved_file_error = f"{type(exc).__name__}: {exc}"
    else:
        saved_file_error = "CANONICAL_SYNTHETIC_CSV_NOT_FOUND"

    # ----------------------------------------------------------------------------------------------
    # 9. Range, domain, and missingness findings
    # ----------------------------------------------------------------------------------------------

    range_rows = dataset_rows(RANGE_VALIDATION_DF, dataset)
    domain_rows = dataset_rows(DOMAIN_VALIDATION_DF, dataset)
    missingness_rows = dataset_rows(MISSINGNESS_DF, dataset)

    range_review_count = report_review_count(
        RANGE_VALIDATION_DF,
        dataset,
        ["range_status", "validation_status", "status"],
    )

    domain_review_count = report_review_count(
        DOMAIN_VALIDATION_DF,
        dataset,
        ["domain_status", "validation_status", "status"],
    )

    domain_violation_count = report_count_sum(
        DOMAIN_VALIDATION_DF,
        dataset,
        [
            "below_domain_min_count",
            "above_domain_max_count",
            "fractional_value_count",
            "nonnumeric_count",
            "nonfinite_count",
        ],
    )

    missingness_review_count = report_review_count(
        MISSINGNESS_DF,
        dataset,
        ["missingness_status", "validation_status", "status"],
    )

    range_available = not range_rows.empty
    domain_available = not domain_rows.empty
    missingness_available = not missingness_rows.empty

    # ----------------------------------------------------------------------------------------------
    # 10. Structural verification gate
    # ----------------------------------------------------------------------------------------------

    required_checks = [
        schema_pass,
        sample_pass,
        invalid_pass,
        saved_file_pass,
        target_present,
    ]

    required_evidence_complete = all(
        value is not None for value in required_checks
    )

    structural_pass = bool(
        required_evidence_complete
        and all(value is True for value in required_checks)
    )

    structural_status = "PASS" if structural_pass else "REVIEW"

    # ----------------------------------------------------------------------------------------------
    # 11. Conservative quality-review gate
    # ----------------------------------------------------------------------------------------------

    quality_evidence_complete = bool(
        range_available
        and domain_available
        and missingness_available
        and range_review_count is not None
        and domain_review_count is not None
        and missingness_review_count is not None
        and domain_violation_count is not None
    )

    if quality_evidence_complete:
        has_review_flags = any([
            range_review_count > 0,
            domain_review_count > 0,
            missingness_review_count > 0,
        ])

        has_domain_violations = domain_violation_count > 0

        quality_status = (
            "PASS_NO_REPORTED_REVIEW_FLAGS"
            if not has_review_flags and not has_domain_violations
            else "REVIEW"
        )
    else:
        quality_status = "REVIEW"

    record = {
        "dataset": dataset,
        "synthetic_rows": synthetic_rows,
        "synthetic_columns": synthetic_columns,
        "target_column": target_col,
        "target_present": target_present,
        "schema_pass": schema_pass,
        "sample_size_pass": sample_pass,
        "invalid_values_pass": invalid_pass,
        "invalid_values_note": invalid_note,
        "saved_file_pass": saved_file_pass,
        "saved_rows_match": saved_rows_match,
        "saved_columns_match": saved_columns_match,
        "saved_target_present": saved_target_present,
        "saved_column_order_match": saved_column_order_match,
        "range_report_available": range_available,
        "range_review_count": range_review_count,
        "domain_report_available": domain_available,
        "domain_review_count": domain_review_count,
        "domain_violation_count": domain_violation_count,
        "missingness_report_available": missingness_available,
        "missingness_review_count": missingness_review_count,
        "required_evidence_complete": required_evidence_complete,
        "quality_evidence_complete": quality_evidence_complete,
        "structural_verification_status": structural_status,
        "quality_review_status": quality_status,
        "saved_csv_path": str(csv_path),
        "saved_file_error": saved_file_error,
    }

    verification_rows.append(record)

    print(f"Rows / columns              : {synthetic_rows:,} / {synthetic_columns:,}")
    print(f"Target present              : {target_present}")
    print(f"Schema check                : {schema_pass}")
    print(f"Sample-size check           : {sample_pass}")
    print(f"Feature/target value check  : {invalid_pass}")
    print(f"Saved CSV structure check   : {saved_file_pass}")
    print(f"Range review rows           : {range_review_count}")
    print(f"Domain review rows          : {domain_review_count}")
    print(f"Domain violation count      : {domain_violation_count}")
    print(f"Missingness review rows     : {missingness_review_count}")
    print(f"Structural status           : {structural_status}")
    print(f"Quality evidence complete   : {quality_evidence_complete}")
    print(f"Quality review status       : {quality_status}")

    if saved_file_error:
        print(f"Saved CSV note              : {saved_file_error}")

# --------------------------------------------------------------------------------------------------
# 12. Save final verification reports
# --------------------------------------------------------------------------------------------------

FINAL_VERIFICATION_DF = pd.DataFrame(verification_rows)

FINAL_VERIFICATION_CSV = (
    NB13_VALIDATION_ROOT / "final_verification.csv"
)

FINAL_VERIFICATION_JSON = (
    NB13_VALIDATION_ROOT / "final_verification_summary.json"
)

FINAL_VERIFICATION_DF.to_csv(
    FINAL_VERIFICATION_CSV,
    index=False,
)

structural_pass_count = int(
    FINAL_VERIFICATION_DF[
        "structural_verification_status"
    ].eq("PASS").sum()
)

structural_review_count = int(
    FINAL_VERIFICATION_DF[
        "structural_verification_status"
    ].eq("REVIEW").sum()
)

quality_review_count = int(
    FINAL_VERIFICATION_DF[
        "quality_review_status"
    ].eq("REVIEW").sum()
)

quality_pass_count = int(
    FINAL_VERIFICATION_DF[
        "quality_review_status"
    ].eq("PASS_NO_REPORTED_REVIEW_FLAGS").sum()
)

summary = {
    "section": "Notebook 13 — Section 25",
    "verification_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "datasets_checked": int(len(FINAL_VERIFICATION_DF)),
    "structural_pass_count": structural_pass_count,
    "structural_review_count": structural_review_count,
    "quality_pass_count": quality_pass_count,
    "quality_review_count": quality_review_count,
    "structural_verification_scope": [
        "schema validation report",
        "sample-size report and row-count consistency",
        "feature/target invalid-value statuses and violation counts",
        "target-column presence",
        "saved CSV row count, column count, target, and column order",
    ],
    "quality_review_scope": [
        "range validation",
        "domain validation",
        "missingness validation",
    ],
    "interpretation": (
        "Structural PASS indicates only that explicit structural checks passed. "
        "It does not establish statistical fidelity, downstream utility, acceptable "
        "domain validity, or end-to-end differential privacy. Missing or "
        "uninterpretable evidence is classified as REVIEW."
    ),
    "reports_used": {
        name: (
            "AVAILABLE"
            if isinstance(globals().get(name), pd.DataFrame)
            else "UNAVAILABLE"
        )
        for name in REPORT_CANDIDATES
    },
    "final_csv": str(FINAL_VERIFICATION_CSV),
    "final_json": str(FINAL_VERIFICATION_JSON),
}

with open(FINAL_VERIFICATION_JSON, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2, ensure_ascii=False)

# --------------------------------------------------------------------------------------------------
# 13. Reload and verify saved report artifacts
# --------------------------------------------------------------------------------------------------

RELOADED_FINAL_VERIFICATION_DF = pd.read_csv(
    FINAL_VERIFICATION_CSV,
    low_memory=False,
)

with open(FINAL_VERIFICATION_JSON, "r", encoding="utf-8") as file:
    RELOADED_FINAL_VERIFICATION_SUMMARY = json.load(file)

assert len(RELOADED_FINAL_VERIFICATION_DF) == len(DATASET_IDS), (
    "Saved verification CSV row count does not match registered datasets."
)

assert (
    RELOADED_FINAL_VERIFICATION_SUMMARY["datasets_checked"]
    == len(DATASET_IDS)
), "Saved JSON dataset count does not match registered datasets."

assert (
    RELOADED_FINAL_VERIFICATION_SUMMARY["structural_pass_count"]
    == structural_pass_count
), "Saved JSON structural pass count is inconsistent."

assert (
    RELOADED_FINAL_VERIFICATION_SUMMARY["quality_review_count"]
    == quality_review_count
), "Saved JSON quality review count is inconsistent."

# --------------------------------------------------------------------------------------------------
# 14. Final summary
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 25 — FINAL SUMMARY")
print("=" * 100)

print(f"Datasets checked                    : {len(FINAL_VERIFICATION_DF)}")
print(f"Structural checks passed            : {structural_pass_count}")
print(f"Datasets requiring structural review: {structural_review_count}")
print(f"Quality checks passed                : {quality_pass_count}")
print(f"Datasets requiring quality review    : {quality_review_count}")
print(f"Final verification CSV               : {FINAL_VERIFICATION_CSV}")
print(f"Final verification JSON              : {FINAL_VERIFICATION_JSON}")

display_columns = [
    "dataset",
    "synthetic_rows",
    "synthetic_columns",
    "schema_pass",
    "sample_size_pass",
    "invalid_values_pass",
    "saved_file_pass",
    "range_review_count",
    "domain_review_count",
    "domain_violation_count",
    "missingness_review_count",
    "required_evidence_complete",
    "quality_evidence_complete",
    "structural_verification_status",
    "quality_review_status",
]

display(
    FINAL_VERIFICATION_DF[
        [
            column
            for column in display_columns
            if column in FINAL_VERIFICATION_DF.columns
        ]
    ]
)

print("\nIMPORTANT:")
print(
    "Structural PASS does not certify statistical fidelity, downstream utility, "
    "acceptable domain validity, or end-to-end differential privacy."
)
print(
    "Quality REVIEW indicates reported review flags or missing/uninterpretable "
    "evidence. Inspect the underlying validation reports."
)
print("Section 25 completed. Review the summary before accepting the quality gate.")

# ==================================================================================================
# END OF SECTION 25
# ==================================================================================================

SECTION 25 — FINAL VERIFICATION AND STRUCTURAL QUALITY GATE
✓ SAMPLE_SIZE_DF: in-memory report (3 rows)
✓ INVALID_VALUES_DF: in-memory report (83 rows)
✓ SCHEMA_VALIDATION_DF: in-memory report (3 rows)
✓ RANGE_VALIDATION_DF: recovered from /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/range_validation.csv (24 rows)
✓ DOMAIN_VALIDATION_DF: recovered from /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/domain_validation.csv (22 rows)
✓ MISSINGNESS_DF: in-memory report (83 rows)

----------------------------------------------------------------------------------------------------
VERIFYING DATASET: adult_income
----------------------------------------------------------------------------------------------------
Rows / columns              : 34,189 / 15
Target present              : True
Schema check                : True
Sample-size check           : True
Feature/target value check  : True
Saved CSV structure check   : True

,dataset,synthetic_rows,synthetic_columns,schema_pass,sample_size_pass,invalid_values_pass,saved_file_pass,range_review_count,domain_review_count,domain_violation_count,missingness_review_count,required_evidence_complete,quality_evidence_complete,structural_verification_status,quality_review_status
0,adult_income,34189,15,True,True,True,True,0,0,0,3,True,True,PASS,REVIEW
1,bank_marketing,31647,17,True,True,True,True,0,0,0,0,True,True,PASS,PASS_NO_REPORTED_REVIEW_FLAGS
2,diabetes_130us,71236,48,True,True,True,True,0,0,0,7,True,True,PASS,REVIEW



IMPORTANT:
Structural PASS does not certify statistical fidelity, downstream utility, acceptable domain validity, or end-to-end differential privacy.
Quality REVIEW indicates reported review flags or missing/uninterpretable evidence. Inspect the underlying validation reports.
Section 25 completed. Review the summary before accepting the quality gate.


In [ ]:

# =============================================================================
# SECTION 26 — COMPLETION SUMMARY
# SPP-GAN Synthetic Data Generation | Notebook 13
# =============================================================================
# Purpose:
#   1. Summarize final synthetic datasets, including target columns.
#   2. Cross-check structural verification results from Section 25.
#   3. Record generation errors and unresolved quality-review findings.
#   4. Persist a reproducible completion summary.
#
# Important:
#   GENERATION_COMPLETED does not mean QUALITY_VALIDATED or PRIVACY_CERTIFIED.
# =============================================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

print("=" * 100)
print("NOTEBOOK 13 — COMPLETION SUMMARY")
print("=" * 100)

# -----------------------------------------------------------------------------
# 1. Validate required objects
# -----------------------------------------------------------------------------

required_objects = [
    "DATASET_IDS",
    "SYNTHETIC_DATAFRAMES_WITH_TARGET",
    "GENERATION_ERRORS",
    "PROJECT_ROOT",
    "NB13_SYNTHETIC_ROOT",
    "NB13_CONFIG_ROOT",
    "NB13_METADATA_ROOT",
    "NB13_MANIFEST_ROOT",
    "NB13_VALIDATION_ROOT",
    "NB13_LOG_ROOT",
    "FINAL_VERIFICATION_DF",
]

missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise NameError(
        "Required Notebook 13 variables are missing: "
        + ", ".join(missing_objects)
    )

DATASET_IDS = list(DATASET_IDS)
FINAL_DATAFRAMES = SYNTHETIC_DATAFRAMES_WITH_TARGET

# Ensure the metadata directory exists.
NB13_METADATA_ROOT = Path(NB13_METADATA_ROOT)
NB13_METADATA_ROOT.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# 2. Identify generated, missing, and malformed datasets
# -----------------------------------------------------------------------------

generated_datasets = [
    dataset
    for dataset in DATASET_IDS
    if dataset in FINAL_DATAFRAMES
    and isinstance(FINAL_DATAFRAMES[dataset], pd.DataFrame)
    and not FINAL_DATAFRAMES[dataset].empty
]

failed_or_skipped_datasets = sorted(
    set(DATASET_IDS) - set(generated_datasets)
)

malformed_datasets = []
target_missing_datasets = []

for dataset in generated_datasets:
    df = FINAL_DATAFRAMES[dataset]
    target_col = TARGET_COLUMNS.get(dataset) if "TARGET_COLUMNS" in globals() else None

    if df.columns.duplicated().any():
        malformed_datasets.append(dataset)

    if target_col is None or target_col not in df.columns:
        target_missing_datasets.append(dataset)

# -----------------------------------------------------------------------------
# 3. Summarize final verification results
# -----------------------------------------------------------------------------

if not isinstance(FINAL_VERIFICATION_DF, pd.DataFrame):
    raise TypeError("FINAL_VERIFICATION_DF must be a pandas DataFrame.")

verification_records = (
    FINAL_VERIFICATION_DF.to_dict(orient="records")
    if not FINAL_VERIFICATION_DF.empty
    else []
)

verification_status_col = (
    "structural_verification_status"
    if "structural_verification_status" in FINAL_VERIFICATION_DF.columns
    else None
)

quality_status_col = (
    "quality_review_status"
    if "quality_review_status" in FINAL_VERIFICATION_DF.columns
    else None
)

if verification_status_col:
    structural_pass_count = int(
        FINAL_VERIFICATION_DF[verification_status_col]
        .astype(str).str.upper().eq("PASS").sum()
    )
    structural_review_count = int(
        FINAL_VERIFICATION_DF[verification_status_col]
        .astype(str).str.upper().eq("REVIEW").sum()
    )
else:
    structural_pass_count = 0
    structural_review_count = len(DATASET_IDS)

if quality_status_col:
    quality_review_count = int(
        FINAL_VERIFICATION_DF[quality_status_col]
        .astype(str).str.upper().eq("REVIEW").sum()
    )
else:
    quality_review_count = None

verification_dataset_col = (
    "dataset"
    if "dataset" in FINAL_VERIFICATION_DF.columns
    else "dataset_id" if "dataset_id" in FINAL_VERIFICATION_DF.columns
    else None
)

verified_datasets = set()
if verification_dataset_col:
    verified_datasets = set(
        FINAL_VERIFICATION_DF[verification_dataset_col]
        .dropna().astype(str).str.strip().str.lower()
    )

missing_verification_datasets = sorted(
    set(dataset.lower() for dataset in DATASET_IDS) - verified_datasets
)

all_structurally_verified = bool(
    not missing_verification_datasets
    and structural_pass_count == len(DATASET_IDS)
    and len(FINAL_VERIFICATION_DF) >= len(DATASET_IDS)
)

# -----------------------------------------------------------------------------
# 4. Check final saved CSV artifacts
# -----------------------------------------------------------------------------

saved_artifact_results = {}

for dataset in DATASET_IDS:
    csv_path = (
        Path(NB13_SYNTHETIC_ROOT)
        / f"{dataset}_sppgan_synthetic.csv"
    )

    in_memory_df = FINAL_DATAFRAMES.get(dataset)

    result = {
        "path": str(csv_path),
        "exists": csv_path.exists(),
        "readable": False,
        "rows_match": False,
        "columns_match": False,
        "target_present": False,
        "artifact_status": "REVIEW",
    }

    if csv_path.exists() and isinstance(in_memory_df, pd.DataFrame):
        try:
            saved_df = pd.read_csv(csv_path, low_memory=False)

            result["readable"] = True
            result["rows_match"] = bool(
                len(saved_df) == len(in_memory_df)
            )
            result["columns_match"] = bool(
                list(saved_df.columns) == list(in_memory_df.columns)
            )

            target_col = (
                TARGET_COLUMNS.get(dataset)
                if "TARGET_COLUMNS" in globals()
                else None
            )

            result["target_present"] = bool(
                target_col is not None and target_col in saved_df.columns
            )

            result["artifact_status"] = (
                "PASS"
                if result["readable"]
                and result["rows_match"]
                and result["columns_match"]
                and result["target_present"]
                else "REVIEW"
            )

        except Exception as exc:
            result["read_error"] = f"{type(exc).__name__}: {exc}"

    saved_artifact_results[dataset] = result

saved_artifacts_pass = all(
    item["artifact_status"] == "PASS"
    for item in saved_artifact_results.values()
)

# -----------------------------------------------------------------------------
# 5. Determine generation and verification statuses independently
# -----------------------------------------------------------------------------

generation_completed = bool(
    len(generated_datasets) == len(DATASET_IDS)
    and not failed_or_skipped_datasets
    and not malformed_datasets
    and not target_missing_datasets
)

generation_status = (
    "COMPLETED"
    if generation_completed
    else "PARTIAL_OR_FAILED"
)

verification_status = (
    "STRUCTURALLY_VERIFIED"
    if all_structurally_verified and saved_artifacts_pass
    else "REVIEW_REQUIRED"
)

quality_status = (
    "REVIEW_REQUIRED"
    if quality_review_count is None or quality_review_count > 0
    else "NO_REPORTED_REVIEW_FLAGS"
)

# Generation errors are preserved in their original form but converted to a
# JSON-compatible value when serialized.
generation_errors_serializable = GENERATION_ERRORS

# -----------------------------------------------------------------------------
# 6. Build completion summary
# -----------------------------------------------------------------------------

COMPLETION_SUMMARY = {
    "framework": "SPP-GAN",
    "notebook": NOTEBOOK_NAME if "NOTEBOOK_NAME" in globals() else "Notebook 13",
    "completion_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),

    "requested_datasets": DATASET_IDS,
    "generated_datasets": generated_datasets,
    "failed_or_skipped_datasets": failed_or_skipped_datasets,
    "malformed_datasets": malformed_datasets,
    "target_missing_datasets": target_missing_datasets,

    "requested_dataset_count": len(DATASET_IDS),
    "generated_dataset_count": len(generated_datasets),
    "failed_or_skipped_dataset_count": len(failed_or_skipped_datasets),

    "generation_status": generation_status,
    "generation_errors": generation_errors_serializable,

    "structural_verification": {
        "status": verification_status,
        "structural_pass_count": structural_pass_count,
        "structural_review_count": structural_review_count,
        "missing_verification_datasets": missing_verification_datasets,
        "all_datasets_structurally_verified": all_structurally_verified,
    },

    "saved_artifact_verification": {
        "all_saved_artifacts_pass": saved_artifacts_pass,
        "artifacts": saved_artifact_results,
    },

    "quality_review": {
        "status": quality_status,
        "datasets_requiring_quality_review": quality_review_count,
        "interpretation": (
            "Quality review is separate from generation and structural verification. "
            "Range, domain, missingness, fidelity, utility, and privacy require their "
            "own evidence and must not be inferred from generation completion."
        ),
    },

    "final_verification": verification_records,

    "artifact_directories": {
        "synthetic_datasets": str(NB13_SYNTHETIC_ROOT),
        "configuration": str(NB13_CONFIG_ROOT),
        "metadata": str(NB13_METADATA_ROOT),
        "manifests": str(NB13_MANIFEST_ROOT),
        "validation": str(NB13_VALIDATION_ROOT),
        "logs": str(NB13_LOG_ROOT),
    },
}

# -----------------------------------------------------------------------------
# 7. Save completion summary
# -----------------------------------------------------------------------------

COMPLETION_SUMMARY_PATH = (
    NB13_METADATA_ROOT / "notebook_13_completion_summary.json"
)

with open(COMPLETION_SUMMARY_PATH, "w", encoding="utf-8") as handle:
    json.dump(
        COMPLETION_SUMMARY,
        handle,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

# -----------------------------------------------------------------------------
# 8. Reload and validate the completion summary
# -----------------------------------------------------------------------------

with open(COMPLETION_SUMMARY_PATH, "r", encoding="utf-8") as handle:
    RELOADED_COMPLETION_SUMMARY = json.load(handle)

assert (
    RELOADED_COMPLETION_SUMMARY["requested_dataset_count"]
    == len(DATASET_IDS)
), "Completion summary dataset count does not match."

assert (
    RELOADED_COMPLETION_SUMMARY["generated_dataset_count"]
    == len(generated_datasets)
), "Completion summary generated count does not match."

# -----------------------------------------------------------------------------
# 9. Final output
# -----------------------------------------------------------------------------

print("=" * 100)
print("NOTEBOOK 13 — COMPLETION SUMMARY")
print("=" * 100)

print("Generation status         :", generation_status)
print("Requested datasets        :", len(DATASET_IDS))
print("Generated datasets        :", len(generated_datasets))
print("Failed/skipped datasets   :", len(failed_or_skipped_datasets))
print("Malformed datasets        :", len(malformed_datasets))
print("Missing target datasets   :", len(target_missing_datasets))
print("Structural pass count     :", structural_pass_count)
print("Structural review count   :", structural_review_count)
print("Artifact verification     :", "PASS" if saved_artifacts_pass else "REVIEW")
print("Overall verification      :", verification_status)
print("Quality review status     :", quality_status)
print("Synthetic data directory  :", NB13_SYNTHETIC_ROOT)

print(
    "Manifest                  :",
    globals().get(
        "MANIFEST_JSON_PATH",
        Path(NB13_MANIFEST_ROOT) / "synthetic_generation_manifest.json",
    ),
)
print(
    "Metadata                  :",
    globals().get(
        "GENERATION_METADATA_PATH",
        Path(NB13_METADATA_ROOT) / "sppgan_synthetic_generation_metadata.json",
    ),
)
print(
    "Final verification        :",
    Path(NB13_VALIDATION_ROOT) / "final_verification.csv",
)
print("Completion summary        :", COMPLETION_SUMMARY_PATH)

if failed_or_skipped_datasets:
    print("\nDatasets not generated:")
    for dataset in failed_or_skipped_datasets:
        print(" -", dataset)

if target_missing_datasets:
    print("\nDatasets missing target columns:")
    for dataset in target_missing_datasets:
        print(" -", dataset)

print("\nSaved artifact verification:")
for dataset, result in saved_artifact_results.items():
    print(f"  {dataset}: {result['artifact_status']}")

print(
    "\nNotebook 13 generation summary saved. "
    "Review unresolved quality findings before using the outputs in research."
)


NOTEBOOK 13 — COMPLETION SUMMARY
NOTEBOOK 13 — COMPLETION SUMMARY
Generation status         : COMPLETED
Requested datasets        : 3
Generated datasets        : 3
Failed/skipped datasets   : 0
Malformed datasets        : 0
Missing target datasets   : 0
Structural pass count     : 3
Structural review count   : 0
Artifact verification     : PASS
Overall verification      : STRUCTURALLY_VERIFIED
Quality review status     : REVIEW_REQUIRED
Synthetic data directory  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/synthetic_datasets
Manifest                  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/manifests/synthetic_generation_manifest.json
Metadata                  : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/metadata/sppgan_synthetic_generation_metadata.json
Final verification        : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_13/validation/final_verification.csv
Completion summary